# General
Project structure:
```
project_root/
├── configs/
│   └── config.py
├── src/
│   ├── core/
│   │   ├── utils.py
│   │   └── geometry.py
│   ├── data/
│   │   ├── augmentations.py
│   │   ├── cache.py
│   │   └── generators.py
│   ├── models/
│   │   ├── builder.py
│   │   └── losses.py
│   ├── training/
│   │   ├── callbacks.py
│   │   └── trainer.py
│   ├── inference/
│   │   └── reconstructor.py
│   └── evaluation/
│   │   ├── metrics.py
│   │   └── visualizer.py
└── main.py
```

In [ ]:
# %rm -rf *.keras databases

In [ ]:
# %pip install git+https://github.com/ValV/SynthSeg.git
# %pip install -e ./SynthSeg

In [ ]:
import os
import glob
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)


def configure_xla_paths():
    """
    Locates libdevice.10.bc in the Conda environment and sets XLA_FLAGS.
    Fixes 'libdevice not found' errors in Conda environments.
    """
    conda_prefix = os.environ.get('CONDA_PREFIX')
    if not conda_prefix:
        return # Not in conda, assume system paths work

    # Common locations for libdevice in Conda envs
    # It often moves around depending on nvcc/cudatoolkit version
    candidates = [
        f"{conda_prefix}/lib/libdevice.10.bc",
        f"{conda_prefix}/lib/nvidia/cuda_nvvm/libdevice/libdevice.10.bc",
        f"{conda_prefix}/nvvm/libdevice/libdevice.10.bc"
    ]

    # Recursive search if standard paths fail (slower but robust)
    if not any(os.path.exists(p) for p in candidates):
        found = glob.glob(f"{conda_prefix}/**/libdevice.10.bc", recursive=True)
        if found:
            candidates = found

    for path in candidates:
        if os.path.exists(path):
            cuda_dir = os.path.dirname(os.path.dirname(os.path.dirname(path))) # usually goes up to where 'bin' and 'lib' are

            # Often XLA wants the root of the CUDA installation (where /nvvm/libdevice lives)
            # Strategy: Point --xla_gpu_cuda_data_dir to the folder containing 'nvvm' or 'lib/nvidia'

            # Simplest for TF 2.x in Conda:
            # Point to the specific directory containing the libdevice file is not supported by the flag directly,
            # it wants the CUDA_DIR.

            # BUT, we can just disable XLA if we can't find it, OR simply set the flag to the conda prefix
            # which usually works if the layout is standard.

            print(f"✅ Found libdevice at: {path}")

            # This flag tells XLA where to look for CUDA libraries
            os.environ['XLA_FLAGS'] = f"--xla_gpu_cuda_data_dir={conda_prefix}"

            # Sometimes LD_LIBRARY_PATH needs help too
            lib_path = os.environ.get('LD_LIBRARY_PATH', '')
            if f"{conda_prefix}/lib" not in lib_path:
                os.environ['LD_LIBRARY_PATH'] = f"{conda_prefix}/lib:{lib_path}"

            return

    print("⚠️ WARNING: libdevice.10.bc not found in Conda environment. XLA might fail.")
    # If not found, safer to disable XLA to prevent crash
    os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'

KAGGLE = False  # determine either we're in Kaggle

if os.environ.get('KAGGLE_URL_BASE', None):
    print(f"Kaggle has XLA paths configured.")
    KAGGLE = True
else:
    # Run immediately
    configure_xla_paths()

In [ ]:
if KAGGLE:
    PATH_DATA_IXI = '/kaggle/input/preprocessed-oasis-and-epilepsy-and-ixi'
    PATH_DATA_BRATS = '/kaggle/input/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
else:
    PATH_DATA_IXI = 'data/ixi'
    PATH_DATA_BRATS = 'data/brats/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'

# Cell 1: Imports & Profiling Utilities
- **Target File:** src/core/utils.py
- **Content:**
    - PipelineTimer (Context manager for timing)
    - set_global_seeds (Reproducibility)
    - Basic logging configuration

In [ ]:
import json
import logging
import os
import random
import sys
import time

import psutil
import subprocess
import csv

from contextlib import contextmanager
from dataclasses import dataclass, field, asdict
from platform import python_version
from typing import Tuple, List, Optional, Union, Any, Dict

import nibabel as nib
import numpy as np
import tensorflow as tf
import tensorflow.keras.backend as K

from scipy.ndimage import find_objects
from tensorflow.keras import mixed_precision


# --- Robust Logging Configuration ---
def setup_logger():
    logger = logging.getLogger("RefactoredPipeline")
    # logger.setLevel(logging.INFO)
    logger.setLevel(logging.DEBUG)

    # Check if handlers already exist to avoid duplicate logs
    if not logger.handlers:
        handler = logging.StreamHandler(sys.stdout)
        formatter = logging.Formatter('[%(asctime)s] %(levelname)s: %(message)s',
                                      datefmt='%H:%M:%S')
        handler.setFormatter(formatter)
        logger.addHandler(handler)

    # Also force Tensorflow to be quiet unless it's an error
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
    logging.getLogger('tensorflow').setLevel(logging.ERROR)

    return logger

logger = setup_logger()

# --- Profiling / Debugging Utilities ---

class PipelineTimer:
    """
    A context manager to measure execution time of code blocks.
    """
    def __init__(self, name: str, active: bool = True):
        self.name = name
        self.active = active

    def __enter__(self):
        if self.active:
            self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.active:
            duration = (time.perf_counter() - self.start) # sec # * 1000  # ms
            if duration > 1.0:
                 logger.info(f"⏱️ {self.name}: {duration:.2f} sec") # changed debug->info to ensure visibility


class InferenceProfiler:
    """
    Tracks Execution Time and Peak VRAM during Inference for Pareto plotting.
    Supports both TensorFlow and PyTorch backends.
    """
    def __init__(self, model_name: str, results_dir: str):
        self.model_name = model_name
        self.results_dir = results_dir
        self.csv_path = os.path.join(results_dir, "inference_performance.csv")
        self.start_time = 0

        # Initialize CSV
        if not os.path.exists(self.csv_path):
            with open(self.csv_path, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Model', 'Volume_ID', 'Time_Seconds', 'Peak_VRAM_GB'])

    def start(self):
        # Reset Peak Memory Stats
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        try:
            tf.config.experimental.reset_memory_stats('GPU:0')
        except: pass

        self.start_time = time.perf_counter()

    def stop_and_log(self, vol_id: str):
        elapsed = time.perf_counter() - self.start_time

        # Gather Peak VRAM
        vram_gb = 0.0

        # Check PyTorch VRAM
        if torch.cuda.is_available():
            pt_vram = torch.cuda.max_memory_allocated() / (1024**3)
            vram_gb = max(vram_gb, pt_vram)

        # Check TensorFlow VRAM
        try:
            tf_info = tf.config.experimental.get_memory_info('GPU:0')
            tf_vram = tf_info['peak'] / (1024**3)
            vram_gb = max(vram_gb, tf_vram)
        except: pass

        with open(self.csv_path, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([self.model_name, vol_id, f"{elapsed:.2f}", f"{vram_gb:.2f}"])

        logger.info(f"[{self.model_name}] {vol_id} -> Time: {elapsed:.1f}s | Peak VRAM: {vram_gb:.2f}GB")


class MemoryTelemetry:
    def __init__(self, filename="memory_telemetry.csv"):
        self.filename = filename
        # Initialize CSV with headers if it doesn't exist
        if not os.path.exists(self.filename):
            with open(self.filename, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Event', 'Trial', 'RAM', 'VRAM'])

    def log(self, event: str, trial_num: int):
        # 1. System RAM (GB)
        ram_gb = psutil.virtual_memory().used / (1024 ** 3)

        # 2. GPU VRAM (GB)
        try:
            result = subprocess.check_output(
                ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,nounits,noheader'],
                encoding='utf-8'
            )
            # Sum VRAM across all GPUs (if multiple exist)
            vram_mb = sum(int(x.strip()) for x in result.strip().split('\n'))
            vram_gb = vram_mb / 1024.0
        except Exception:
            vram_mb = -1 # fallback if nvidia-smi fails
            vram_gb = -1

        with open(self.filename, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([event, trial_num, f"{ram_gb:.2f}", vram_gb])


def enforce_gpu_presence():
    """Protects against silent fallback to CPU training which destroys productivity."""
    gpus = tf.config.list_physical_devices('GPU')
    if not gpus:
        raise RuntimeError(
            "🚨 FATAL: No GPUs detected by TensorFlow! "
            "Aborting execution to prevent a massive waste of time on CPU."
        )
    logger.info(f"✅ Hardware Protection Pass: Verified {len(gpus)} GPU(s) available.")


def set_global_seeds(seed: int = 42, use_fp16=False, use_dmem=False):
    """Sets random seeds for reproducibility across Python, Numpy, and TF."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    logger.info(f"Global seed set to {seed}")

    # Enable Mixed Precision for VRAM efficiency
    if use_fp16:
        policy = mixed_precision.Policy('mixed_float16')
        mixed_precision.set_global_policy(policy)
    logger.info(f"Mixed Precision Policy = {mixed_precision.global_policy()}")

    # TITAN RTX Tensor cores workaround
    if use_dmem:
        gpus = tf.config.list_physical_devices('GPU')

        if gpus:
            try:
                for gpu in gpus:
                    logger.info(f"Setting dynamic memory growth for {gpu}...")
                    tf.config.experimental.set_memory_growth(gpu, True)
            except RuntimeError as e:
                print(e)


# Initialize environment
if not KAGGLE:
    enforce_gpu_presence()  # <--- DROP IN HERE
set_global_seeds(42)
logger.info(f"Process ID = {os.getpid()}")
logger.info(f"Python version = {python_version()}")

# Instantiate globally
telemetry = MemoryTelemetry()

In [ ]:
import glob
import re

from tqdm import tqdm


# Constants for file filtering (from original script)
EXCLUDE_FILES = set([
    'wmIXI338-HH-1971-MADisoTFE1_-s3T188_-0301-00003-000001-01.nii',
    'wmIXI357-HH-2076-MADisoTFE1_-s3T199_-0301-00003-000001-01.nii',
    'wmIXI402-Guys-0961-MPRAGESEN_-s413_-0301-00003-000001-01.nii',
    'wmIXI423-IOP-0974-SAGFSPGR_-sIXI00_-0003-00001-000001-01.nii',
])

def get_subject_id_from_path(path: str) -> str:
    """Extracts subject ID (e.g., IXI306) from filename."""
    filename = os.path.basename(path)
    match = re.search(r'IXI\d{3}', filename)
    if match:
        return match.group(0)
    return filename

def find_t1_files(root: str) -> List[str]:
    """Recursive search for wm*.nii files (IXI/OASIS)."""
    # Pattern matches the Kaggle 'RawT1' pattern from your original script
    pattern = os.path.join(root, '**', 'wm*.nii')
    matches = glob.glob(pattern, recursive=True)

    valid_matches = [
        p for p in matches
        if os.path.getsize(p) > 1024
        and os.path.basename(p) not in EXCLUDE_FILES
    ]
    logger.info(f"Found {len(valid_matches)} valid T1 volumes in {root}")
    return sorted(valid_matches)

def get_brats_subjects(root: str) -> List[Dict]:
    """Scans BraTS directory for T1/Seg pairs."""
    subjects = []
    # Search for T1 files
    t1_files = glob.glob(os.path.join(root, '**', '*_t1.nii'), recursive=True)

    for t1_path in sorted(t1_files):
        if os.path.getsize(t1_path) <= 1024: continue

        folder = os.path.dirname(t1_path)
        subject_id = os.path.basename(t1_path).replace('_t1.nii', '')

        # Standard BraTS naming: ID_seg.nii
        seg_path = os.path.join(folder, f"{subject_id}_seg.nii")

        if not os.path.exists(seg_path):
            # Fallback search
            alts = glob.glob(os.path.join(folder, '*seg*.nii'))
            seg_path = alts[0] if alts else None

        subjects.append({
            'id': subject_id,
            't1': t1_path,
            'seg': seg_path
        })

    logger.info(f"Found {len(subjects)} BraTS subjects in {root}")
    return subjects

def make_slice_index_list(vol_paths: List[str], neighborhood: int = 3,
                          batch_mode=False) -> List[Tuple[str, int, str]]:
    """
    Scans headers to build a list of valid (Volume, SliceIndex, Direction) samples.
    """
    idxs = []
    logger.info(f"Indexing slices for {len(vol_paths)} volumes...")

    for p in tqdm(vol_paths, desc="Indexing", disable=batch_mode):
        try:
            # Fast header check for Z-dim
            # Note: We rely on the DataManager's logic that Z is usually the smallest dim < 128
            # But header read is strictly (X,Y,Z).
            # If our heuristic in DataManager transposes, we need to know WHICH dim becomes Z
            img = nib.load(p)
            shape = img.shape

            # Replicate DataManager heuristic to find Z dimension size
            z_dim = shape[2] # Default NIfTI
            if len(shape) == 3 and shape[2] < shape[0] and shape[2] < 128:
                z_dim = shape[2] # Z is last (standard)
            elif shape[0] < 128:
                z_dim = shape[0] # Z is first
            else:
                z_dim = shape[2] # Fallback

        except Exception as e:
            logger.warning(f"Skipping {os.path.basename(p)}: {e}")
            continue

        # Forward samples: [i, i+1, i+2] -> predict i+3
        # Valid if i+3 < z_dim
        # So max i = z_dim - 1 - neighborhood
        limit_fwd = z_dim - neighborhood - 1
        for i in range(limit_fwd + 1):
            idxs.append((p, i, 'forward'))

        # Backward samples: [i, i-1, i-2] -> predict i-3
        # Valid if i-3 >= 0
        # So min i = neighborhood
        for i in range(neighborhood, z_dim):
            idxs.append((p, i, 'backward'))

    return idxs

# Cell 2: The Configuration Class
- **Target File:** configs/config.py
- **Content:**
    - ModelConfig, DataConfig, AugmentationConfig, TrainingConfig dataclasses
    - Config master class
    - Global CFG instantiation (logic for HPO integration)

In [ ]:
@dataclass
class ModelConfig:
    """Hyperparameters for the Neural Network Architecture."""
    name: str = '2p5d'              # prefix, e. g. 2p5d_unet, 2p5d_spade, etc
    architecture: str = 'spade'     # 'unet', 'spade', 'vae' etc (TODO: 'vae', 'gan', etc)
    backbone: Optional[str] = 'B0'  # 'B0'...'B7' for EfficientNet, or None for custom
    use_attention: bool = False     # use Attention Gates in Decoder
    use_stem: bool = False          # use initial Conv stem
    dropout_rate: float = 0.1
    activation: str = 'gelu'
    # Input channels: Neighborhood (slices) + 1 (Mask, optional)
    input_channels: int = 3 + 0
    base_filters: int = 64          # replaces hardcoded 32/64 starts
    use_positional_encoding: bool = False
    num_hypotheses: int = 1      # set to 1 for baseline, >1 (e.g., 4) for MHP
    mhp_epsilon: float = 0.05    # relaxation parameter to prevent dead heads (https://arxiv.org/abs/1612.00197)


@dataclass
class DataConfig:
    """Settings for Data Loading, Caching, and Preprocessing."""
    # Paths
    data_root_ixi: str = PATH_DATA_IXI
    data_root_brats: str = PATH_DATA_BRATS
    ixi_sampling_weight: float = 1.0  # 70% IXI, 30% BraTS by default

    # Dimensions
    # The native patch size found in EDA (approximate)
    native_patch_size: Tuple[int, int] = (113, 137)
    # The size fed into the model (multiples of 32 for UNet/EffNet)
    # padded_size: Tuple[int, int] = (128, 160)
    # Increased to square to allow safe rotation
    padded_size: Tuple[int, int] = (160, 160)
    # Number of slices for 2.5D context (Must be odd: t-1, t, t+1)
    neighborhood: int = 3
    # Whether to append the target mask as an input channel
    input_last_target_mask: bool = True

    # RAM Management
    # Capacity: How many full volumes to keep in RAM at once.
    # Increasing this improves diversity but requires more RAM

    # Projections to train on.
    # 0: Axial (standard), 1: Coronal, 2: Sagittal
    projections: Tuple[int, ...] = (0,) # (0, 1, 2) for all three

    # Caching (RAM Management)
    # BraTS takes ~2x RAM per volume due to segmentation masks
    ixi_cache_size: int = 120 if KAGGLE else 240
    brats_cache_size: int = 120 if KAGGLE else 240
    # Number of slices in the replay buffer
    # On Kaggle we sample 128 * 3 * 5 = 1920 (last 5 epochs);
    # on cluster we sample 384 * 3 * 7 = 8064 (last 7 epochs)
    hallucination_buffer_size: int = 1920 if KAGGLE else 8064 # 5760 for the last 5 epochs

    # ---> NEW: Centralized Ablation Path <---
    weights_dir: str = '/kaggle/input/brain-reconstruction-pretrain/ablations-B0' if KAGGLE else 'ablations'
    # preload_dir: str = '/kaggle/input/brain-reconstruction-pretrain/ablations-latest' if KAGGLE else 'ablations-latest'
    preload_dir: str = '/kaggle/input/notebooks/valvex/brain-reconstruction-brats/ablation_results' if KAGGLE else 'ablations-latest'
    results_dir: str = 'ablation_results'
    figures_dir: str = 'paper_figures'


@dataclass
class AugmentationConfig:
    """Probabilities and params for Data Augmentation."""
    # Tumor Injection
    prob_glioma: float = 0.65
    brats_labels_to_keep: Tuple[int, ...] = (0,)       # keep background, replace others
    brats_labels_for_void: Tuple[int, ...] = (1, 2, 4) # tumor labels

    # Photometric / Corruption
    prob_autoregressive_candidate: float = 0.65 # chance a slice is corruptible
    # prob_blur: float = 0.25 # asymmetric blur
    # prob_noise: float = 0.25 # asymmetric noise
    blur_sigma: float = 0.85
    noise_std: float = 0.025
    prob_hallucination_max: float = 0.65   # maximum possible hallucination probability
    prob_hallucination_replay: float = 0.0 # starts at 0, ramps up via callback
    prob_hallucination_warmup: int = 5     # epochs to increase aug prob to max

    # Geometric
    prob_flip: float = 0.5          # horizontal flip
    prob_rotate: float = 0.35       # rotation
    rotate_range: float = 3.5       # degrees (+/-) 45 is too aggressive for brain fitting


@dataclass
class TrainingConfig:
    """Hyperparameters for the Training Loop."""
    batch_size: int = 64
    epochs: int = 20 if KAGGLE else 60 # 80 for U-Net
    # U-Net
    learning_rate: float = np.pi * 1e-4
    # Loss Weights (CompositeLoss)
    # --- Balanced (HPO) coefficients ---
    lambda_tumor: float = 0.225       # MAE inside tumor
    lambda_healthy: float = 0.268     # MAE healthy tissue
    lambda_grad: float = 0.416        # Gradient loss (sharpness)
    lambda_background: float = 0.727  # MAE background
    lambda_perceptual: float = 0.0003 # VGG/EffNet feature loss
    lambda_spectral: float = 0.078    # Fourier feature loss
    # --- Texture-forced (heuristic) coefficients ---
    # lambda_tumor: float = 0.385       # MAE inside tumor
    # lambda_healthy: float = 0.415     # MAE healthy tissue
    # lambda_grad: float = 1.000        # Gradient loss (sharpness)
    # lambda_background: float = 0.215  # MAE background
    # lambda_perceptual: float = 0.0003 # VGG/EffNet feature loss
    # lambda_spectral: float = 0.105    # Fourier feature loss
    # Alternative
    use_spatial_loss: bool = False  # toggle to switch between CompositeLoss and SpatialL1 (U-Net)
    # TRAINING MODE FLAG
    # If True: Uses SPADEGANTrainer (Discriminator + Adversarial Loss)
    # If False: Uses Standard model.fit (L1/Composite Loss only)
    gan_mode: bool = False
    # GAN Weights
    learning_rate_g: float = 1e-4   # generator
    learning_rate_d: float = 1e-4   # discriminator
    weight_gan: float = 1.5         # adversarial
    weight_fm: float = 0.45         # feature matching
    weight_l1: float = 1.75         # spatial reconstruction
    weight_perceptual: float = 0.25 # VGG/EffNet perceptual
    weight_spectral: float = 0.25   # VGG/EffNet perceptual
    weight_kl: float = 0.05         # standard starting point for VAE-GANs (0.01 to 0.1)
    # Loss
    perceptual_backbone: str = 'effnet' # 'vgg' or 'effnet'
    perceptual_init: str = 'imagenet'   # 'imagenet' or None
    # Model saving and resuming
    resume_training: str = None


@dataclass
class HPOConfig:
    n_trials: int = 12 if not KAGGLE else 12
    # Increase steps for better statistical significance
    train_steps_per_trial: int = 200
    val_steps_per_trial: int = 50

    # --- New: HPO Data Supply Proportions ---
    base_train_volumes: int = 160  # total training volumes in RAM during HPO
    base_val_volumes: int = 40     # total validation volumes in RAM during HPO
    base_eval_volumes: int = 20    # total evaluation volumes for Autoregressive scoring

    # --- New: HPO Evaluation Metrics ---
    ar_rollout_steps: int = 5      # steps to predict into the future for AR score
    metric_ar_weight: float = 0.65 # weight of AR score vs. One-Shot Validation Loss (0.0 to 1.0)

    # Output
    best_params_file: str = "best_hyperparams.json"

    # Distributed HPO settings
    study_name: str = "scratchnet"
    storage_dir: str = "./databases"  # HPO storage where to save sqlite files
    database_name: str = "hpo_final.db"
    # database_preload: str = '/kaggle/input/notebooks/valvex/brain-reconstruction-brats/hpo_final.db'
    database_preload: str = ''
    tracemalloc: bool = False


@dataclass
class Config:
    """Master Configuration Object."""
    model: ModelConfig = field(default_factory=ModelConfig)
    data: DataConfig = field(default_factory=DataConfig)
    aug: AugmentationConfig = field(default_factory=AugmentationConfig)
    train: TrainingConfig = field(default_factory=TrainingConfig)
    hpo: HPOConfig = field(default_factory=HPOConfig)

    run_type: str = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Interactive')
    batch_mode: bool = False # Interactive by default
    seed: int = 42

    def __post_init__(self):
        self.model.name = f"{self.model.name}_{self.model.architecture}"
        self.batch_mode = self.run_type != 'Interactive'
        # Interactive mode adjustment
        if self.run_type == 'Interactive':
            logger.info("⚠️ Interactive mode detected. Reducing Epochs.")
            self.train.epochs = 5 # debug
        # Update channel count based on data settings
        self.model.input_channels = self.data.neighborhood + int(self.data.input_last_target_mask)
        if not KAGGLE:
            self.hpo.database_preload = ''

    def to_dict(self):
        return asdict(self)

    def save(self, path):
        with open(path, 'w') as f:
            json.dump(asdict(self), f, indent=4)

    @classmethod
    def load(cls, path):
        with open(path, 'r') as f:
            data = json.load(f)
        # Helper to recursively map dict to dataclasses
        # (Simplified logic, assuming standard structure)
        cfg = cls()
        for section, params in data.items():
            if hasattr(cfg, section):
                sec_obj = getattr(cfg, section)
                for k, v in params.items():
                    setattr(sec_obj, k, v)
        return cfg


# Instantiate the global config
CFG = Config()

logger.info(f"Configuration loaded. Model Input Shape: {CFG.data.padded_size} x {CFG.model.input_channels}")

# Cell 3: Unified Geometry (TensorFlow Native)
- **Target File:** src/core/geometry.py
- **Content:**
    - GeometryOps class
    - get_smart_crop_coords (Numpy-based)
    - normalize_volume (TF-based)
    - resize_and_pad and inverse_resize_pad (TF-based letterboxing logic)

In [ ]:
class GeometryOps:
    """
    A collection of pure TensorFlow and Numpy geometry operations.
    Designed to be used within tf.data pipelines or inference loops.
    """

    @staticmethod
    def get_smart_crop_coords(volume: np.ndarray,
                              target_shape: Tuple[int, int]
                             ) -> Tuple[Tuple[int, int], Tuple[int, int]]:
        """
        Calculates the bounding box to center the brain in the frame.
        Uses Numpy/Scipy (CPU) as this runs once during volume loading.

        Args:
            volume: 3D Numpy array (Z, H, W)
            target_shape: (H_out, W_out)

        Returns:
            ((h_start, h_end), (w_start, w_end))
        """
        # 1. Projection to find brain content
        projection = np.max(volume, axis=0) > 0.01

        # 2. Find bounding box
        slices = find_objects(projection.astype(int))

        if not slices:
            # Fallback: Center crop
            h, w = volume.shape[1], volume.shape[2]
            ch, cw = h // 2, w // 2
        else:
            h_slice, w_slice = slices[0]
            ch = h_slice.start + (h_slice.stop - h_slice.start) // 2
            cw = w_slice.start + (w_slice.stop - w_slice.start) // 2

        # 3. Calculate Bounds
        th, tw = target_shape
        h_start = ch - th // 2
        w_start = cw - tw // 2

        # 4. Clip to image boundaries
        max_h, max_w = volume.shape[1], volume.shape[2]

        # Adjust if out of bounds (left/top)
        if h_start < 0: h_start = 0
        if w_start < 0: w_start = 0

        # Adjust if out of bounds (right/bottom) - shift back
        if h_start + th > max_h: h_start = max(0, max_h - th)
        if w_start + tw > max_w: w_start = max(0, max_w - tw)

        return (int(h_start), int(h_start + th)), (int(w_start), int(w_start + tw))

    @staticmethod
    @tf.function
    def normalize_volume(vol_tensor: tf.Tensor) -> tf.Tensor:
        """
        Min-Max normalization robust to outliers (TF Graph compatible).
        """
        # Cast to float32
        vol_tensor = tf.cast(vol_tensor, tf.float32)

        min_val = tf.reduce_min(vol_tensor)
        vol_tensor = vol_tensor - min_val

        # Use a safe division
        max_val = tf.reduce_max(vol_tensor)
        scale = tf.maximum(max_val, 1e-8)

        return vol_tensor / scale

    @staticmethod
    @tf.function
    def resize_and_pad(
        image: tf.Tensor,
        target_shape: Tuple[int, int],
        method: str = 'bicubic'
    ) -> Tuple[tf.Tensor, tf.Tensor, tf.Tensor]:
        """
        Resizes image to fit target_shape preserving aspect ratio (Letterbox),
        then pads the rest.

        Args:
            image: (H, W, C) or (H, W) Tensor
            target_shape: (H_out, W_out)

        Returns:
            image_padded: Resized and padded image
            scale: float scale factor used
            padding: (pad_top, pad_left)
        """
        # Ensure 3D (H, W, C) for resize ops
        input_shape = tf.shape(image)
        h, w = tf.cast(input_shape[0], tf.float32), tf.cast(input_shape[1], tf.float32)
        target_h, target_w = tf.cast(target_shape[0], tf.float32), tf.cast(target_shape[1],
                                                                           tf.float32)

        # Calculate Scale
        scale = tf.minimum(target_h / h, target_w / w)

        new_h = tf.cast(h * scale, tf.int32)
        new_w = tf.cast(w * scale, tf.int32)

        # Resize
        # image_resized = tf.image.resize(image, [new_h, new_w], method='bicubic')
        image_resized = tf.image.resize(image, [new_h, new_w], method=method)

        # Calculate Padding
        pad_h = tf.cast(target_h, tf.int32) - new_h
        pad_w = tf.cast(target_w, tf.int32) - new_w

        pad_top = pad_h // 2
        pad_bottom = pad_h - pad_top
        pad_left = pad_w // 2
        pad_right = pad_w - pad_left

        # Apply Padding
        if len(image.shape) == 3:
            paddings = [[pad_top, pad_bottom], [pad_left, pad_right], [0, 0]]
        else:
            paddings = [[pad_top, pad_bottom], [pad_left, pad_right]]

        image_padded = tf.pad(image_resized, paddings, constant_values=0.0)

        # Return aux info for inverse operation
        return image_padded, scale, tf.stack([pad_top, pad_left])

    @staticmethod
    @tf.function
    def inverse_resize_pad(image: tf.Tensor,
                           original_shape: Tuple[int, int],
                           scale: float,
                           pads: tf.Tensor) -> tf.Tensor:
        """
        Reverses the resize_and_pad operation (Crop -> Inverse Resize).
        Useful for reconstructing full volumes from model outputs.
        """
        pad_top = pads[0]
        pad_left = pads[1]

        orig_h, orig_w = original_shape[0], original_shape[1]

        # 1. Crop Center (Remove Padding)
        # Calculate height/width of the actual content inside the padded image
        active_h = tf.cast(tf.cast(orig_h, tf.float32) * scale, tf.int32)
        active_w = tf.cast(tf.cast(orig_w, tf.float32) * scale, tf.int32)

        image_cropped = tf.image.crop_to_bounding_box(image, pad_top, pad_left, active_h, active_w)

        # 2. Resize Up (Bicubic usually better for upsampling)
        image_restored = tf.image.resize(image_cropped, [orig_h, orig_w], method='bicubic')

        return image_restored

In [ ]:
# --- Test the Geometry Ops (Sanity Check) ---
with PipelineTimer("GeometryOps Test"):
    # Mock data
    dummy_vol = np.random.rand(113, 137, 1).astype(np.float32)
    dummy_tensor = tf.convert_to_tensor(dummy_vol)

    # 1. Test Normalize
    norm_tensor = GeometryOps.normalize_volume(dummy_tensor)

    # 2. Test Resize/Pad
    padded, scale, pads = GeometryOps.resize_and_pad(norm_tensor, CFG.data.padded_size)

    # 3. Test Inverse
    restored = GeometryOps.inverse_resize_pad(padded, (113, 137), scale, pads)

    logger.info(f"Original Shape: {dummy_tensor.shape}")
    logger.info(f"Padded Shape: {padded.shape} (Target: {CFG.data.padded_size})")
    logger.info(f"Restored Shape: {restored.shape}")
    logger.info("Geometry Sanity Check Passed.")

In [ ]:
from matplotlib import pyplot as plt


def analyze_dataset_geometry(data_list, num_samples=3):
    """
    Loads samples, enforces Canonical Orientation (RAS+), and plots views.
    Works for both IXI (list of str) and BraTS (list of dicts).
    """
    print(f"--- Analyzing {num_samples} samples ---")

    # Handle mixed input types
    valid_list = [d['t1'] if isinstance(d, dict) else d for d in data_list]
    valid_list = [p for p in valid_list if p] # filter None

    samples = random.sample(valid_list, min(len(valid_list), num_samples))

    for path in samples:
        filename = os.path.basename(path)
        print(f"\nFile: {filename}")

        # 1. Load Raw
        img = nib.load(path)
        print(f"  Raw Shape: {img.shape}")

        # 2. Enforce Canonical (RAS+)
        # This aligns axes to: 0=Left-Right, 1=Posterior-Anterior, 2=Inferior-Superior
        canonical_img = nib.as_closest_canonical(img)
        can_data = canonical_img.get_fdata(dtype=np.float32)
        print(f"  Canonical (RAS+): {can_data.shape} [X=Sag, Y=Cor, Z=Axial]")

        # 3. Transpose to Logical (Z, Y, X) for Training
        # We want the "Depth" (Slices) to be the first axis.
        # Usually Axial (Z) is the slice axis.
        # Transpose (2, 1, 0) -> (Axial, Coronal, Sagittal)
        std_data = np.transpose(can_data, (2, 1, 0))
        print(f"  Training Input (Z,Y,X): {std_data.shape}")

        # 4. Crop Analysis
        mask = std_data > np.mean(std_data) * 0.1
        if np.any(mask):
            coords = np.argwhere(mask)
            bbox_shape = coords.max(axis=0) - coords.min(axis=0) + 1
            print(f"  Brain Content Size: {bbox_shape}")

        # 5. Visualize
        c_z, c_y, c_x = np.array(std_data.shape) // 2

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        fig.suptitle(f"{filename} (Canonical -> Transposed 2,1,0)", fontsize=10, y=0.98)

        # View 0: Slice along Axis 0 (Z-axis / Axial)
        # Result is Y-X plane (Coronal-Sagittal axes) -> Axial Image
        axes[0].imshow(std_data[c_z, :, :], cmap='bone', aspect='equal')
        axes[0].set_title(f"Axis 0 (Index {c_z})\nPlane: Axial")

        # View 1: Slice along Axis 1 (Y-axis / Coronal)
        # Result is Z-X plane
        axes[1].imshow(std_data[:, c_y, :], cmap='bone', aspect='equal') # no rotation needed if layout is Z-Y-X
        axes[1].set_title(f"Axis 1 (Index {c_y})\nPlane: Coronal")

        # View 2: Slice along Axis 2 (X-axis / Sagittal)
        # Result is Z-Y plane
        axes[2].imshow(std_data[:, :, c_x], cmap='bone', aspect='equal')
        axes[2].set_title(f"Axis 2 (Index {c_x})\nPlane: Sagittal")

        # FIX: rect=[left, bottom, right, top]
        # Restricts subplots to lower 90% of figure, leaving room for suptitle
        plt.tight_layout(rect=[0, 0.03, 1, 0.90])
        plt.show()

# Cell 4: Unified Input Processing
- **Target File:** src/core/geometry.py
- **Content:**
    - InputProcessor class

In [ ]:
class InputProcessor:
    @staticmethod
    def prepare_model_input(input_stack_native: np.ndarray,
                            target_slice_native: np.ndarray,
                            start_idx: int,
                            target_idx: int,
                            axis_size: int,
                            direction: str,
                            void_mask_native: np.ndarray = None,
                            config: Config = None):
        N = config.data.neighborhood

        # 1. Handle History Image Stack (Ensure H, W, N)
        if input_stack_native.shape[0] == N and input_stack_native.shape[2] != N:
             input_stack_native = np.transpose(input_stack_native, (1, 2, 0))

        # 2. Handle Target Mask (1 Channel)
        brain_mask = (target_slice_native > 0.01).astype(np.float32)[..., None]

        # 3. Positional Encodings (Absolute)
        p_abs = np.array([target_idx / max(1.0, float(axis_size - 1))], dtype=np.float32)

        step = 1 if direction == 'forward' else -1
        p_history = []
        for i in range(N):
            current_slice_idx = start_idx + (i * step)
            norm_pos = current_slice_idx / max(1.0, float(axis_size - 1))
            p_history.append(norm_pos)

        p_history_arr = np.array(p_history, dtype=np.float32)

        # RETURN PURE NUMPY DICT (No resizing here!)
        model_inputs = {
            'history_input': input_stack_native.astype(np.float32),
            'mask_input': brain_mask,
            'p_history_input': p_history_arr,
            'p_abs_input': p_abs
        }

        return model_inputs, 1.0, np.array([0, 0]) # mock scale/pads for generator compatibility

# Cell 5: Data Manager with Mask Caching
- **Target File:** src/data/cache.py
- **Content:**
    - VolumeCache class (Thread-safe LRU cache)
    - DataManager class (Handles IXI/BraTS loading, normalization, and the Hallucination Buffer)

In [ ]:
import threading
import time
import collections
import random
import nibabel as nib
import numpy as np

from abc import ABC, abstractmethod
from collections import Counter, defaultdict


# --- 0. Statistics Container ---
class DataStatistics:
    """
    Tracks data access patterns to verify sampling uniformity.
    """
    def __init__(self):
        # Counter Key: (dataset_name, volume_id, axis, slice_idx)
        # We track direction separately or fold it in
        self.hits = Counter()
        self.volume_map = {} # id -> path (for sorting)
        self.lock = threading.Lock()

    def record(self, dataset, vol_id, axis, slice_idx, direction):
        with self.lock:
            self.hits[(dataset, vol_id, axis, slice_idx, direction)] += 1

    def register_volume(self, vol_id, path):
        self.volume_map[vol_id] = path

    def reset(self):
        self.stats.clear()

# --- 1. Unified Container ---
class MedicalVolume:
    """Unified container for IXI and BraTS data."""
    def __init__(self, t1_data: np.ndarray, seg_data: np.ndarray = None,
                 path: str = ""):
        self.t1 = t1_data
        self.seg = seg_data
        self.path = path
        self.shape = t1_data.shape

        self.indices = {
            'any': {0:[], 1:[], 2:[]},
            'clean': {0:[], 1:[], 2:[]},
            'tumor': {0:[], 1:[], 2:[]}
        }
        self._analyze()

    def _analyze(self):
        for axis in [0, 1, 2]:
            reduce_axes = tuple(i for i in range(3) if i != axis)
            tissue_map = (
                np.sum(self.t1, axis=reduce_axes) >
                (self.shape[reduce_axes[0]] *
                 self.shape[reduce_axes[1]] * 0.01)
            )
            self.indices['any'][axis] = np.where(tissue_map)[0]

            if self.seg is not None:
                tumor_map = np.max(self.seg, axis=reduce_axes) > 0
                self.indices['tumor'][axis] = np.where(tumor_map)[0]
                self.indices['clean'][axis] = np.where(tissue_map & (~tumor_map))[0]
            else:
                self.indices['clean'][axis] = self.indices['any'][axis]

# --- 2. The Universal Loader ---
class VolumeLoader:
    """Single source of truth for loading and preprocessing."""
    @staticmethod
    def load(t1_path: str, seg_path: str = None) -> Optional[MedicalVolume]:
        try:
            # 1. Load & Enforce Canonical
            t1_img = nib.load(t1_path)
            t1_img = nib.as_closest_canonical(t1_img)
            t1_data = t1_img.get_fdata(dtype=np.float32)

            # 2. Transpose (X,Y,Z) -> (Z,Y,X)
            t1_data = np.transpose(t1_data, (2, 1, 0))

            # 3. Load Seg
            seg_data = None
            if seg_path:
                seg_img = nib.load(seg_path)
                seg_img = nib.as_closest_canonical(seg_img)
                seg_data = seg_img.get_fdata(dtype=np.float32)
                seg_data = np.transpose(seg_data, (2, 1, 0))

            # 4. Crop to Content
            mask = t1_data > np.mean(t1_data) * 0.1
            if np.any(mask):
                coords = np.argwhere(mask)
                z_min, y_min, x_min = coords.min(axis=0)
                z_max, y_max, x_max = coords.max(axis=0) + 1

                t1_data = t1_data[z_min:z_max, y_min:y_max, x_min:x_max]
                if seg_data is not None:
                    seg_data = seg_data[z_min:z_max, y_min:y_max, x_min:x_max]

            # 5. Normalize
            p99 = np.percentile(t1_data, 99)
            if p99 > 0:
                t1_data = np.clip(t1_data, 0, p99) / p99

            return MedicalVolume(t1_data, seg_data, path=t1_path)

        except Exception as e:
            # logger.warning(f"Load Failed {t1_path}: {e}") # logger needs import or pass
            return None

# --- 3. Manager Base ---
class DataLoaderBase(ABC):
    def __init__(self, config):
        self.cfg = config
        self.hallucination_buffer = collections.OrderedDict()
        self.lock = threading.Lock()
        self.stats = DataStatistics()
        self.pool = {}  # initialized here to guarantee existence for subclasses

    @abstractmethod
    def get_volume(self, collection: str) -> Optional[MedicalVolume]: pass

    def update_hallucination(self, key, data):
        with self.lock:
            self.hallucination_buffer[key] = data
            if len(self.hallucination_buffer) > self.cfg.data.hallucination_buffer_size:
                self.hallucination_buffer.popitem(last=False)

    def get_hallucination(self, key):
        with self.lock: return self.hallucination_buffer.get(key)

    def reset_buffer(self):
        """Clears the hallucination buffer to prevent state leakage between HPO trials."""
        with self.lock:
            self.hallucination_buffer.clear()

    def get_matched_glioma_stack(self, axis: int, target_norm_pos: float,
                                 neighborhood: int, tolerance: float = 0.05
                                ) -> Optional[np.ndarray]:
        """
        Searches the RAM pool for a BraTS volume containing a glioma at the specified
        normalized anatomical position (+/- tolerance) along the given projection axis.
        Returns the (N+1, H, W) segmentation stack, or None if no match is found.
        """
        with self.lock:
            # Safely grab the current BraTS pool regardless of ActiveLoader (dict) or StaticLoader (list)
            brats_pool = self.pool.get('brats', {})
            if isinstance(brats_pool, dict):
                brats_vols = list(brats_pool.values())
            else:
                brats_vols = list(brats_pool)

        if not brats_vols:
            return None

        # Shuffle volumes to ensure we don't stamp the exact same tumor every time
        random.shuffle(brats_vols)

        for vol in brats_vols:
            if vol.seg is None:
                continue

            # Look up which indices actually contain a tumor for this projection axis
            tumor_indices = vol.indices['tumor'][axis]
            if len(tumor_indices) == 0:
                continue

            axis_size = vol.shape[axis]
            candidate_starts =[]

            # Find matching slices within tolerance
            for t_idx in tumor_indices:
                norm_pos = t_idx / max(1.0, float(axis_size - 1))

                if abs(norm_pos - target_norm_pos) <= tolerance:
                    start_idx = t_idx - neighborhood

                    # Verify sequence boundaries: we need [t_idx - neighborhood, t_idx]
                    if start_idx >= 0 and (start_idx + neighborhood) < axis_size:
                        candidate_starts.append(start_idx)

            if candidate_starts:
                # Pick a random valid start position from this specific volume
                start_idx = random.choice(candidate_starts)

                # Orient the view according to the projection axis
                if axis == 0:   view = vol.seg
                elif axis == 1: view = np.moveaxis(vol.seg, 1, 0)
                else:           view = np.moveaxis(vol.seg, 2, 0)

                try:
                    # Extract precisely N + 1 slices
                    seg_stack = view[start_idx : start_idx + neighborhood + 1]
                    if seg_stack.shape[0] == neighborhood + 1:
                        return seg_stack
                except Exception as e:
                    # Catch boundary slicing errors silently and try the next volume
                    continue

        # No volume in the current RAM pool had a tumor at this specific relative depth
        return None


# --- 4. Static Loader ---
class StaticLoader(DataLoaderBase):
    def __init__(self, config, ixi_files, brats_list):
        super().__init__(config)
        self.pool = {'ixi': [], 'brats': []}

        # Load sequentially
        for f in ixi_files:
            vol = VolumeLoader.load(f)
            if vol: self.pool['ixi'].append(vol)

        for b in brats_list:
            vol = VolumeLoader.load(b['t1'], b['seg'])
            if vol: self.pool['brats'].append(vol)

    def get_volume(self, collection: str):
        if not self.pool[collection]: return None
        return random.choice(self.pool[collection])


# --- 5. Active Loader ---
class ActiveLoader(DataLoaderBase):
    def __init__(self, config, ixi_files, brats_list):
        super().__init__(config)
        self.files = {}
        if ixi_files: self.files['ixi'] = ixi_files
        if brats_list: self.files['brats'] = brats_list

        self.pool = {k: {} for k in self.files.keys()}
        self.keys = {k: [] for k in self.files.keys()}
        self.active = False
        self.thread = None

    def __enter__(self): self.start(); return self
    def __exit__(self, *args): self.stop()

    def start(self):
        if self.active: return
        self.active = True
        self.thread = threading.Thread(target=self._worker, daemon=True)
        self.thread.start()
        # Warmup
        while len(self.pool['ixi']) < min(5, self.cfg.data.ixi_cache_size // 10):
            time.sleep(1)

    def stop(self):
        self.active = False
        if self.thread: self.thread.join(timeout=2)

    def get_volume(self, collection):
        with self.lock:
            if not self.keys[collection]: return None
            key = random.choice(self.keys[collection])
            return self.pool[collection][key]

    def _worker(self):
        # 0. Helper to extract ID based on dataset type
        def get_id(item):
            return item['id'] if isinstance(item, dict) else os.path.basename(item)

        # 1. Prepare index counters only for active datasets
        indices = {k: 0 for k in self.files.keys()}
        for k in self.files:
            random.shuffle(self.files[k])

        while self.active:
            # 2. Load from each available dataset to keep the cache full.
            # We don't need modulo math here;
            # the Dataset pipeline handles the probability weights
            for col in self.files.keys():
                self._load_next(col, indices, key_fn=get_id)

            # 3. Simple throttler
            if (sum(len(p) for p in self.pool.values()) >=
                self.cfg.data.ixi_cache_size + self.cfg.data.brats_cache_size):
                time.sleep(0.1)
            else:
                time.sleep(0.01)

    def _load_next(self, col, indices, key_fn):
        idx = indices[col]
        item = self.files[col][idx]
        indices[col] = (idx + 1) % len(self.files[col])

        if col == 'ixi': args = (item, None)
        else:            args = (item['t1'], item['seg'])

        vol = VolumeLoader.load(*args)
        if vol:
            key = key_fn(item)
            with self.lock:
                limit = self.cfg.data.ixi_cache_size if col == 'ixi' else self.cfg.data.brats_cache_size
                if len(self.pool[col]) >= limit:
                    rem = self.keys[col].pop(0)
                    del self.pool[col][rem]
                self.pool[col][key] = vol
                self.keys[col].append(key)

# Cell 6 (Updated): TF-Based Augmentation Logic
- **Target File:** src/data/augmentations.py (Append to file)
- **Content:**
    - AugmentationLogic class
    - High-level logic: apply_autoregressive_corruption (cascading noise) and apply_tumor_void (mask injection)

In [ ]:
from scipy.ndimage import gaussian_filter, zoom, rotate


class AugmentationLogic:
    """
    Fast NumPy/SciPy implementations for CPU-based Data Generators.
    Eliminates TF-Eager execution overhead during data loading.
    """
    @staticmethod
    def apply_geometric(input_stack, target_slice, target_masks, config):
        """
        Applies consistent Flip and Rotation to inputs and targets.
        input_stack: (C, H, W)
        target_slice: (H, W)
        target_masks: list of (H, W) masks (tumor, brain_mask)
        """
        # 1. Flip Left-Right (Axis 2)
        if random.random() < config.prob_flip:
            # Numpy flip is fast
            input_stack = np.flip(input_stack, axis=2)
            target_slice = np.flip(target_slice, axis=1)
            target_masks = [np.flip(m, axis=1) for m in target_masks]

        # 2. Rotation
        if random.random() < config.prob_rotate:
            angle = random.uniform(-config.rotate_range, config.rotate_range)

            # Rotate Input (C, H, W) - Rotate axes 1, 2
            input_stack = rotate(input_stack, angle, axes=(1, 2), reshape=False,
                                 order=1, mode='constant', cval=0.0)

            # Rotate Target (H, W) - Standard
            target_slice = rotate(target_slice, angle, axes=(0, 1), reshape=False,
                                  order=1, mode='constant', cval=0.0)

            # Rotate Masks (H, W) - Nearest Neighbor
            target_masks = [
                rotate(m, angle, axes=(0, 1), reshape=False, order=0,
                       mode='constant', cval=0.0)
                for m in target_masks
            ]

        return input_stack, target_slice, target_masks

    @staticmethod
    def apply_autoregressive_corruption(input_stack: np.ndarray,
                                        config: AugmentationConfig
                                       ) -> Tuple[np.ndarray, Dict[str, bool]]:
        stack = input_stack.copy()
        flags = {'blur_applied': False, 'noise_applied': False}
        num_slices = stack.shape[0] # [t-N ... t-1]

        # Decide if the SEQUENCE is corrupted starting from the most recent
        # We iterate backwards: t-1 -> t-N

        current_mode = None

        # Check t-1 (Most recent history)
        if random.random() < config.prob_autoregressive_candidate:
            current_mode = random.choice(['blur', 'noise', 'both'])
        else:
            return stack, flags # clean sequence

        # Apply cascade
        # Slice index: num_slices-1 is t-1
        for i in range(num_slices - 1, -1, -1):

            # Apply current mode
            slice_data = stack[i]

            # Apply based on current mode
            apply_blur = (current_mode == 'blur') or (current_mode == 'both')
            apply_noise = (current_mode == 'noise') or (current_mode == 'both')

            if apply_blur:
                slice_data = gaussian_filter(slice_data, sigma=config.blur_sigma)
                flags['blur_applied'] = True

            if apply_noise:
                mask = (slice_data > 0.01).astype(np.float32)
                noise = np.random.normal(0, config.noise_std, slice_data.shape)
                slice_data = slice_data + (noise * mask)
                flags['noise_applied'] = True

            stack[i] = np.clip(slice_data, 0.0, 1.0)

            # Degrade mode for next (older) slice
            # If current is 'both', next can stay 'both' or drop to 'blur'/'noise'
            # If current is single, it stays single
            if current_mode == 'both':
                current_mode = random.choice(['blur', 'noise', 'both'])

            # Check if cascade stops (error started at t-X, so t-(X-1) is clean)
            if random.random() > config.prob_autoregressive_candidate:
                break

        return stack, flags

    @staticmethod
    def apply_tumor_void(input_stack: np.ndarray,
                         manager: Any, # changed type hint
                         brats_files: list,
                         config: AugmentationConfig
                        ) -> Tuple[np.ndarray, np.ndarray, bool]:
        """
        Injects a BraTS tumor mask as a void.
        Note: This is legacy logic used by Static Generator. Active Gen uses direct method below.
        """
        if not brats_files:
             return input_stack, np.zeros_like(input_stack[0]), False

        # 1. Fetch Mask
        b_subj = random.choice(brats_files)

        # Check if manager is Active or Static/Legacy
        if hasattr(manager, 'get_brats_pair'):
             _, seg_vol = manager.get_brats_pair(b_subj['t1'], b_subj['seg'])
        else:
             # StaticLoader doesn't have get_brats_pair, it has get_volume('brats')
             # But this method is designed for file-based access.
             # If using StaticLoader, we should use apply_tumor_void_direct.
             # This method is kept for compatibility if цу revert to file-based logic
             return input_stack, np.zeros_like(input_stack[0]), False

        if seg_vol is None:
            return input_stack, np.zeros_like(input_stack[0]), False

        # 2. Select Random Crop
        z_input, h_in, w_in = input_stack.shape
        z_seg = seg_vol.shape[0]

        if z_seg <= z_input:
            return input_stack, np.zeros_like(input_stack[0]), False

        start_z = random.randint(0, z_seg - z_input - 1)
        seg_stack = seg_vol[start_z : start_z + z_input + 1]

        if not np.any(np.isin(seg_stack, config.brats_labels_for_void)):
            return input_stack, np.zeros_like(input_stack[0]), False

        # 3. Resize
        scale_h = h_in / seg_stack.shape[1]
        scale_w = w_in / seg_stack.shape[2]
        seg_resized = zoom(seg_stack, (1, scale_h, scale_w), order=0)

        # 4. Crop/Pad safety
        if seg_resized.shape[1:] != (h_in, w_in):
             temp = np.zeros((z_input+1, h_in, w_in), dtype=seg_resized.dtype)
             min_h = min(seg_resized.shape[1], h_in)
             min_w = min(seg_resized.shape[2], w_in)
             temp[:, :min_h, :min_w] = seg_resized[:, :min_h, :min_w]
             seg_resized = temp

        # 5. Masking
        input_seg = seg_resized[:z_input]
        target_seg = seg_resized[z_input]

        is_tumor = np.isin(input_seg, config.brats_labels_for_void)
        input_stack[is_tumor] = 0.0
        target_tumor_mask = np.isin(
            target_seg, config.brats_labels_for_void
        ).astype(np.float32)

        return input_stack, target_tumor_mask, True

    @staticmethod
    def apply_tumor_void_direct(input_stack: np.ndarray,
                                seg_stack: np.ndarray,
                                config: AugmentationConfig):
        """
        Applies a pre-extracted N+1 segmentation stack as a void to the input stack.
        seg_stack shape: (N+1, H_seg, W_seg)
        input_stack shape: (N, H_in, W_in)
        """
        z_input, h_in, w_in = input_stack.shape

        # 1. Verify sizes match our logic (seg_stack should be exactly 1 slice larger for the target)
        if seg_stack.shape[0] != z_input + 1:
            return input_stack, np.zeros_like(input_stack[0]), False

        # 2. Fast exit if there are no tumor labels in the provided stack at all
        if not np.any(np.isin(seg_stack, config.brats_labels_for_void)):
            return input_stack, np.zeros_like(input_stack[0]), False

        # 3. Spatial Resizing (Matching bounding boxes across different brains)
        scale_h = h_in / seg_stack.shape[1]
        scale_w = w_in / seg_stack.shape[2]

        # Use nearest neighbor (order=0) to preserve integer label classes
        seg_resized = zoom(seg_stack, (1, scale_h, scale_w), order=0)

        # 4. Safety padding/cropping to guarantee exact (N+1, H_in, W_in) shape
        if seg_resized.shape[1:] != (h_in, w_in):
             temp = np.zeros((z_input + 1, h_in, w_in), dtype=seg_resized.dtype)
             min_h = min(seg_resized.shape[1], h_in)
             min_w = min(seg_resized.shape[2], w_in)
             temp[:, :min_h, :min_w] = seg_resized[:, :min_h, :min_w]
             seg_resized = temp

        # 5. Split into Context (t-N ... t-1) and Target (t)
        input_seg = seg_resized[:z_input]
        target_seg = seg_resized[z_input]

        target_tumor_mask = np.isin(
            target_seg, config.brats_labels_for_void
        ).astype(np.float32)

        # FIX: Abort if the target slice has no tumor pixels.
        # (Otherwise the model learns to erase an input without reconstructing
        # anything on the target)
        if not np.any(target_tumor_mask):
            return input_stack, np.zeros_like(input_stack[0]), False

        # --- THE FIX: TISSUE OVERLAP CHECK ---
        is_tumor = np.isin(input_seg, config.brats_labels_for_void)

        # A. Create a rough mask of actual brain tissue in the input stack
        brain_tissue = input_stack > 0.01

        # B. Check where the tumor mask actually intersects with brain tissue
        actual_void_overlap = is_tumor & brain_tissue

        # C. Require a meaningful visible void (e.g., at least 10 pixels erased)
        # This prevents both the "Air Void" and the "Surprise Tumor"
        if np.sum(actual_void_overlap) < 10:
            return input_stack, np.zeros_like(input_stack[0]), False

        # -------------------------------------

        # 6. Apply the void
        input_stack = input_stack.copy() # Only copy right before modifying to save RAM
        is_tumor = np.isin(input_seg, config.brats_labels_for_void)
        input_stack[is_tumor] = 0.0

        return input_stack, target_tumor_mask, True

# Cell 7: Combined Generators & Mixing Strategy
- **Target File:** src/data/generators.py
- **Content:**
    - BaseGenerator class
    - IXIDataGenerator class (The main training generator)
    - BraTSDataGenerator class (For mixing real data)
    - create_tf_dataset (Wraps generator in tf.data)
    - get_training_dataset (Mixes the two datasets)

In [ ]:
class GeneratorBase(tf.keras.utils.Sequence):
    def __init__(self, manager: DataLoaderBase, **kwargs):
        super().__init__(**kwargs)  # make Keras 3 happy (it is not happy anyway)
        self.manager = manager
        self.cfg = manager.cfg

    def __len__(self):
        return 1000

    @staticmethod
    def _get_stack(vol_data, axis, start_idx, neighborhood):
        """Helper to extract a 2.5D stack along any axis."""
        # Handle None data (e.g. missing seg) by returning None
        if vol_data is None: return None

        if axis == 0:   view = vol_data
        elif axis == 1: view = np.moveaxis(vol_data, 1, 0)
        else:           view = np.moveaxis(vol_data, 2, 0)

        # Slicing
        try:
            return view[start_idx : start_idx + neighborhood + 1]
        except Exception:
            return None

    @staticmethod
    # def _pick_start_index(candidates, axis_size, neighborhood):
    def _pick_start_index(vol, axis, neighborhood, mode='clean'):
        """
        Selects a start index that includes boundary conditions (empty-to-brain).
        Expands the candidate range by 'neighborhood' size.
        """
        axis_size = vol.shape[axis]

        # 1. Define what is ALLOWED and what is FORBIDDEN based on the mode
        if mode == 'clean':
            invalid_set = set(vol.indices['tumor'][axis]) # MUST NOT touch these
            required_set = set(vol.indices['any'][axis])  # MUST touch brain tissue
        elif mode == 'tumor':
            invalid_set = set()                           # Nothing is forbidden
            required_set = set(vol.indices['tumor'][axis])# MUST touch tumor
        else:
            return None

        valid_starts =[]

        # 2. Slide a window across the entire axis
        for s in range(axis_size - neighborhood):
            seq = set(range(s, s + neighborhood + 1))

            # 3. STRICT CHECK: Does this specific sequence violate our rules?
            if not seq.intersection(invalid_set) and seq.intersection(required_set):
                valid_starts.append(s)

        if not valid_starts:
            return None

        return random.choice(valid_starts)

In [ ]:
class IXIActiveGenerator(GeneratorBase):
    def __call__(self):
        while True:
            # A. Get Volume
            vol = self.manager.get_volume('ixi')
            if vol is None:
                time.sleep(0.05)
                continue

            # B. Pick Projection
            axis = random.choice(self.cfg.data.projections)
            candidates = vol.indices['clean'][axis]
            N = self.cfg.data.neighborhood

            if len(candidates) < N + 2: continue

            # Pick Slice
            # start_idx = self._pick_start_index(candidates, vol.shape[axis], N)
            start_idx = self._pick_start_index(vol, axis, N, mode='clean')
            if start_idx is None: continue

            # C. Extract Stack
            stack = self._get_stack(vol.t1, axis, start_idx, N)
            if stack is None: continue

            # Detach from Cache
            stack = stack.copy()

            # Randomly Flip Temporal Order (Bidirectional Training)
            is_backward = False
            if random.random() < 0.5:
                stack = np.flip(stack, axis=0) # flip along slice dimension
                is_backward = True

            # Record spatial index of target
            direction_label = 'backward' if is_backward else 'forward'
            real_target_idx = start_idx if is_backward else (start_idx + N)

            self.manager.stats.record('ixi', os.path.basename(vol.path), axis,
                                      real_target_idx, direction_label)

            input_stack = stack[:-1]
            target_slice = stack[-1]

            info = {
                'glioma_applied': False,
                'blur_applied': False,
                'noise_applied': False
            }

            # Initialize empty masks
            target_tumor_mask = np.zeros_like(target_slice)
            detailed_mask = np.zeros_like(target_slice)

            # --- 1. Tumor Voiding (Anatomically matched) ---
            if random.random() < self.cfg.aug.prob_glioma:
                # Calculate absolute normalized depth of the target slice
                target_norm_pos = real_target_idx / max(
                    1.0, float(vol.shape[axis] - 1.0)
                )

                # Query manager for an exact anatomical match
                seg_stack = self.manager.get_matched_glioma_stack(
                    axis, target_norm_pos, N
                )

                if seg_stack is not None:
                    # Ensure we match the flip state so the temporal sequence of the void aligns
                    if is_backward:
                        seg_stack = np.flip(seg_stack, axis=0)

                    input_stack, target_tumor_mask, applied = AugmentationLogic.apply_tumor_void_direct(
                        input_stack, seg_stack, self.cfg.aug
                    )
                    if applied:
                        info['glioma_applied'] = True

            # Pre-calculate brain mask so it can be geometrically augmented
            brain_mask = (target_slice > 0.01).astype(np.float32)

            # --- 2. Geometric Augmentation (FIXED MASK ALIGNMENT) ---
            # Pack all spatial masks into the array so they rotate with the image
            masks_to_rotate =[target_tumor_mask, detailed_mask, brain_mask]

            input_stack, target_slice, rotated_masks = AugmentationLogic.apply_geometric(
                input_stack, target_slice, masks_to_rotate, self.cfg.aug
            )

            # Unpack the synchronized masks
            target_tumor_mask, detailed_mask, brain_mask = rotated_masks

            # --- 3. Autoregressive Noise (Photometric) ---
            input_stack, flags = AugmentationLogic.apply_autoregressive_corruption(
                input_stack, self.cfg.aug
            )
            info.update(flags)

            # --- 4. Hallucination (FIXED AXIS AND INDEX) ---
            if random.random() < self.cfg.aug.prob_hallucination_replay:
                # Get the spatial index of the most recent slice in the history (input_stack[-1])
                h_idx = (start_idx + 1) if is_backward else (start_idx + N - 1)

                # Use strict Axis-aware dictionary key
                h_key = (vol.path, axis, h_idx)

                hal = self.manager.get_hallucination(h_key)
                if hal is not None and hal.shape == input_stack[-1].shape:
                    input_stack[-1] = hal
                hallucination_proc = True
            else:
                hallucination_proc = False

            # --- EDA TRACKING INFO ---
            # Map temporal sequence back to absolute spatial brain indices
            if is_backward:
                # Sequence was flipped. Original physical was [start ... start+N]
                # Target is start. Inputs are start+N, start+N-1, ..., start+1
                spatial_indices = [start_idx + N - k for k in range(N)] + [start_idx]
            else:
                # Target is start+N. Inputs are start, start+1, ..., start+N-1
                spatial_indices = [start_idx + k for k in range(N)] +[start_idx + N]

            info.update({
                'volume_path': vol.path,
                'axis_size': vol.shape[axis],
                'axis': axis,
                'direction': direction_label,
                'spatial_indices': spatial_indices,
                'target_tumor_mask': target_tumor_mask, # to draw contours
                'hallucination_proc': hallucination_proc,
                'mode': getattr(self, 'mode', 'clean') # identify IXI/BraTS mode
            })
            # -------------------------

            # --- D. Formatting ---
            true_start_idx = (start_idx + N) if is_backward else start_idx

            model_inputs, _, _ = InputProcessor.prepare_model_input(
                input_stack_native=input_stack,
                target_slice_native=target_slice,
                start_idx=true_start_idx,
                target_idx=real_target_idx,
                axis_size=vol.shape[axis],
                direction=direction_label,
                void_mask_native=target_tumor_mask if np.any(target_tumor_mask) else None,
                config=self.cfg
            )

            y = np.stack([
                target_slice,
                target_tumor_mask,
                detailed_mask,
                brain_mask
            ], axis=-1)

            yield model_inputs, y, info


class BraTSActiveGenerator(GeneratorBase):
    def __init__(self, manager, mode='clean', **kwargs):
        super().__init__(manager, **kwargs)
        self.mode = mode

    def __call__(self):
        while True:
            vol = self.manager.get_volume('brats')
            if vol is None:
                time.sleep(0.05)
                continue

            axis = random.choice(self.cfg.data.projections)

            if self.mode == 'tumor':
                if vol.seg is None: continue
                candidates = vol.indices['tumor'][axis]
            else:
                candidates = vol.indices['clean'][axis]

            N = self.cfg.data.neighborhood
            if len(candidates) == 0: continue

            if self.mode == 'tumor':
                target_idx = random.choice(candidates)
                start_idx = target_idx - N
            else:
                # start_idx = self._pick_start_index(candidates, vol.shape[axis], N)
                start_idx = self._pick_start_index(vol, axis, N, mode=self.mode)

            if start_idx is None or start_idx < 0 or start_idx + N >= vol.shape[axis]:
                continue

            # B. Extract Stacks
            t1_stack = self._get_stack(vol.t1, axis, start_idx, N)
            seg_stack_native = self._get_stack(
                vol.seg, axis, start_idx, N
            ) if vol.seg is not None else None

            if t1_stack is None: continue

            is_backward = False
            if random.random() < 0.5:
                t1_stack = np.flip(t1_stack, axis=0)
                if seg_stack_native is not None:
                    seg_stack_native = np.flip(seg_stack_native, axis=0)
                is_backward = True

            direction_label = 'backward' if is_backward else 'forward'
            real_target_idx = start_idx if is_backward else (start_idx + N)

            self.manager.stats.record('brats', os.path.basename(vol.path), axis,
                                      real_target_idx, direction_label)

            input_stack = t1_stack[:-1].copy()
            target_slice = t1_stack[-1].copy()

            info = {
                'glioma_applied': False,
                'blur_applied': False,
                'noise_applied': False
            }

            target_tumor_mask = np.zeros_like(target_slice)
            detailed_mask = np.zeros_like(target_slice)

            # --- 1. Tumor Voiding (Context Aware) ---
            if self.mode == 'tumor' and seg_stack_native is not None:
                # In tumor mode, we use the volume's own actual tumor!
                input_stack, target_tumor_mask, applied = AugmentationLogic.apply_tumor_void_direct(
                    input_stack, seg_stack_native, self.cfg.aug
                )
                if applied:
                    info['glioma_applied'] = True
                    detailed_mask = seg_stack_native[-1].astype(np.float32)
            else:
                # In clean mode, we query the manager to inject an artificial void just like IXI
                if random.random() < self.cfg.aug.prob_glioma:
                    target_norm_pos = real_target_idx / max(
                        1.0, float(vol.shape[axis] - 1.0)
                    )
                    seg_stack = self.manager.get_matched_glioma_stack(
                        axis, target_norm_pos, N
                    )

                    if seg_stack is not None:
                        if is_backward:
                            seg_stack = np.flip(seg_stack, axis=0)

                        input_stack, target_tumor_mask, applied = AugmentationLogic.apply_tumor_void_direct(
                            input_stack, seg_stack, self.cfg.aug
                        )
                        if applied:
                            info['glioma_applied'] = True

            # Pre-calculate brain mask
            brain_mask = (target_slice > 0.01).astype(np.float32)

            # --- 2. Geometric Augmentation (FIXED MASK ALIGNMENT) ---
            masks_to_rotate =[target_tumor_mask, detailed_mask, brain_mask]

            input_stack, target_slice, rotated_masks = AugmentationLogic.apply_geometric(
                input_stack, target_slice, masks_to_rotate, self.cfg.aug
            )
            target_tumor_mask, detailed_mask, brain_mask = rotated_masks

            # --- 3. Autoregressive Noise ---
            input_stack, flags = AugmentationLogic.apply_autoregressive_corruption(
                input_stack, self.cfg.aug
            )
            info.update(flags)

            # --- 4. Hallucination (FIXED AXIS AND INDEX) ---
            if random.random() < self.cfg.aug.prob_hallucination_replay:
                h_idx = (start_idx + 1) if is_backward else (start_idx + N - 1)
                h_key = (vol.path, axis, h_idx)

                hal = self.manager.get_hallucination(h_key)
                if hal is not None and hal.shape == input_stack[-1].shape:
                    input_stack[-1] = hal
                hallucination_proc = True
            else:
                hallucination_proc = False

            # --- EDA TRACKING INFO ---
            # Map temporal sequence back to absolute spatial brain indices
            if is_backward:
                # Sequence was flipped. Original physical was [start ... start+N]
                # Target is start. Inputs are start+N, start+N-1, ..., start+1
                spatial_indices = [start_idx + N - k for k in range(N)] + [start_idx]
            else:
                # Target is start+N. Inputs are start, start+1, ..., start+N-1
                spatial_indices = [start_idx + k for k in range(N)] +[start_idx + N]

            info.update({
                'volume_path': vol.path,
                'axis_size': vol.shape[axis],
                'axis': axis,
                'direction': direction_label,
                'spatial_indices': spatial_indices,
                'target_tumor_mask': target_tumor_mask, # to draw contours
                'hallucination_proc': hallucination_proc,
                'mode': getattr(self, 'mode', 'clean') # identify IXI/BraTS mode
            })
            # -------------------------

            # --- D. Formatting ---
            true_start_idx = (start_idx + N) if is_backward else start_idx

            model_inputs, _, _ = InputProcessor.prepare_model_input(
                input_stack_native=input_stack,
                target_slice_native=target_slice,
                start_idx=true_start_idx,
                target_idx=real_target_idx,
                axis_size=vol.shape[axis],
                direction=direction_label,
                void_mask_native=target_tumor_mask if np.any(target_tumor_mask) else None,
                config=self.cfg
            )

            y = np.stack([target_slice, target_tumor_mask, detailed_mask, brain_mask], axis=-1)
            yield model_inputs, y, info

In [ ]:
class SequentialValidationGenerator(GeneratorBase):
    """
    Finite, deterministic generator for accurate Validation tracking.
    Iterates over both IXI and BraTS RAM pools.
    """
    def __init__(self, manager: DataLoaderBase, max_slices_per_vol=10, **kwargs):
        super().__init__(manager, **kwargs)  # <--- ADD **kwargs (Keras happiness)
        self.max_slices = max_slices_per_vol

    def __call__(self):
        ixi_pool = self.manager.pool.get('ixi', {})
        brats_pool = self.manager.pool.get('brats', {})
        ixi_vols = list(ixi_pool.values()) if isinstance(ixi_pool, dict) else list(ixi_pool)
        brats_vols = list(brats_pool.values()) if isinstance(brats_pool, dict) else list(brats_pool)
        volumes = [('ixi', v) for v in ixi_vols] + [('brats', v) for v in brats_vols]

        for ds_type, vol in volumes:
            axis = 0
            candidates = vol.indices['clean'][axis]
            N = self.cfg.data.neighborhood
            is_backward = False
            if len(candidates) < N + 2: continue

            mid = len(candidates) // 2
            half_window = self.max_slices // 2
            start_c = max(0, mid - half_window)
            end_c = min(len(candidates), mid + half_window)

            for target_idx in candidates[start_c:end_c]:
                start_idx = target_idx - N
                if start_idx < 0 or start_idx + N >= vol.shape[axis]: continue

                # NEW: RIGOROUS VALIDATION SAFETY CHECK
                # Ensure the validation sequence is strictly clean
                seq_set = set(range(start_idx, start_idx + N + 1))
                if seq_set.intersection(set(vol.indices['tumor'][axis])):
                    continue # skip this sequence, it crossed into a tumor!

                stack = self._get_stack(vol.t1, axis, start_idx, N)
                if stack is None: continue

                seg_stack = None
                if ds_type == 'brats' and vol.seg is not None:
                    seg_stack = self._get_stack(vol.seg, axis, start_idx, N)

                input_stack = stack[:-1].copy()
                target_slice = stack[-1].copy()

                if seg_stack is not None:
                    target_tumor_mask = np.isin(
                        seg_stack[-1], self.cfg.aug.brats_labels_for_void
                    ).astype(np.float32)
                    detailed_mask = seg_stack[-1].astype(np.float32)
                else:
                    target_tumor_mask = np.zeros_like(target_slice)
                    detailed_mask = np.zeros_like(target_slice)

                # --- FIX: Convert to Dictionary Format ---
                model_inputs, _, _ = InputProcessor.prepare_model_input(
                    input_stack_native=input_stack,
                    target_slice_native=target_slice,
                    start_idx=start_idx,
                    target_idx=target_idx,
                    axis_size=vol.shape[axis],
                    direction='forward',
                    void_mask_native=target_tumor_mask if np.any(target_tumor_mask) else None,
                    config=self.cfg
                )

                brain_mask = (target_slice > 0.01).astype(np.float32)
                y = np.stack([target_slice, target_tumor_mask, detailed_mask, brain_mask], axis=-1)
                info = {'glioma_applied': False, 'blur_applied': False, 'noise_applied': False}

            # --- EDA TRACKING INFO ---
            # Map temporal sequence back to absolute spatial brain indices
            if is_backward:
                # Sequence was flipped. Original physical was [start ... start+N]
                # Target is start. Inputs are start+N, start+N-1, ..., start+1
                spatial_indices = [start_idx + N - k for k in range(N)] + [start_idx]
                direction_label = 'backward'
            else:
                # Target is start+N. Inputs are start, start+1, ..., start+N-1
                spatial_indices = [start_idx + k for k in range(N)] +[start_idx + N]
                direction_label = 'forward'

            info.update({
                'volume_path': vol.path,
                'axis_size': vol.shape[axis],
                'axis': axis,
                'direction': direction_label,
                'spatial_indices': spatial_indices,
                'target_tumor_mask': target_tumor_mask, # to draw contours
                'hallucination_proc': False,
                'mode': getattr(self, 'mode', 'clean') # identify IXI/BraTS mode
            })
            # -------------------------

            yield model_inputs, y, info

In [ ]:
class HpoTrainSequence(tf.keras.utils.Sequence):
    """
    Pure Python data pipeline. Bypasses tf.data C++ memory leaks.
    """
    def __init__(self, ixi_gen_instance, brats_gen_instance, config,
                 steps_per_epoch, **kwargs):
        super().__init__(**kwargs)  # make Keras 3 happy (it is not happy anyway)
        self.ixi_iter = iter(ixi_gen_instance())
        self.brats_iter = iter(brats_gen_instance())
        self.cfg = config
        self.steps = steps_per_epoch
        self.batch_size = config.train.batch_size

    def __len__(self):
        return self.steps

    def __getitem__(self, idx):
        batch_x, batch_y = [], []
        w_ixi = max(0.1, min(0.9, self.cfg.data.ixi_sampling_weight))

        for _ in range(self.batch_size):
            # 1. Randomly sample
            if random.random() < w_ixi:
                x, y, _ = next(self.ixi_iter)
            else:
                x, y, _ = next(self.brats_iter)

            # 2. Convert to tensors
            x_t = tf.convert_to_tensor(x, dtype=tf.float32)
            y_t = tf.convert_to_tensor(y, dtype=tf.float32)

            # 3. Apply geometry ops dynamically
            xp, _, _ = GeometryOps.resize_and_pad(x_t, self.cfg.data.padded_size,
                                                  'bicubic')
            yp, _, _ = GeometryOps.resize_and_pad(y_t, self.cfg.data.padded_size,
                                                  'nearest')

            batch_x.append(xp)
            batch_y.append(yp)

        return tf.stack(batch_x), tf.stack(batch_y)


class HpoValidSequence(tf.keras.utils.Sequence):
    """
    Eagerly materializes the validation set into RAM.
    Zero-overhead, deterministic evaluation.
    """
    def __init__(self, val_gen_instance, config, **kwargs):
        super().__init__(**kwargs)  # make Keras 3 happy (it is not happy anyway)
        self.cfg = config
        self.batch_size = config.train.batch_size

        # Extract all slices into a list exactly once! (~150 MB footprint)
        self.data = list(val_gen_instance())

    def __len__(self):
        return int(np.ceil(len(self.data) / self.batch_size))

    def __getitem__(self, idx):
        batch_data = self.data[idx * self.batch_size : (idx + 1) * self.batch_size]
        batch_x, batch_y = [], []

        for x, y, _ in batch_data:
            x_t = tf.convert_to_tensor(x, dtype=tf.float32)
            y_t = tf.convert_to_tensor(y, dtype=tf.float32)

            xp, _, _ = GeometryOps.resize_and_pad(x_t, self.cfg.data.padded_size,
                                                  'bicubic')
            yp, _, _ = GeometryOps.resize_and_pad(y_t, self.cfg.data.padded_size,
                                                  'nearest')

            batch_x.append(xp)
            batch_y.append(yp)

        return tf.stack(batch_x), tf.stack(batch_y)

In [ ]:
def create_tf_dataset(generator: GeneratorBase, config: Config,
                      is_training: bool = True, include_info: bool = False):
    h, w = config.data.padded_size
    N = config.data.neighborhood

    # 1. Signature for Model Inputs (X)
    x_sig = {
        'history_input': tf.TensorSpec(shape=(None, None, N), dtype=tf.float32),
        'mask_input': tf.TensorSpec(shape=(None, None, 1), dtype=tf.float32),
        'p_history_input': tf.TensorSpec(shape=(N,), dtype=tf.float32),
        'p_abs_input': tf.TensorSpec(shape=(1,), dtype=tf.float32)
    }

    # 2. Signature for Rich EDA Info Dictionary
    # This precisely matches the 11 keys we are yielding for debugging
    info_sig = {
        "glioma_applied": tf.TensorSpec(shape=(), dtype=tf.bool),
        "blur_applied": tf.TensorSpec(shape=(), dtype=tf.bool),
        "noise_applied": tf.TensorSpec(shape=(), dtype=tf.bool),
        "volume_path": tf.TensorSpec(shape=(), dtype=tf.string),
        "axis_size": tf.TensorSpec(shape=(), dtype=tf.int32),
        "axis": tf.TensorSpec(shape=(), dtype=tf.int32),
        "direction": tf.TensorSpec(shape=(), dtype=tf.string),
        "spatial_indices": tf.TensorSpec(shape=(None,), dtype=tf.int32), # 1D variable length list
        "target_tumor_mask": tf.TensorSpec(shape=(None, None), dtype=tf.float32), # 2D native mask
        "hallucination_proc": tf.TensorSpec(shape=(), dtype=tf.bool),
        "mode": tf.TensorSpec(shape=(), dtype=tf.string)
    }

    # 3. Full Yield Signature
    output_sig = (
        x_sig,
        tf.TensorSpec(shape=(None, None, 4), dtype=tf.float32), # targets (Y)
        info_sig
    )

    ds = tf.data.Dataset.from_generator(generator, output_signature=output_sig)

    def map_fn(x_dict, y, info):
        # 2. PERFORM RESIZING HERE (Multi-threaded C++)
        hist_pad, _, _ = GeometryOps.resize_and_pad(x_dict['history_input'], (h, w),
                                                    method='bicubic')
        mask_pad, _, _ = GeometryOps.resize_and_pad(x_dict['mask_input'], (h, w),
                                                    method='nearest')
        y_padded, _, _ = GeometryOps.resize_and_pad(y, (h, w), method='nearest')

        # Enforce static shapes for the compiler
        hist_pad.set_shape((h, w, N))
        mask_pad.set_shape((h, w, 1))
        y_padded.set_shape((h, w, 4))

        x_dict_out = {
            'history_input': hist_pad,
            'mask_input': mask_pad,
            'p_history_input': x_dict['p_history_input'],
            'p_abs_input': x_dict['p_abs_input']
        }

        if include_info:
            return x_dict_out, y_padded, info
        else:
            return x_dict_out, y_padded

    # 3. Apply the map function concurrently
    ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)

    if not is_training:
        batch_size = 1 if include_info else config.train.batch_size
        ds = ds.batch(batch_size)

    return ds


def get_training_dataset(ixi_gen, brats_gen, config):
    w_ixi = config.data.ixi_sampling_weight

    # Check for pure scenarios to avoid TF Graph sampling bugs
    if w_ixi >= 1.0 - 1e-8:
        logger.info("Dataset Mix: 100% IXI Pipeline")
        ds = create_tf_dataset(ixi_gen, config, is_training=True,
                               include_info=False).repeat()
    elif w_ixi < 1e-8:
        logger.info("Dataset Mix: 100% BraTS Pipeline")
        ds = create_tf_dataset(brats_gen, config, is_training=True,
                               include_info=False).repeat()
    else:
        logger.info(f"Dataset Mix: {w_ixi * 100:.1f}% IXI, {(1.0 - w_ixi) * 100:.1f}% BraTS")
        ds_ixi = create_tf_dataset(ixi_gen, config, is_training=True,
                                   include_info=False).repeat()
        ds_brats = create_tf_dataset(brats_gen, config, is_training=True,
                                     include_info=False).repeat()
        ds = tf.data.Dataset.sample_from_datasets([ds_ixi, ds_brats],
                                                  weights=[w_ixi, 1.0 - w_ixi])

    ds = ds.batch(config.train.batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# Cell 8 (Revised): Generic Model Builder

In [ ]:
import tensorflow as tf

from tensorflow.keras import layers, models, applications

- **Target File:** src/models/layers.py
- **Content:**
    - SPADELayer
    - SPADEResBlock

In [ ]:
try:
    from tensorflow.keras.layers import SpectralNormalization
except ImportError:
    # Fallback: Just identity wrapper (no-op) if not available, to avoid complex custom code risks
    class SpectralNormalization(layers.Wrapper):
        def __init__(self, layer, **kwargs): super().__init__(layer, **kwargs)
        def call(self, inputs, training=None): return self.layer(inputs)

In [ ]:
# class SpectralNormalization(layers.Wrapper):
#     def __init__(self, layer, iteration=1, **kwargs):
#         super(SpectralNormalization, self).__init__(layer, **kwargs)
#         self.iteration = iteration

#     def build(self, input_shape):
#         if not self.layer.built:
#             self.layer.build(input_shape)
#         if not hasattr(self.layer, 'kernel'):
#             raise ValueError('Layer must have a kernel weight to use SpectralNormalization.')

#         self.w = self.layer.kernel
#         self.w_shape = self.w.shape.as_list()
#         self.u = self.add_weight(shape=(1, self.w_shape[-1]), initializer=tf.initializers.TruncatedNormal(stddev=0.02), trainable=False, name='sn_u', dtype=self.dtype)

#     def call(self, inputs, training=None):
#         # Power iteration
#         # simple implementation for brevity
#         # For full robustness, use tf.keras.layers.SpectralNormalization if available
#         # But here is a simplified version if needed, or rely on standard Conv for now if this is too complex to inject.
#         # Actually, let's try to import the native one first.
#         return self.layer(inputs)

VAE helper classes:

In [ ]:
class Sampling(layers.Layer):
    """
    Uses (z_mean, z_log_var) to sample z, the vector encoding a digit.
    Includes Float32 casting for stability in Mixed Precision.
    """
    def call(self, inputs):
        z_mean, z_log_var = inputs

        # 1. Force Float32 to prevent overflow in exp/random
        z_mean = tf.cast(z_mean, tf.float32)
        z_log_var = tf.cast(z_log_var, tf.float32)

        # 2. Hard Clip Log-Variance to prevent explosion
        # exp(10) ~ 22000, safe for float16, very safe for float32
        z_log_var = tf.clip_by_value(z_log_var, -10.0, 10.0)

        epsilon = tf.random.normal(shape=tf.shape(z_mean), dtype=tf.float32)

        # Sample
        z = z_mean + tf.exp(0.5 * z_log_var) * epsilon

        # Cast back to model dtype (likely float16)
        return tf.cast(z, self.compute_dtype)

    def compute_output_shape(self, input_shape):
        return input_shape[0]


class VAELossLayer(layers.Layer):
    """Identity layer that adds KL Divergence loss."""
    def __init__(self, weight=1.0, **kwargs):
        super().__init__(**kwargs)
        self.weight = weight

    def call(self, inputs):
        z_mean, z_log_var = inputs

        # 1. Force Float32
        z_mean = tf.cast(z_mean, tf.float32)
        z_log_var = tf.cast(z_log_var, tf.float32)

        # 2. Hard Clip (Must match Sampling logic)
        z_log_var = tf.clip_by_value(z_log_var, -10.0, 10.0)

        # 3. KL Calculation
        kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
        kl_loss = tf.reduce_mean(kl_loss)

        # Add loss
        self.add_loss(kl_loss * self.weight)

        return inputs[0] # pass-through

In [ ]:
class InstanceNormalization(layers.Layer):
    """
    Standard Instance Normalization (not always available in base Keras).
    Normalizes (H, W) per channel, per sample.
    """
    def __init__(self, epsilon=1e-5, **kwargs):
        super().__init__(**kwargs)
        self.epsilon = epsilon

    def build(self, input_shape):
        self.gamma = self.add_weight(name='gamma', shape=(input_shape[-1],),
                                     initializer='ones', trainable=True)
        self.beta = self.add_weight(name='beta', shape=(input_shape[-1],),
                                    initializer='zeros', trainable=True)

    def call(self, x):
        mean, variance = tf.nn.moments(x, axes=[1, 2], keepdims=True)
        return self.gamma * (x - mean) / tf.sqrt(variance + self.epsilon) + self.beta

In [ ]:
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers


class FourierEmbedding(layers.Layer):
    """
    Pure sinusoidal feature expansion.
    """
    def __init__(self, num_freqs=8, **kwargs):
        super().__init__(**kwargs)
        self.num_freqs = num_freqs
        freq_bands = 2.0 ** np.linspace(0.0, num_freqs - 1, num_freqs) * np.pi
        self.freq_bands = tf.constant(freq_bands, dtype=tf.float32)

    def call(self, p):
        p_expanded = tf.expand_dims(p, -1)
        args = p_expanded * self.freq_bands

        sin_emb = tf.sin(args)
        cos_emb = tf.cos(args)

        emb = tf.concat([sin_emb, cos_emb], axis=-1)

        # Flatten: (Batch, Channels * Num_Freqs * 2)
        input_channels = tf.shape(p)[1]
        if p.shape[1] is not None:
            input_channels = p.shape[1]

        flat_dim = input_channels * self.num_freqs * 2
        return tf.reshape(emb, [-1, flat_dim])


# class FiLMLayer(layers.Layer):
#     """
#     Feature-wise Linear Modulation.
#     Modulates a feature map using a conditioning vector.
#     """
#     def __init__(self, **kwargs):
#         super().__init__(**kwargs)

#     def build(self, input_shape):
#         # input_shape = [feature_shape, condition_shape]
#         feature_channels = input_shape[0][-1]

#         # Project condition vector to 2 * feature_channels (for Gamma and Beta)
#         self.dense = layers.Dense(feature_channels * 2, kernel_initializer='zeros')

#     def call(self, inputs):
#         x, condition = inputs

#         # Generate Gamma and Beta from condition
#         mod_params = self.dense(condition)

#         # Split into Gamma and Beta
#         # Reshape to (Batch, 1, 1, Channels) for spatial broadcasting
#         channels = tf.shape(x)[-1]
#         gamma, beta = tf.split(mod_params, 2, axis=-1)

#         gamma = tf.reshape(gamma, [-1, 1, 1, channels])
#         beta = tf.reshape(beta, [-1, 1, 1, channels])

#         # Apply FiLM: x * (1 + gamma) + beta
#         return x * (1.0 + gamma) + beta

In [ ]:
class SPADELayer(layers.Layer):
    def __init__(self, channels, kernel_size=3, **kwargs):
        super().__init__(**kwargs)
        self.channels = channels
        self.bn = layers.BatchNormalization(center=False, scale=False)
        self.conv_mask = layers.Conv2D(128, kernel_size=kernel_size,
                                       padding='same', activation='relu')
        self.conv_gamma = layers.Conv2D(channels, kernel_size=kernel_size,
                                        padding='same')
        self.conv_beta = layers.Conv2D(channels, kernel_size=kernel_size,
                                       padding='same')

    def call(self, inputs):
        x, mask = inputs # purely spatial

        normalized = self.bn(x)
        target_size = tf.shape(x)[1:3]
        mask_resized = tf.image.resize(mask, target_size, method='nearest')
        mask_resized = tf.cast(mask_resized, x.dtype)

        mask_feat = self.conv_mask(mask_resized)
        gamma = self.conv_gamma(mask_feat)
        beta = self.conv_beta(mask_feat)

        return normalized * (1.0 + gamma) + beta


class SPADEResBlock(layers.Layer):
    def __init__(self, filters, input_channels, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.learned_shortcut = (filters != input_channels)
        f_mid = min(filters, input_channels)

        self.spade1 = SPADELayer(input_channels)
        self.conv1 = layers.Conv2D(f_mid, 3, padding='same')
        self.spade2 = SPADELayer(f_mid)
        self.conv2 = layers.Conv2D(filters, 3, padding='same')

        if self.learned_shortcut:
            self.spade_s = SPADELayer(input_channels)
            self.conv_s = layers.Conv2D(filters, 1, padding='same',
                                        use_bias=False)

    def call(self, inputs):
        x, mask = inputs # purely spatial

        x_s = x
        if self.learned_shortcut:
            x_s = self.conv_s(self.spade_s([x, mask]))

        dx = self.spade1([x, mask])
        dx = tf.nn.relu(dx)
        dx = self.conv1(dx)
        dx = self.spade2([dx, mask])
        dx = tf.nn.relu(dx)
        dx = self.conv2(dx)

        return x_s + dx

- **Target File:** src/models/builder.py
- **Content:**
    - ModelBuilder (Factory pattern)
    - _build_unet logic
    - Encoder abstraction (get_efficientnet_encoder)

In [ ]:
class DiscriminatorBuilder:
    """
    Builds a PatchGAN Discriminator.
    Input: [Reconstructed_Image, Condition_Stack]
    Output: Patch Map of Real/Fake scores.
    """
    @staticmethod
    def build(input_shape, condition_shape, base_filters=64):
        img_input = layers.Input(shape=input_shape, name='img_input')
        cond_input = layers.Input(shape=condition_shape, name='cond_input')

        # Concat Image + Condition (History + Mask)
        x = layers.Concatenate()([img_input, cond_input])

        # Store intermediate features for Feature Matching Loss
        features = []

        # C64
        # x = layers.Conv2D(base_filters, 4, strides=2, padding='same')(x)
        # x = layers.LeakyReLU(0.2)(x)
        x = SpectralNormalization(layers.Conv2D(base_filters, 4, strides=2,
                                                padding='same'))(x)
        x = layers.LeakyReLU(0.2)(x)
        features.append(x)

        # C128
        # x = layers.Conv2D(base_filters*2, 4, strides=2, padding='same')(x)
        # x = layers.GroupNormalization(groups=-1)(x)
        # x = layers.LeakyReLU(0.2)(x)
        x = SpectralNormalization(layers.Conv2D(base_filters*2, 4, strides=2,
                                                padding='same'))(x)
        x = InstanceNormalization()(x)
        x = layers.LeakyReLU(0.2)(x)
        features.append(x)

        # C256
        # x = layers.Conv2D(base_filters*4, 4, strides=2, padding='same')(x)
        # x = layers.GroupNormalization(groups=-1)(x)
        # x = layers.LeakyReLU(0.2)(x)
        x = SpectralNormalization(layers.Conv2D(base_filters*4, 4, strides=2,
                                                padding='same'))(x)
        x = InstanceNormalization()(x)
        x = layers.LeakyReLU(0.2)(x)
        features.append(x)

        # C512 (Stride 1)
        # x = layers.Conv2D(base_filters*8, 4, strides=1, padding='same')(x)
        # x = layers.GroupNormalization(groups=-1)(x)
        # x = layers.LeakyReLU(0.2)(x)
        x = SpectralNormalization(layers.Conv2D(base_filters*8, 4, strides=1,
                                                padding='same'))(x)
        x = InstanceNormalization()(x)
        x = layers.LeakyReLU(0.2)(x)
        features.append(x)

        # Output Map (1 channel)
        outputs = layers.Conv2D(1, 4, strides=1, padding='same')(x)

        return models.Model([img_input, cond_input], [outputs] + features,
                            name="discriminator")

In [ ]:
class ModelBuilder:
    """
    Factory to instantiate different neural network architectures.
    Currently supports: U-Net (Custom & EfficientNet backbones).
    Future support: VAE, GAN.
    """
    @staticmethod
    def build(config: Config):
        arch = config.model.architecture.lower()

        if arch == 'unet':
            return ModelBuilder._build_unet(config)
        elif arch == 'spade':
            return ModelBuilder._build_spade_generator(config)
        elif arch == 'vae':
            return ModelBuilder._build_spade_vae_generator(config)
        elif arch == 'gan':
            raise NotImplementedError("GAN architecture not yet implemented.")
        else:
            raise ValueError(f"Unknown architecture: {arch}")

    @staticmethod
    def conv_block(x, filters, activation='gelu'):
        x = layers.BatchNormalization()(x)
        x = layers.Activation(activation)(x)
        x = layers.Conv2D(filters, 3, padding='same',
                          kernel_initializer='he_normal')(x)

        x = layers.BatchNormalization()(x)
        x = layers.Activation(activation)(x)
        x = layers.Conv2D(filters, 3, padding='same',
                          kernel_initializer='he_normal')(x)
        return x

    @staticmethod
    def attention_gate(x, skip, filters):
        """Additive Attention Gate"""
        g = layers.Conv2D(filters, 1, padding='same')(x)
        g = layers.BatchNormalization()(g)
        s = layers.Conv2D(filters, 1, padding='same')(skip)
        s = layers.BatchNormalization()(s)

        psi = layers.Activation('relu')(layers.Add()([g, s]))
        psi = layers.Conv2D(1, 1, padding='same', activation='sigmoid')(psi)
        return layers.Multiply()([skip, psi])

    @staticmethod
    def up_block(x, skip, filters, activation='gelu', use_attention=False):
        """
        Upsampling block using PixelShuffle (Sub-pixel Convolution) for sharpness.
        Replaces standard UpSampling2D to avoid blur/checkerboard artifacts.
        """
        # 1. PixelShuffle Upsampling
        # To upscale 2x, we need 4x channels.
        # We assume 'x' currently has 'in_filters'. We want 'filters' output.
        # So we project to filters * 4

        # x = layers.UpSampling2D((2, 2), interpolation='bilinear')(x)
        # x = layers.Conv2D(filters, 3, padding='same')(x) # adjust channels
        x = layers.Conv2D(filters * 4, kernel_size=1, padding='same',
                          kernel_initializer='he_normal')(x)
        # PixelShuffle trick: tf.nn.depth_to_space
        x = layers.Lambda(lambda z: tf.nn.depth_to_space(z, block_size=2))(x)

        # x is now (2H, 2W, filters)

        # 2. Skip Connection
        if use_attention and skip is not None:
            # Attention gates act on the skip connection
            skip = ModelBuilder.attention_gate(x, skip, filters // 2)

        if skip is not None:
            x = layers.Concatenate()([x, skip])

        # 3. Refinement Convolution
        x = ModelBuilder.conv_block(x, filters, activation)
        return x

    @staticmethod
    def get_efficientnet_encoder(input_tensor, backbone_name):
        """Returns the model and the skip connection tensors."""
        # Map names to constructors
        ctors = {
            'B0': applications.EfficientNetB0, 'B1': applications.EfficientNetB1,
            'B2': applications.EfficientNetB2, 'B3': applications.EfficientNetB3,
            'B4': applications.EfficientNetB4, 'B5': applications.EfficientNetB5,
            'B6': applications.EfficientNetB6, 'B7': applications.EfficientNetB7,
        }
        # Standard EfficientNet naming convention for block activation layers
        # Key: Backbone Name -> List of layer names for [Skip 1/2, Skip 1/4, Skip 1/8, Skip 1/16]
        # Skip 1/32 comes from bridge output
        skip_names = {
            'B0': ['block2a_expand_activation', 'block3a_expand_activation',
                   'block4a_expand_activation', 'block6a_expand_activation'],
            'B1': ['block2a_expand_activation', 'block3a_expand_activation',
                   'block4a_expand_activation', 'block6a_expand_activation'],
            'B2': ['block2a_expand_activation', 'block3a_expand_activation',
                   'block4a_expand_activation', 'block6a_expand_activation'],
            'B3': ['block2a_expand_activation', 'block3a_expand_activation',
                   'block4a_expand_activation', 'block6a_expand_activation'],
            'B4': ['block2a_expand_activation', 'block3a_expand_activation',
                   'block4a_expand_activation', 'block6a_expand_activation'],
            'B5': ['block2a_expand_activation', 'block3a_expand_activation',
                   'block4a_expand_activation', 'block6a_expand_activation'],
            'B6': ['block2a_expand_activation', 'block3a_expand_activation',
                   'block4a_expand_activation', 'block6a_expand_activation'],
            'B7': ['block2a_expand_activation', 'block3a_expand_activation',
                   'block4a_expand_activation', 'block6a_expand_activation'],
        }

        # B0-B7 share similar early block naming for the expand activation.
        # However, deeper networks repeat blocks. We target the FIRST block of each stage.
        # Actually, for U-Net skips, we usually want the LAST block of a stage (deepest features at that res).
        # But 'expand_activation' of the FIRST block of the NEXT stage works too (pre-downsample).
        # The list above targets the start of stage 2, 3, 4, 6.
        # This has worked for B0. For B4+, resolutions might shift.
        # Verified for standard Keras EfficientNet implementation: These names are stable
        names = skip_names.get(backbone_name, skip_names['B0'])
        base = ctors[backbone_name](
            name=f"encoder_{backbone_name}",
            include_top=False,
            weights=None,
            input_tensor=input_tensor
        )

        try:
            skips = [base.get_layer(n).output for n in names]
        except ValueError as e:
            # Fallback for debugging if layer names shift in newer TF versions
            print(f"Error finding skip layers for {backbone_name}. Available layers:")
            raise e
        return base.output, skips

    @staticmethod
    def _build_spade_generator(config: Config):
        arch = config.model.architecture.lower()
        N = config.data.neighborhood

        # 1. Inputs
        history_input = layers.Input(shape=(*config.data.padded_size, N),
                                     name='history_input')
        mask_input = layers.Input(shape=(*config.data.padded_size, 1),
                                  name='mask_input')

        # p_rel_input = layers.Input(shape=(N,), name='p_rel_input')
        # p_abs_input = layers.Input(shape=(1,), name='p_abs_input')

        base_f = config.model.base_filters

        # 2. Positional Encodings
        # p_rel_emb = FourierEmbedding(hidden_dim=256, num_freqs=8, name='p_rel_embed')(p_rel_input)
        # p_abs_emb = FourierEmbedding(hidden_dim=256, num_freqs=8, name='p_abs_embed')(p_abs_input)
        p_history_input = layers.Input(shape=(N,), name='p_history_input')
        p_abs_input = layers.Input(shape=(1,), name='p_abs_input')

        # 4. Encoder (Processes History Images)
        decoder_filters = [base_f * 8, base_f * 4, base_f * 2, base_f]

        if config.model.backbone:
            bridge, skips = ModelBuilder.get_efficientnet_encoder(history_input,
                                                                  config.model.backbone)
        else:
            s1 = ModelBuilder.conv_block(history_input, base_f)
            p1 = layers.MaxPooling2D()(s1)
            s2 = ModelBuilder.conv_block(p1, base_f * 2)
            p2 = layers.MaxPooling2D()(s2)
            s3 = ModelBuilder.conv_block(p2, base_f * 4)
            p3 = layers.MaxPooling2D()(s3)
            s4 = ModelBuilder.conv_block(p3, base_f * 8)
            bridge = layers.MaxPooling2D()(s4)
            bridge = ModelBuilder.conv_block(bridge, base_f * 16)
            skips = [s4, s3, s2, s1]

        # 4. Bottleneck Injection (Absolute Position)
        # Modulates the global bridge features using the exact slice depth
        # x = FiLMLayer(name='bottleneck_film_abs')([bridge, p_abs_emb])

        # -------------------------------
        # # --- THE BOTTLENECK INJECTION ---
        # # 5. Two-Layer Adapter (MLP) with Zero-Weight Initialization
        # bridge_channels = K.int_shape(bridge)[-1]

        # # --- THE POSITIONAL PIPELINE ---
        # # 2. Glue input (N) and target (1) absolute positions into a single vector
        # p_combined = layers.Concatenate(axis=-1)([p_history_input, p_abs_input])

        # # 3. Fourier Encoding
        # fourier_feats = FourierEmbedding(num_freqs=8)(p_combined)

        # # Hidden layer
        # emb = layers.Dense(256, activation='swish', name='pos_mlp_hidden')(fourier_feats)

        # # Projection layer (ZERO INITIALIZED)
        # emb = layers.Dense(bridge_channels,
        #                    kernel_initializer='zeros',
        #                    bias_initializer='zeros',
        #                    name='pos_mlp_out')(emb)

        # # Reshape to (Batch, 1, 1, Channels) to broadcast spatially
        # emb_spatial = layers.Reshape((1, 1, bridge_channels))(emb)

        # # 6. Inject via ADDITION
        # x = layers.Add(name='bottleneck_pos_add')([bridge, emb_spatial])
        # # --------------------------------

        # --- THE CONDITIONAL POSITIONAL PIPELINE ---
        if config.model.use_positional_encoding:
            # Glue input (N) and target (1) absolute positions into a single vector
            p_combined = layers.Concatenate(axis=-1)([p_history_input, p_abs_input])
            fourier_feats = FourierEmbedding(num_freqs=8)(p_combined)

            bridge_channels = K.int_shape(bridge)[-1]
            emb = layers.Dense(256, activation='swish', name='pos_mlp_hidden')(fourier_feats)
            emb = layers.Dense(bridge_channels,
                               kernel_initializer='zeros',
                               bias_initializer='zeros',
                               name='pos_mlp_out')(emb)

            emb_spatial = layers.Reshape((1, 1, bridge_channels))(emb)
            x = layers.Add(name='bottleneck_pos_add')([bridge, emb_spatial])
        else:
            # Completely bypass positional encoding. Model becomes shift-invariant
            x = bridge
        # -------------------------------------------

        # 7. Decoder with SPADE and Skip Modulation
        for i, skip in enumerate(reversed(skips)):
            filters = decoder_filters[i] if i < len(decoder_filters) else base_f

            x = layers.Conv2D(filters * 4, 1, padding='same')(x)
            x = layers.Lambda(lambda z: tf.nn.depth_to_space(z, block_size=2))(x)

            # if skip is not None:
            #     # 8. Skip Connection Injection (Relative Position)
            #     # Modulates the local skip features using the sequence trajectory
            #     modulated_skip = FiLMLayer(name=f'skip_film_rel_{i}')([skip, p_rel_emb])

            #     x = layers.Concatenate()([x, modulated_skip])
            if skip is not None:
                x = layers.Concatenate()([x, skip])

            # 8. SPADE Block (Conditioned strictly on Spatial Mask)
            # SPADE Block receives ONLY the feature map and spatial mask
            x = SPADEResBlock(filters, 1)([x, mask_input])

        # 9. Final PixelShuffle
        if config.model.backbone:
            current_channels = K.int_shape(x)[-1]
            x = layers.Conv2D(current_channels * 4, 1, padding='same')(x)
            x = layers.Lambda(lambda z: tf.nn.depth_to_space(z, block_size=2))(x)

        # outputs = layers.Conv2D(1, 1, activation='sigmoid', name='reconstruction')(x)
        # Feature switch
        if config.model.num_hypotheses == 1:
            outputs = layers.Conv2D(1, 1, activation='sigmoid', name='reconstruction')(x)
        else:
            # Outputs shape: (Batch, H, W, M)
            outputs = layers.Conv2D(
                config.model.num_hypotheses,
                1,
                activation='sigmoid',
                name='reconstruction_mhp'
            )(x)

        # Return Multi-Input Model
        return models.Model(
            inputs=[history_input, mask_input, p_history_input, p_abs_input],
            outputs=outputs,
            name=f"{arch}_{config.model.backbone}"
        )

    @staticmethod
    def _build_spade_vae_generator(config: Config):
        arch = config.model.architecture.lower()

        # TODO: assert we're using target layer mask
        # 1. Inputs
        # Image Input: History [t-N ... t-1]
        # img_shape = (*config.data.padded_size, config.model.input_channels)
        # img_input = layers.Input(shape=img_shape, name='history_input')
        # Input: [History (N) + Mask (1)]
        total_channels = config.model.input_channels
        full_input = layers.Input(shape=(*config.data.padded_size, total_channels),
                                  name='full_input')

        # Split:
        # History: Channels [0 ... N-1]
        # Mask: Channel [-1] (assuming input_last_target_mask=True)
        N = config.data.neighborhood

        # Mask Input: Target Geometry [t] (Binary Mask)
        # Note: SPADE usually takes a 1-channel label map or one-hot.
        # We use 1-channel binary mask
        # mask_shape = (*config.data.padded_size, 1)
        # mask_input = layers.Input(shape=mask_shape, name='mask_input')
        # Slicing using Lambda layers to support serialization

        # History (first N channels) -> Goes to Encoder
        history_input = layers.Lambda(lambda x: x[..., :N], name='split_history')(full_input)

        # Mask (last channel) -> Goes to SPADE
        mask_input = layers.Lambda(lambda x: x[..., N:], name='split_mask')(full_input)

        base_f = config.model.base_filters

        # Encoder uses History
        bridge, skips = ModelBuilder.get_efficientnet_encoder(history_input,
                                                              config.model.backbone)

        # 2. Encoder (Processes History Images)
        if config.model.backbone:
            # bridge, skips = ModelBuilder.get_efficientnet_encoder(inputs, config.model.backbone)
            bridge, skips = ModelBuilder.get_efficientnet_encoder(history_input,
                                                                  config.model.backbone)
            # Decoder filters logic: scaling down from bridge
            # Bridge (B0=1280, B3=1536) -> 4 * base_f -> 2 * base_f -> ...
            decoder_filters = [base_f * 8, base_f * 4, base_f * 2, base_f]
        else:
            # Custom Encoder using base_f
            # s1 = ModelBuilder.conv_block(inputs, base_f)
            s1 = ModelBuilder.conv_block(history_input, base_f)
            p1 = layers.MaxPooling2D()(s1)

            s2 = ModelBuilder.conv_block(p1, base_f * 2)
            p2 = layers.MaxPooling2D()(s2)

            s3 = ModelBuilder.conv_block(p2, base_f * 4)
            p3 = layers.MaxPooling2D()(s3)

            s4 = ModelBuilder.conv_block(p3, base_f * 8)
            bridge = layers.MaxPooling2D()(s4)

            bridge = ModelBuilder.conv_block(bridge, base_f * 16)
            skips = [s4, s3, s2, s1]
            decoder_filters = [base_f * 8, base_f * 4, base_f * 2, base_f]

        # --- VAE BOTTLENECK START ---
        # 1. Project Bridge to latent parameters
        # Keep spatial dims (e.g. 5x5). Filters = latent_dim.
        # We can reuse base_filters * 16 (e.g. 1024) or reduce it.
        latent_dim = bridge.shape[-1]

        z_mean = layers.Conv2D(latent_dim, 1, padding='same', name='z_mean')(bridge)
        z_log_var = layers.Conv2D(latent_dim, 1, padding='same', name='z_log_var')(bridge)

        # 2. Re-parameterization Trick (Sampling)
        z = Sampling(name='z_sampling')([z_mean, z_log_var])

        # 3. Inject KL Loss (Internal to model)
        # This adds the loss to model.losses automatically
        VAELossLayer(weight=config.train.weight_kl)([z_mean, z_log_var])

        # 4. Decoder with SPADE (Conditioned on Mask)
        # Decoder uses Mask for SPADE
        # x = bridge
        # 4. Decoder starts from sampled Z
        x = z
        # --- VAE BOTTLENECK END ---

        for i, skip in enumerate(reversed(skips)):
            filters = decoder_filters[i] if i < len(decoder_filters) else base_f

            # Use PixelShuffle
            # x = layers.Conv2D(x.shape[-1] * 4, 1, padding='same')(x)
            x = layers.Conv2D(filters * 4, 1, padding='same')(x)
            x = layers.Lambda(lambda z: tf.nn.depth_to_space(z, block_size=2))(x)

            if skip is not None:
                x = layers.Concatenate()([x, skip])

            # Use SPADE Block
            # Input to SPADE is always the Raw Input Stack (inputs)
            # CRITICAL CHANGE:
            # SPADE Block conditioned on 'mask_input', NOT 'img_input'
            x = SPADEResBlock(filters, 1)([x, mask_input]) # 1 channel mask

        # Final
        if config.model.backbone:
            # Same PixelShuffle logic
            # x = layers.Conv2D(x.shape[-1] * 4, 1, padding='same')(x)
            # Safe logic using K.int_shape
            current_channels = K.int_shape(x)[-1]
            x = layers.Conv2D(current_channels * 4, 1, padding='same')(x)
            x = layers.Lambda(lambda z: tf.nn.depth_to_space(z, block_size=2))(x)

        outputs = layers.Conv2D(1, 1, activation='sigmoid', name='reconstruction')(x)
        return models.Model(inputs=full_input, outputs=outputs,
                            name=f"{arch}_{config.model.backbone}")
        # return models.Model(inputs=full_input, outputs=outputs,
        #                     name=f"spade_vae_{config.model.backbone}")

    @staticmethod
    def _build_unet(config: Config):
        arch = config.model.architecture.lower()

        # Copying the core build logic for completeness of context:
        # input_shape = (*config.data.padded_size, config.model.input_channels)
        # inputs = layers.Input(shape=input_shape, name='input_image')
        # x = inputs
        N = config.data.neighborhood

        # 1. Standardize Inputs to match the Data Pipeline (Dictionary format)
        history_input = layers.Input(shape=(*config.data.padded_size, N), name='history_input')
        mask_input = layers.Input(shape=(*config.data.padded_size, 1), name='mask_input')
        p_history_input = layers.Input(shape=(N,), name='p_history_input')
        p_abs_input = layers.Input(shape=(1,), name='p_abs_input')

        # 2. Concatenate history and mask for standard U-Net processing
        if config.data.input_last_target_mask:
            x = layers.Concatenate(axis=-1)([history_input, mask_input])
        else:
            x = history_input

        inputs_list =[history_input, mask_input, p_history_input, p_abs_input]

        if config.model.use_stem:
            x = layers.Conv2D(32, 3, padding='same')(x)
            x = layers.BatchNormalization()(x)
            x = layers.Activation(config.model.activation)(x)

        if config.model.backbone:
            bridge, skips = ModelBuilder.get_efficientnet_encoder(x, config.model.backbone)
            # Decoder filters logic: scaling down from bridge
            # B0 bridge is 1280.
            # We want to smooth the transition
            decoder_filters = [192, 80, 40, 24]
            # decoder_filters = [256, 128, 64, 32]
        else:
            s1 = ModelBuilder.conv_block(x, 32)
            p1 = layers.MaxPooling2D()(s1)
            s2 = ModelBuilder.conv_block(p1, 64)
            p2 = layers.MaxPooling2D()(s2)
            s3 = ModelBuilder.conv_block(p2, 128)
            p3 = layers.MaxPooling2D()(s3)
            s4 = ModelBuilder.conv_block(p3, 256)
            bridge = layers.MaxPooling2D()(s4)
            bridge = ModelBuilder.conv_block(bridge, 512)
            skips = [s4, s3, s2, s1]
            decoder_filters = [256, 128, 64, 32]

        x = bridge
        for i, skip in enumerate(reversed(skips)):
            filters = decoder_filters[i] if i < len(decoder_filters) else 16
            x = ModelBuilder.up_block(x, skip, filters, config.model.activation,
                                      config.model.use_attention)

        if config.model.backbone:
             x = layers.UpSampling2D((2,2), interpolation='bilinear')(x)

        # outputs = layers.Conv2D(1, 1, activation='sigmoid', name='reconstruction')(x)
        # 3. Feature switch to match output shapes across ablations
        if config.model.num_hypotheses == 1:
            outputs = layers.Conv2D(1, 1, activation='sigmoid', name='reconstruction')(x)
        else:
            outputs = layers.Conv2D(
                config.model.num_hypotheses,
                1,
                activation='sigmoid',
                name='reconstruction_mhp'
            )(x)

        return models.Model(inputs=inputs_list, outputs=outputs,
                            name=f"{arch}_{config.model.backbone}")

# Cell 9: Loss Functions
- **Target File:** src/models/losses.py
- **Content:**
    - CompositeLoss class (Inherits from tf.keras.losses.Loss)
    - Perceptual Loss (VGG/EffNet) lazy initialization logic
    - Spatially Weighted Loss

In [ ]:
from tensorflow.keras import applications, models

In [ ]:
class PerceptualLoss(tf.keras.losses.Loss):
    """
    Computes distance in EfficientNet feature space.
    Reusable component for U-Net and GAN training.
    """
    def __init__(self, name='perceptual_loss', backbone='effnet',
                 weights='imagenet', input_shape=(None, None, 3)):
        # tf.keras.backend.clear_session() # may cause memory leak
        super().__init__(name=name)

        self.backbone = backbone

        if self.backbone == 'vgg':
            logger.info("Initializing VGG19 Perceptual Model...")

            # Use VGG19. It avoids conflict with EfficientNet generator backbones.
            # VGG expects (224, 224, 3) but works with (None, None, 3)
            base = applications.VGG19(
                include_top=False, weights=weights, # input_shape=(None, None, 3)
            )

        else:
            # --- Eager Initialization of Perceptual Model ---
            # We initialize this HERE (in eager mode) so it's ready before model.fit() builds the graph.
            # Otherwise, loading weights inside the training loop causes 'InaccessibleTensorError'
            logger.info("Initializing EfficientNetB0 Perceptual Model...")

            # Load backbone without top layers
            base = applications.EfficientNetB0(
                # name='effnet_perceptual',
                include_top=False,
                weights=weights, # 'imagenet'
            )

        base.trainable = False

        if self.backbone == 'vgg':
            # Standard layers for style/texture loss:
            # block1_conv1, block2_conv1, block3_conv1, block4_conv1, block5_conv1
            layer_names = [
                'block1_conv1', 'block2_conv1', 'block3_conv1', 'block4_conv1', 'block5_conv1'
            ]
            outputs = [base.get_layer(name).output for name in layer_names]

            self.model = models.Model(base.input, outputs, name="extractor_vgg")
        else:
            # Select specific feature layers for texture comparison
            # 'block2a' and 'block3a' capture low-to-mid level features (edges, textures):
            # 'block2a' (low-level texture) and 'block3a' (mid-level shape)
            outputs = [
                base.get_layer(n).output for n in [
                    'block2a_expand_activation', 'block3a_expand_activation'
                ]
            ]

            self.model = models.Model(base.input, outputs,
                                      name="extractor_effnet")

    def call(self, y_true, y_pred):
        # Expects single channel 0-1 inputs
        # Convert to 3-channel RGB 0-255 for EfficientNet
        true_rgb = tf.image.grayscale_to_rgb(y_true) * 255.0
        pred_rgb = tf.image.grayscale_to_rgb(y_pred) * 255.0

        # VGG Preprocessing: expects 0-255 BGR, but we have 0-1 RGB.
        # Simple shift: scale to 0-255.
        # Keras VGG preprocess_input does mean subtraction, let's do a rough approx
        # or use the official function if possible
        if self.backbone == 'vgg':
            # Official preprocessing (handles mean subtraction)
            true_rgb = applications.vgg19.preprocess_input(true_rgb)
            pred_rgb = applications.vgg19.preprocess_input(pred_rgb)

        feats_true = self.model(true_rgb)
        feats_pred = self.model(pred_rgb)

        loss = 0.0
        for f_t, f_p in zip(feats_true, feats_pred):
            loss += tf.reduce_mean(tf.abs(f_t - f_p))

        return loss


_GLOBAL_PERCEPTUAL_LOSS = None


def get_perceptual_loss(config: Config):
    global _GLOBAL_PERCEPTUAL_LOSS
    if _GLOBAL_PERCEPTUAL_LOSS is None:
        _GLOBAL_PERCEPTUAL_LOSS = PerceptualLoss(
            backbone=config.train.perceptual_backbone,
            weights=config.train.perceptual_init,
            input_shape=(*config.data.padded_size, 3)
        )
    return _GLOBAL_PERCEPTUAL_LOSS

In [ ]:
class SpectralLoss(tf.keras.losses.Loss):
    """
    Computes the L1 distance between the Log-Magnitude of the Fourier Transforms.
    Forces the model to match the frequency distribution (sharpness/texture) of the target.
    """
    def __init__(self, name="spectral_loss"):
        super().__init__(name=name)

    def call(self, y_true, y_pred):
        # 1. Cast to float32 (FFT requires it, and mixed precision might be float16)
        y_true_f32 = tf.cast(y_true[..., 0], tf.float32) # take 1st channel (Image)
        y_pred_f32 = tf.cast(y_pred[..., 0], tf.float32)

        # 2. Compute 2D Real FFT
        # Output shape: (Batch, H, W/2 + 1) complex64
        fft_true = tf.signal.rfft2d(y_true_f32)
        fft_pred = tf.signal.rfft2d(y_pred_f32)

        # 3. Compute Magnitude
        mag_true = tf.abs(fft_true)
        mag_pred = tf.abs(fft_pred)

        # 4. Log Scaling (Critical for balancing low/high freqs)
        # Add epsilon or 1.0 to avoid log(0)
        log_true = tf.math.log(mag_true + 1.0)
        log_pred = tf.math.log(mag_pred + 1.0)

        # 5. L1 Distance in Frequency Domain
        return tf.reduce_mean(tf.abs(log_true - log_pred))


class FocalFrequencyLoss(tf.keras.losses.Loss):
    """
    Focal Frequency Loss (FFL) for Image Reconstruction.
    Based on: https://arxiv.org/pdf/2012.12821
    Penalizes both magnitude and phase differences in the complex frequency domain,
    dynamically up-weighting frequencies where the model performs poorly.
    """
    def __init__(self, alpha: float = 1.0, name: str = "focal_frequency_loss"):
        super().__init__(name=name)
        self.alpha = alpha

    def call(self, y_true: tf.Tensor, y_pred: tf.Tensor) -> tf.Tensor:
        # 1. Cast to complex64 for FFT
        # Assuming inputs are (B, H, W, 1), we strip the channel dim for 2D FFT
        y_true_c = tf.cast(y_true[..., 0], tf.complex64)
        y_pred_c = tf.cast(y_pred[..., 0], tf.complex64)

        # 2. Compute 2D Complex FFT
        fft_true = tf.signal.fft2d(y_true_c)
        fft_pred = tf.signal.fft2d(y_pred_c)

        # 3. Shift zero-frequency component to center
        fft_true = tf.signal.fftshift(fft_true, axes=[1, 2])
        fft_pred = tf.signal.fftshift(fft_pred, axes=[1, 2])

        # 4. Calculate complex difference matrix d(u,v) = |F_r - F_f|^2
        # tf.abs on complex returns the magnitude (sqrt(real^2 + imag^2)).
        # Squaring it gives the squared distance
        diff_matrix = tf.square(tf.abs(fft_true - fft_pred))

        # 5. Normalize difference matrix to [0, 1] per image in the batch
        max_diff = tf.reduce_max(diff_matrix, axis=[1, 2], keepdims=True)
        # Safe division to prevent NaNs
        diff_norm = tf.math.divide_no_nan(diff_matrix, max_diff)

        # 6. Calculate focal weights w(u,v) = d_norm(u,v)^alpha
        weight_matrix = tf.pow(diff_norm, self.alpha)

        # 7. Compute final weighted frequency loss
        loss = tf.reduce_mean(weight_matrix * diff_matrix)

        return loss


class CompositeLoss(tf.keras.losses.Loss):
    def __init__(self, config: Config, name="composite_loss"):
        super().__init__(name=name)

        self.cfg = config.train

        # Only initialize the heavy perceptual model if we are actually going to use it!
        if self.cfg.lambda_perceptual >= 0.01:
            self.perceptual = get_perceptual_loss(config)
        else:
            self.perceptual = None

        self.spectral = SpectralLoss()

        # FIX: Convert static floats to dynamic TensorFlow Variables
        self.w_healthy = tf.Variable(self.cfg.lambda_healthy, dtype=tf.float32,
                                     trainable=False)
        self.w_tumor = tf.Variable(self.cfg.lambda_tumor, dtype=tf.float32,
                                   trainable=False)
        self.w_bg = tf.Variable(self.cfg.lambda_background, dtype=tf.float32,
                                trainable=False)
        self.w_grad = tf.Variable(self.cfg.lambda_grad, dtype=tf.float32,
                                  trainable=False)
        self.w_perc = tf.Variable(self.cfg.lambda_perceptual, dtype=tf.float32,
                                  trainable=False)
        self.w_spec = tf.Variable(self.cfg.lambda_spectral, dtype=tf.float32,
                                  trainable=False)

    def call(self, y_true, y_pred):
        """
        y_true: (B, H, W, 4) -> [Image, TumorMask, DetailedMask, BrainMask]
        y_pred: (B, H, W, 1)
        """
        # Unpack
        gt_img = y_true[..., 0:1]
        tumor_mask = y_true[..., 1:2]
        brain_mask = y_true[..., 3:4]

        # 1. Pixel Losses (MAE)
        mae = tf.abs(gt_img - y_pred)

        # Healthy Tissue (Inside brain, outside tumor)
        # Fix: Robust Division
        healthy_mask = tf.maximum(0.0, brain_mask - tumor_mask)
        loss_healthy = tf.math.divide_no_nan(
            tf.reduce_sum(mae * healthy_mask),
            tf.reduce_sum(healthy_mask)# + 1e-8
        )

        # Tumor Region (The Inpainting Target - Critical)
        # Fix: Robust Division prevents NaN when batch has no tumor
        loss_tumor = tf.math.divide_no_nan(
            tf.reduce_sum(mae * tumor_mask),
            tf.reduce_sum(tumor_mask)# + 1e-8
        )

        # Background
        bg_mask = 1.0 - brain_mask
        loss_bg = tf.math.divide_no_nan(
            tf.reduce_sum(mae * bg_mask),
            tf.reduce_sum(bg_mask)# + 1e-8
        )

        # 2. Gradient Loss (Sharpness)
        dy_true, dx_true = tf.image.image_gradients(gt_img)
        dy_pred, dx_pred = tf.image.image_gradients(y_pred)
        grad_loss = tf.reduce_mean(tf.abs(dy_true - dy_pred) +
                                   tf.abs(dx_true - dx_pred))

        # 3. Perceptual Loss (Conditional Bypass)
        if self.perceptual is not None:
            perc_loss = self.perceptual(gt_img, y_pred)
        else:
            perc_loss = tf.constant(0.0, dtype=tf.float32)

        # 4. Spectral Loss
        spec_loss = self.spectral(y_true, y_pred)

        # Combine weighted losses
        # FIX: Multiply by the dynamic Variables
        total = (self.w_healthy * loss_healthy +
                 self.w_tumor * loss_tumor +
                 self.w_bg * loss_bg +
                 self.w_grad * grad_loss +
                 self.w_perc * perc_loss +
                 self.w_spec * spec_loss)

        return total


class SpatiallyWeightedL1Loss(tf.keras.losses.Loss):
    """
    L1 Loss that applies heavy penalties to Brain and Tumor regions.
    Ported from SPADE GAN logic: weights = 1 (Bg) + 14 (Brain) + 10 (Tumor).
    """
    def __init__(self, config: Config, name="spatial_l1_loss"):
        super().__init__(name=name)
        self.w_brain_add = 14.0 # base 1 + 14 = 15
        self.w_tumor_add = 10.0 # 15 + 10 = 25 for tumor regions

    def call(self, y_true, y_pred):
        # Unpack
        gt_img = y_true[..., 0:1]
        tumor_mask = y_true[..., 1:2]
        brain_mask = y_true[..., 3:4]

        # Absolute Error
        l1_error = tf.abs(gt_img - y_pred)

        # Construct Weight Map
        # Start with Background (1.0)
        weights = tf.ones_like(l1_error)

        # Add Brain Penalty
        weights = weights + (brain_mask * self.w_brain_add)

        # Add Tumor Penalty
        weights = weights + (tumor_mask * self.w_tumor_add)

        # Weighted Mean
        return tf.reduce_mean(l1_error * weights)

In [ ]:
import tensorflow as tf

class RelaxedMHPLossWrapper(tf.keras.losses.Loss):
    def __init__(self, base_loss_fn, num_hypotheses=1, epsilon=0.05, **kwargs):
        # We handle reduction manually
        super().__init__(reduction=tf.keras.losses.Reduction.NONE, **kwargs)
        self.base_loss_fn = base_loss_fn
        self.M = num_hypotheses
        self.epsilon = epsilon

        if self.M > 1:
            self.winner_weight = 1.0 - epsilon
            self.loser_weight = epsilon / (self.M - 1)

    def call(self, y_true, y_pred):
        # BASELINE FALLBACK:
        if self.M == 1:
            return self.base_loss_fn(y_true, y_pred)

        # MULTIPLE HYPOTHESES LOGIC:
        # y_true shape: (B, H, W, 1)
        # y_pred shape: (B, H, W, M)

        # 1. Unstack the M predictions into a list of M tensors of shape (B, H, W)
        preds = tf.unstack(y_pred, axis=-1)

        # 2. Calculate base CompositeLoss for each hypothesis
        # self.base_loss_fn should return shape (B,)
        losses =[]
        for p in preds:
            p_expanded = tf.expand_dims(p, -1) # Restore channel dim -> (B, H, W, 1)
            losses.append(self.base_loss_fn(y_true, p_expanded))

        # Stack losses -> Shape: (M, Batch)
        losses_tensor = tf.stack(losses, axis=0)

        # 3. Find the best hypothesis for each batch item (The Oracle)
        best_indices = tf.argmin(losses_tensor, axis=0) # shape: (Batch,)

        # 4. Create masks for winners and losers
        winner_mask = tf.one_hot(best_indices, depth=self.M) # shape: (Batch, M)
        winner_mask = tf.transpose(winner_mask)              # shape: (M, Batch)

        # 5. Apply Epsilon Relaxation
        weights = (winner_mask * self.winner_weight) + ((1.0 - winner_mask) * self.loser_weight)

        # 6. Weight the losses and sum across the M dimension
        relaxed_losses = tf.reduce_sum(losses_tensor * weights, axis=0) # shape: (Batch,)

        return relaxed_losses

# Cell 10 (Revised): Training Visualization Callback
- **Target File:** src/training/callbacks.py
- **Content:**
    - TrainingVisualizer class
    - Logic to plot Input(t-1), GT(t), and Pred(t) every N epochs

In [ ]:
import matplotlib.pyplot as plt


class TrainingVisualizer(tf.keras.callbacks.Callback):
    """
    Visualizes reconstruction performance on a fixed validation batch.
    Plots: Input (t-1), Ground Truth (t), Prediction (t), Error Map.
    """
    def __init__(self, val_dataset, config, frequency=5, num_samples=3):
        super().__init__()
        self.cfg = config  # store config
        self.frequency = max(1, frequency)
        self.num_samples = num_samples

        # Take one batch for consistent visualization
        # self.vis_batch = next(iter(val_dataset.take(1)))
        # --- FIX: Support both tf.data.Dataset and Keras Sequence ---
        if hasattr(val_dataset, 'take'):
            # It's a tf.data.Dataset
            self.vis_batch = next(iter(val_dataset.take(1)))
        else:
            # It's a Keras Sequence (like HpoValidSequence)
            # Just grab the 0th batch directly!
            self.vis_batch = val_dataset[0]

    def on_epoch_end(self, epoch, logs=None):
        if (epoch == 0) or ((epoch + 1) % self.frequency == 0):
            # Print the first epoch for comparison
            self._plot_results(epoch + 1)

    def _plot_results(self, epoch):
        # Slice the batch to only what we need to plot
        # This saves VRAM during the predict call
        input_batch, target_batch = self.vis_batch

        # Take only N samples
        n = self.num_samples
        # Slice the batch (TensorFlow handles slicing dictionaries of tensors)
        input_micro_batch = {k: v[:n] for k, v in input_batch.items()}
        target_micro_batch = target_batch[:n]

        # Predict on smaller batch
        # pred_batch = self.model(input_micro_batch, training=False)
        # 1. Raw MHP Prediction
        pred_batch_raw = self.model(input_micro_batch, training=False)

        # 2. Extract Oracle & Uncertainty
        pred_batch_oracle = get_oracle_prediction(target_micro_batch,
                                                  pred_batch_raw)

        if self.cfg.model.num_hypotheses > 1:
            uncertainty_batch = tf.math.reduce_variance(pred_batch_raw, axis=-1,
                                                        keepdims=True)
            # Normalize to 0-1 for nice visualization
            uncertainty_batch = uncertainty_batch / (tf.reduce_max(uncertainty_batch,
                                                                   axis=[1,2,3],
                                                                   keepdims=True) + 1e-8)
        else:
            uncertainty_batch = tf.zeros_like(pred_batch_oracle)

        # Extract the image data from the dictionary for plotting
        # 'history_input' contains the t-N...t-1 slices
        inputs = input_micro_batch['history_input'].numpy()
        targets = target_micro_batch.numpy()

        # Config for plot
        # cols = 4
        cols = 5  # added column for Uncertainty
        rows = self.num_samples
        fig, axes = plt.subplots(rows, cols, figsize=(15, 3.5 * rows))
        fig.suptitle(f"Epoch {epoch} Visualization", fontsize=16)

        for i in range(self.num_samples):
            # Dynamic Indexing Logic:
            # Input channels are [t-N, ... t-2, t-1].
            # If input_last_target_mask is True, there is an extra channel at the end.
            # However, the image stack always comes first.
            # Therefore, the index of 't-1' is always (neighborhood - 1)
            idx_t_minus_1 = self.cfg.data.neighborhood - 1

            # Handle if channel is mask (binary)
            img_t_minus_1 = inputs[i, :, :, idx_t_minus_1]

            # 2. GT
            img_gt = targets[i, :, :, 0]

            # 3. Pred
            # img_pred = pred_batch[i, :, :, 0]
            # Using Oracle Prediction and Uncertainty map
            img_pred = pred_batch_oracle[i, :, :, 0].numpy()
            img_unc = uncertainty_batch[i, :, :, 0].numpy()

            # 4. Diff
            diff = np.abs(img_gt - img_pred)

            # Plot
            ax = axes[i] if rows > 1 else axes

            ax[0].imshow(img_t_minus_1, cmap='bone')
            ax[0].set_title("Input (t-1)")
            ax[0].axis('off')

            ax[1].imshow(img_gt, cmap='bone')
            ax[1].set_title("Target (t)")
            ax[1].axis('off')

            ax[2].imshow(img_pred, cmap='bone')
            ax[2].set_title(f"Pred (MSE = {np.mean(diff ** 2):.4f})")
            ax[2].axis('off')

            ax[3].imshow(diff, cmap='inferno', vmin=0, vmax=0.3)
            ax[3].set_title("Error |Pred - Target|")
            ax[3].axis('off')

            # New Uncertainty Column
            ax[4].imshow(img_unc, cmap='magma')
            ax[4].set_title("Uncertainty (Variance)")
            ax[4].axis('off')

        plt.tight_layout()
        plt.show()

        plt.close(fig) # try to release RAM

# Cell 11: Updated Trainer with Callbacks
- **Target File:** src/training/callbacks.py (Part 1) AND src/training/trainer.py (Part 2)
- **Content:**
    - BufferUpdateCallback -> Goes to src/training/callbacks.py
    - Trainer class -> Goes to src/training/trainer.py

In [ ]:
class BufferUpdateCallback(tf.keras.callbacks.Callback):
    """
    Updates the Hallucination Buffer using the Active Data Manager.
    Samples directly from the currently loaded pool in RAM.
    """
    def __init__(self, manager, samples_per_epoch=512, rollout_depth=3):
        super().__init__()
        self.manager = manager
        self.samples = samples_per_epoch
        self.cfg = manager.cfg
        self.rollout_depth = rollout_depth

        self.w_ixi = self.cfg.data.ixi_sampling_weight
        self.w_ixi = max(0.1, min(0.9, self.w_ixi))
        self.w_brats = 1.0 - self.w_ixi

    def on_epoch_end(self, epoch, logs=None):
        warmup = max(1, self.cfg.aug.prob_hallucination_warmup)
        current_prob = min(
            self.cfg.aug.prob_hallucination_max,
            ((epoch + 1) / warmup) * self.cfg.aug.prob_hallucination_max
        )

        self.cfg.aug.prob_hallucination_replay = current_prob

        # Gate Optimization: don't populate buffer if it's not being used
        if current_prob < 1e-4:
            return

        logger.debug(f"Populating hallucination buffer (Next Epoch Prob: {current_prob:.2f})")

        batch_size = self.cfg.train.batch_size // 2
        total_batches = max(1, self.samples // batch_size)

        num_samples_processed = 0

        for _ in range(total_batches):
            contexts =[]

            # A. Fill Initial Contexts
            for _ in range(batch_size):
                if random.random() < self.w_ixi:
                    vol = self.manager.get_volume('ixi')
                else:
                    vol = self.manager.get_volume('brats')

                if vol is None: continue

                axis = random.choice(self.cfg.data.projections)
                candidates = vol.indices['clean'][axis]
                N = self.cfg.data.neighborhood

                if len(candidates) < N + self.rollout_depth + 2: continue

                # Pick start so we have room to roll out
                start_idx = random.choice(candidates[:-self.rollout_depth])
                dim_size = vol.shape[axis]
                if start_idx + N >= dim_size: continue

                full_stack = GeneratorBase._get_stack(vol.t1, axis, start_idx, N)
                if full_stack is None: continue

                stack = full_stack[:-1].copy()
                input_stack = np.transpose(stack, (1, 2, 0))

                direction = 'forward'

                if axis == 0:   view = vol.t1
                elif axis == 1: view = np.moveaxis(vol.t1, 1, 0)
                else:           view = np.moveaxis(vol.t1, 2, 0)

                contexts.append({
                    'stack': input_stack, # (H, W, N)
                    'idx': start_idx + N, # the index we are about to predict
                    'path': vol.path,
                    'axis': axis,         # Storing axis for the key
                    'view': view,
                    'native_h': view.shape[1],
                    'native_w': view.shape[2],
                    'dead': False         # Flag to stop NaN propagation
                })

            if not contexts: continue

            # B. Perform Rollout
            for step in range(self.rollout_depth):
                batch_x = []
                valid_indices =[]

                for i, ctx in enumerate(contexts):
                    # Skip dead contexts (e.g., hit a NaN previously) or out of bounds
                    if ctx['dead'] or ctx['idx'] >= ctx['view'].shape[0]:
                        continue

                    target_slice = ctx['view'][ctx['idx']]

                    true_start_idx = (
                        (ctx['idx'] - N) if direction == 'forward'
                        else min(ctx['idx'] + N, ctx['view'].shape[0] - 1)
                    )

                    model_inputs, scale, pads = InputProcessor.prepare_model_input(
                        input_stack_native=ctx['stack'],
                        target_slice_native=target_slice,
                        start_idx=true_start_idx,
                        target_idx=ctx['idx'],
                        axis_size=ctx['view'].shape[0],
                        direction=direction,
                        config=self.cfg
                    )

                    hist_pad, scale, pads = GeometryOps.resize_and_pad(
                        tf.convert_to_tensor(model_inputs['history_input']),
                        self.cfg.data.padded_size, 'bicubic'
                    )
                    mask_pad, _, _ = GeometryOps.resize_and_pad(
                        tf.convert_to_tensor(model_inputs['mask_input']),
                        self.cfg.data.padded_size, 'nearest'
                    )
                    model_inputs['history_input'] = hist_pad
                    model_inputs['mask_input'] = mask_pad

                    batch_x.append(model_inputs)
                    valid_indices.append(i)

                if not batch_x: break

                try:
                    keys = batch_x[0].keys()
                    inputs = {
                        k: tf.stack(
                            [sample[k] for sample in batch_x]
                        ) for k in keys
                    }
                except Exception as e:
                    logger.error(f"BufferUpdateCallback Batching Failed: {e}")
                    break

                # Predict
                # preds = self.model(inputs, training=False)
                preds_raw = self.model(inputs, training=False)

                # MHP Fix: Collapse hypotheses to a single stable image for the rollout
                if self.cfg.model.num_hypotheses > 1:
                    # preds = tf.reduce_mean(preds_raw, axis=-1, keepdims=True)
                    # Pick a random hypothesis index [0, M-1] (enhanced robustness)
                    idx = tf.random.uniform(
                        shape=[], minval=0, maxval=self.cfg.model.num_hypotheses, dtype=tf.int32
                    )
                    preds = preds_raw[..., idx:idx+1]
                else:
                    preds = preds_raw

                # C. Update Contexts & Store
                for k, ctx_idx in enumerate(valid_indices):
                    ctx = contexts[ctx_idx]
                    pred_padded = preds[k]

                    _, scale, pads = GeometryOps.resize_and_pad(
                        tf.zeros((ctx['native_h'], ctx['native_w'], 1)),
                        self.cfg.data.padded_size
                    )

                    pred_restored = GeometryOps.inverse_resize_pad(
                        pred_padded, (ctx['native_h'], ctx['native_w']),
                        scale, pads
                    ).numpy()[:, :, 0]

                    # NaN Check ("Poison Control")
                    if np.isnan(pred_restored).any() or np.isinf(pred_restored).any():
                        logger.warning(f"⚠️ NaN detected in Hallucination Rollout! Killing sequence for {ctx['path']}")
                        ctx['dead'] = True  # Permanently disable this sequence for the rest of the rollout
                        continue

                    # STORE RESULT USING PROPER AXIS-AWARE KEY
                    self.manager.update_hallucination(
                        (ctx['path'], ctx['axis'], ctx['idx']), pred_restored
                    )
                    num_samples_processed += 1

                    # ROLL THE STACK
                    new_stack = np.concatenate([ctx['stack'][..., 1:],
                                                pred_restored[..., None]],
                                               axis=-1)
                    ctx['stack'] = new_stack
                    ctx['idx'] += 1

                # --- CRITICAL LEAK FIX: Clear Eager Tensors ---
                # Forces TF memory pool to release these before the next step
                if 'inputs' in locals(): del inputs
                if 'preds_raw' in locals(): del preds_raw
                if 'preds' in locals(): del preds
                del batch_x, valid_indices
                # ----------------------------------------------

        logger.debug(f"Hallucination buffer updated: {num_samples_processed} samples")

In [ ]:
class GeneratorCheckpoint(tf.keras.callbacks.Callback):
    """
    Saves ONLY the Generator sub-model from the GAN Trainer.
    """
    def __init__(self, filepath, monitor='val_l1_loss', save_best_only=True, mode='min'):
        super().__init__()
        self.filepath = filepath
        self.monitor = monitor
        self.save_best_only = save_best_only
        self.mode = mode
        self.best = np.inf if mode == 'min' else -np.inf

    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is None: return

        if self.save_best_only:
            if self.mode == 'min':
                improved = current < self.best
            else:
                improved = current > self.best

            if improved:
                self.best = current
                # Save the GENERATOR, not self.model (which is the Trainer)
                self.model.generator.save(self.filepath) # .keras format
                logger.info(f"\nEpoch {epoch + 1}: Generator saved to {self.filepath}")
        else:
            self.model.generator.save(self.filepath)

In [ ]:
class SPADEGANTrainer(tf.keras.Model):
    def __init__(self, generator, discriminator, config, extra_metrics=None):
        super().__init__()
        self.generator = generator
        self.discriminator = discriminator
        self.cfg = config

        # Reuse existing loss classes
        self.spatial_l1 = SpatiallyWeightedL1Loss(config)
        # Conditional initialization
        if self.cfg.train.weight_perceptual >= 0.01:
            self.perceptual = get_perceptual_loss(config)
        else:
            self.perceptual = None
        self.spectral = SpectralLoss()

        # --- FIX: Convert GAN weights to dynamic TF Variables ---
        self.w_gan = tf.Variable(self.cfg.train.weight_gan, dtype=tf.float32,
                                 trainable=False)
        self.w_fm = tf.Variable(self.cfg.train.weight_fm, dtype=tf.float32,
                                trainable=False)
        self.w_l1 = tf.Variable(self.cfg.train.weight_l1, dtype=tf.float32,
                                trainable=False)
        self.w_perc = tf.Variable(self.cfg.train.weight_perceptual,
                                  dtype=tf.float32, trainable=False)
        self.w_spec = tf.Variable(self.cfg.train.weight_spectral,
                                  dtype=tf.float32, trainable=False)
        self.w_kl = tf.Variable(self.cfg.train.weight_kl, dtype=tf.float32,
                                trainable=False)

        # Trackers
        self.g_loss_tracker = tf.keras.metrics.Mean(name="g_loss")
        self.d_loss_tracker = tf.keras.metrics.Mean(name="d_loss")
        self.l1_tracker = tf.keras.metrics.Mean(name="l1_loss")
        self.gan_tracker = tf.keras.metrics.Mean(name="gan_loss")
        self.perc_tracker = tf.keras.metrics.Mean(name="perc_loss")
        self.spec_tracker = tf.keras.metrics.Mean(name="spec_loss")
        if self.cfg.model.architecture == 'vae':
            self.kl_tracker = tf.keras.metrics.Mean(name="kl_loss")

        # Extra Metrics (MAE, SSIM, etc.)
        self.extra_metrics = extra_metrics if extra_metrics else []

    def compile(self, g_optimizer, d_optimizer, **kwargs):
        super().compile(**kwargs)
        self.g_optimizer = g_optimizer
        self.d_optimizer = d_optimizer

    @property
    def metrics(self):
        return [
            self.g_loss_tracker, self.d_loss_tracker, self.l1_tracker,
            self.gan_tracker, self.perc_tracker, self.spec_tracker
        ] + self.extra_metrics

    def train_step(self, data):
        x, y = data
        real_img = y[..., 0:1]

        # 1. Train Discriminator
        with tf.GradientTape() as tape:
            fake_img = self.generator(x, training=True)

            # D returns list: [validity, feat1, feat2, ...]
            d_real_out = self.discriminator([real_img, x], training=True)
            d_fake_out = self.discriminator([fake_img, x], training=True)

            # The first element is the validity score
            d_real_logits = d_real_out[0]
            d_fake_logits = d_fake_out[0]

            # Fix: Cast to float32 for stable loss calc
            d_real_logits = tf.cast(d_real_logits, tf.float32)
            d_fake_logits = tf.cast(d_fake_logits, tf.float32)

            # Hinge Loss
            d_loss = tf.reduce_mean(tf.nn.relu(1.0 - d_real_logits)) + \
                     tf.reduce_mean(tf.nn.relu(1.0 + d_fake_logits))

            # Loss scaling for Mixed Precision (if using standard Optimizer.minimize it's auto,
            # but with manual GradientTape we often need explicit scaling if not handled by optimizer wrapper)
            # Modern Keras optimizers usually handle this if 'jit_compile=True' or standard fit.
            # But let's assume standard float32 calc is enough

        d_grads = tape.gradient(d_loss, self.discriminator.trainable_weights)
        self.d_optimizer.apply_gradients(
            zip(d_grads, self.discriminator.trainable_weights)
        )

        # 2. Train Generator
        with tf.GradientTape() as tape:
            fake_img = self.generator(x, training=True)
            d_fake_out = self.discriminator([fake_img, x], training=False)
            d_real_out = self.discriminator([real_img, x], training=False) # get real features too

            d_fake_logits = d_fake_out[0]

            # A. GAN Loss
            g_gan_loss = -tf.reduce_mean(d_fake_logits)

            # B. Feature Matching Loss (NEW)
            g_fm_loss = 0.0
            # Iterate over features (index 1 onwards)
            for real_feat, fake_feat in zip(d_real_out[1:], d_fake_out[1:]):
                g_fm_loss += tf.reduce_mean(tf.abs(real_feat - fake_feat))

            # C. Spatial L1 Loss
            g_l1_loss = self.spatial_l1(y, fake_img)

            # D. Perceptual Loss (Conditional Bypass)
            if self.perceptual is not None:
                g_perc_loss = self.perceptual(real_img, fake_img)
            else:
                g_perc_loss = tf.constant(0.0, dtype=tf.float32)

            # E. Spectral Loss
            g_spec_loss = self.spectral(y, fake_img)

            # Ensure all losses are float32 before weighted sum
            g_gan_loss = tf.cast(g_gan_loss, tf.float32)
            g_fm_loss = tf.cast(g_fm_loss, tf.float32)
            g_l1_loss = tf.cast(g_l1_loss, tf.float32)
            g_perc_loss = tf.cast(g_perc_loss, tf.float32)
            g_spec_loss = tf.cast(g_spec_loss, tf.float32)

            # Total
            # --- FIX: Multiply by dynamic TF Variables instead of cfg floats ---
            total_g_loss = (
                (g_gan_loss * self.w_gan) +
                (g_fm_loss * self.w_fm) +
                (g_l1_loss * self.w_l1) +
                (g_perc_loss * self.w_perc) +
                (g_spec_loss * self.w_spec)
            )

            # FIX: Add Model Internal Losses (KL Divergence)
            if self.cfg.model.architecture == 'vae' and self.generator.losses:
                # Sum up all regularization losses (KL)
                kl_loss_sum = tf.reduce_sum(self.generator.losses)
                # total_g_loss += kl_loss_sum
                # Multiply KL by dynamic weight
                total_g_loss += (kl_loss_sum * self.w_kl)

                # Optional: Track it
                self.kl_tracker.update_state(kl_loss_sum) # (if there is a tracker)

        g_grads = tape.gradient(total_g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(
            zip(g_grads, self.generator.trainable_weights)
        )

        # Metrics
        self.g_loss_tracker.update_state(total_g_loss)
        self.d_loss_tracker.update_state(d_loss)
        self.l1_tracker.update_state(g_l1_loss)
        self.gan_tracker.update_state(g_gan_loss)
        self.perc_tracker.update_state(g_perc_loss)
        self.spec_tracker.update_state(g_spec_loss)

        # Update Extra Metrics (MAE, SSIM)
        # They compare y (Real) vs fake_img
        for m in self.extra_metrics:
            m.update_state(y, fake_img)

        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        # Validation Logic
        x, y = data
        real_img = y[..., 0:1]
        fake_img = self.generator(x, training=False)

        # Calculate L1 only for validation tracking
        l1 = self.spatial_l1(y, fake_img)
        self.l1_tracker.update_state(l1)

        # Update Extra Metrics
        for m in self.extra_metrics:
            m.update_state(y, fake_img)

        return {m.name: m.result() for m in self.metrics}

    def call(self, inputs):
        # Used for validation/inference
        return self.generator(inputs)

In [ ]:
class Trainer:
    def __init__(self, config: Config, model, train_ds, val_ds, data_manager,
                 train_steps=None, val_steps=None):
        self.cfg = config
        self.model = model
        self.train_ds = train_ds
        self.val_ds = val_ds
        self.manager = data_manager
        self.train_steps = train_steps
        self.val_steps = val_steps

    # def train(self, generator_ref=None, compile_model=True):
    def train(self, compile_model=True):
        logger.info(f"Starting training for {self.cfg.train.epochs} epochs.")

        callbacks_list = []
        # Only add visualizer if we are NOT in batch mode (saves massive time and RAM)
        if not self.cfg.batch_mode:
            callbacks_list.append(
                TrainingVisualizer(self.val_ds, self.cfg,
                                   frequency=max(1, self.cfg.train.epochs // 5))
            )
        # Only attach the heavy Buffer callback if hallucination is actually enabled!
        if self.cfg.aug.prob_hallucination_max > 0.0:
            callbacks_list.append(
                BufferUpdateCallback(
                    self.manager, samples_per_epoch=128 if KAGGLE else 384
                )
            )

        # LOGIC SWITCH: Check both Architecture AND Training Mode
        is_spade_arch = (self.cfg.model.architecture in ['spade', 'vae'])
        is_gan_mode = self.cfg.train.gan_mode

        if is_gan_mode and is_spade_arch:
            logger.info(">>> Mode: SPADE GAN Training (Adversarial)")

            # Callbacking
            callbacks_list.append(
                # Custom Checkpoint for Generator
                GeneratorCheckpoint(
                    f"{self.cfg.model.name}_best.keras",
                    monitor='val_l1_loss',
                    save_best_only=True,
                    mode='min'
                )
            )

            # 1. Build Discriminator
            # Input shape: Image (1ch) + Condition (N+Mask ch)
            img_shape = (*self.cfg.data.padded_size, 1)
            cond_shape = (*self.cfg.data.padded_size, self.cfg.model.input_channels)

            # Note: DiscriminatorBuilder must be defined in Cell 8
            # --- FIX: Only build and compile the GAN wrappers if compiling! ---
            if compile_model:
                img_shape = (*self.cfg.data.padded_size, 1)
                cond_shape = (*self.cfg.data.padded_size, self.cfg.model.input_channels)

                # Build Discriminator
                d_model = DiscriminatorBuilder.build(img_shape, cond_shape)

                extra_metrics = []
                clean_metrics = []

                # Wrap in GAN Trainer
                gan_model = SPADEGANTrainer(self.model, d_model, self.cfg,
                                            extra_metrics=clean_metrics)

                gan_model.compile(
                    g_optimizer=tf.keras.optimizers.Adam(
                        learning_rate=self.cfg.train.learning_rate_g, beta_1=0.5
                    ),
                    d_optimizer=tf.keras.optimizers.Adam(
                        learning_rate=self.cfg.train.learning_rate_d, beta_1=0.5
                    ),
                    metrics=extra_metrics
                )
                # Replace self.model with the GAN wrapper so .fit() works
                self.model = gan_model

            # 5. Train
            # Train (self.model is now either the newly built gan_model,
            # or the pre-existing gan_model passed from HPMEngine)
            history = self.model.fit(
                self.train_ds,
                validation_data=self.val_ds,
                epochs=self.cfg.train.epochs,
                steps_per_epoch=self.train_steps,
                validation_steps=self.val_steps,
                callbacks=callbacks_list,
                verbose=2 ** int(self.cfg.batch_mode),
            )

            # Point self.model back to the trained generator for downstream inference
            # (Note: self.model is the SPADEGANTrainer wrapper here)
            self.model = self.model.generator
        else:
            # REGRESSION MODE
            # Works for both 'unet' AND 'spade' (if gan_mode=False)
            arch_name = self.cfg.model.architecture.upper()
            logger.info(f">>> Mode: Standard Regression Training ({arch_name} Generator)")

            # Callbacking
            callbacks_list.append(
                # Checkpoint: Save model with lowest validation loss (or L1 for GAN)
                tf.keras.callbacks.ModelCheckpoint(
                    f"{self.cfg.model.name}_best.keras",
                    save_best_only=True,
                    monitor='val_loss',
                    # If SPADE, we track l1_loss. If Regression, we track total loss
                    # monitor='val_l1_loss' if is_spade_arch else 'val_loss',
                    save_weights_only=False # we want to save full model
                ),
            )

            # --- FIX: Only compile if the flag is True! ---
            if compile_model:
                optimizer = tf.keras.optimizers.Adam(
                    learning_rate=self.cfg.train.learning_rate
                )
                # Select Loss
                if self.cfg.train.use_spatial_loss:
                    loss_fn = SpatiallyWeightedL1Loss(self.cfg)
                else:
                    # loss_fn = CompositeLoss(self.cfg)
                    # Wrap it in the MHP logic
                    loss_fn = RelaxedMHPLossWrapper(
                        base_loss_fn=CompositeLoss(self.cfg),
                        num_hypotheses=self.cfg.model.num_hypotheses,
                        epsilon=self.cfg.model.mhp_epsilon
                    )
                metrics = [
                    # 'mae', 'mse', SSIMMetric(), PSNRMetric(),
                    # GradientSharpnessMetric()
                    OracleMAE(), # OracleMAEMetric(),
                    OracleMSE(), # OracleMSEMetric(),
                    OracleSSIM(), # SSIMMetric(),
                    OraclePSNR(), # PSNRMetric(),
                    GradientSharpnessMetric()
                ]
                self.model.compile(optimizer=optimizer, loss=loss_fn,
                                   metrics=metrics)

            history = self.model.fit(
                self.train_ds,
                validation_data=self.val_ds,
                epochs=self.cfg.train.epochs,
                steps_per_epoch=self.train_steps,
                validation_steps=self.val_steps,
                callbacks=callbacks_list,
                verbose=2 ** int(self.cfg.batch_mode),
            )

        return history

# Cell 12: Metrics & Visualization
- **Target File:** src/evaluation/metrics.py
- **Content:**
    - Evaluator class
    - calculate_region_metrics (SSIM/PSNR on masks)
    - Basic ortho-slice plotting helpers

In [ ]:
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

In [ ]:
@tf.function
def get_oracle_prediction(y_true, y_pred):
    """
    Extracts the best hypothesis (Oracle) from M predictions using physical gathering.
    This strips M-channel metadata to prevent SSIM XLA errors.
    """
    # 1. Slice GT Image immediately
    gt_img = y_true[..., 0:1]

    # 2. Get Hypotheses Count
    # Using .shape[-1] handles static, tf.shape handles dynamic
    M = y_pred.shape[-1]
    if M is None: M = tf.shape(y_pred)[-1]

    if M == 1:
        return y_pred

    # 3. Find the index of the best hypothesis for each batch item
    # mae_per_hyp shape: (Batch, M)
    mae_per_hyp = tf.reduce_mean(tf.abs(gt_img - y_pred), axis=[1, 2])
    best_indices = tf.argmin(mae_per_hyp, axis=1) # (Batch,)

    # 4. PHYSICAL GATHER (The Fix)
    # Instead of one-hot multiplication, we gather the slices.
    # This forces the compiler to treat the result as a fresh 1-channel tensor.
    batch_size = tf.shape(y_pred)[0]
    batch_range = tf.range(batch_size, dtype=best_indices.dtype)

    # Create indices for gather_nd: [[batch_0, hyp_idx], [batch_1, hyp_idx], ...]
    indices = tf.stack([batch_range, best_indices], axis=1)

    # Transpose y_pred to (Batch, M, H, W) to gather across M
    y_trans = tf.transpose(y_pred, [0, 3, 1, 2])
    oracle_slices = tf.gather_nd(y_trans, indices) # result: (Batch, H, W)

    # Restore channel dimension
    return tf.expand_dims(oracle_slices, axis=-1) # result: (Batch, H, W, 1)

In [ ]:
class GradientSharpnessMetric(tf.keras.metrics.Metric):
    def __init__(self, name='grad_diff', **kwargs):
        super().__init__(name=name, **kwargs)
        self.diff_sum = self.add_weight(name='diff_sum', initializer='zeros')
        self.count = self.add_weight(name='count', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        gt_img = y_true[..., 0:1]
        oracle_pred = get_oracle_prediction(y_true, y_pred) # <--- MHP FIX
        dy_true, dx_true = tf.image.image_gradients(gt_img)
        dy_pred, dx_pred = tf.image.image_gradients(oracle_pred)
        grad_diff = tf.abs(dy_true - dy_pred) + tf.abs(dx_true - dx_pred)
        batch_mean = tf.reduce_mean(grad_diff, axis=(1, 2, 3))
        self.diff_sum.assign_add(tf.reduce_sum(batch_mean))
        self.count.assign_add(tf.cast(tf.shape(y_true)[0], tf.float32))

    def result(self): return self.diff_sum / self.count
    def reset_state(self): self.diff_sum.assign(0.0); self.count.assign(0.0)

In [ ]:
class OracleMAE(tf.keras.metrics.Metric):
    def __init__(self, name="oracle_mae", **kwargs):
        super().__init__(name=name, **kwargs)
        self.tracker = tf.keras.metrics.Mean()

    def update_state(self, y_true, y_pred, sample_weight=None):
        gt_img = y_true[..., 0:1]  # <--- CRITICAL SLICE
        oracle_pred = get_oracle_prediction(y_true, y_pred)

        # Calculate MAE
        batch_mae = tf.reduce_mean(tf.abs(gt_img - oracle_pred), axis=[1, 2, 3])
        self.tracker.update_state(batch_mae)

    def result(self): return self.tracker.result()
    def reset_states(self): self.tracker.reset_states()


class OracleMSE(tf.keras.metrics.Metric):
    def __init__(self, name="oracle_mse", **kwargs):
        super().__init__(name=name, **kwargs)
        self.tracker = tf.keras.metrics.Mean()

    def update_state(self, y_true, y_pred, sample_weight=None):
        gt_img = y_true[..., 0:1]  # <--- CRITICAL SLICE
        oracle_pred = get_oracle_prediction(y_true, y_pred)

        # Calculate MSE
        batch_mse = tf.reduce_mean(tf.square(gt_img - oracle_pred),
                                   axis=[1, 2, 3])
        self.tracker.update_state(batch_mse)

    def result(self): return self.tracker.result()
    def reset_states(self): self.tracker.reset_states()


class OraclePSNR(tf.keras.metrics.Metric):
    def __init__(self, name="oracle_psnr", max_val=1.0, **kwargs):
        super().__init__(name=name, **kwargs)
        self.max_val = max_val
        self.tracker = tf.keras.metrics.Mean()

    def update_state(self, y_true, y_pred, sample_weight=None):
        gt_img = y_true[..., 0:1]  # <--- CRITICAL SLICE
        oracle_pred = get_oracle_prediction(y_true, y_pred)

        # We tell the graph compiler that the channel dimension is exactly 1.
        # The None values represent the dynamic Batch, Height, and Width
        # oracle_pred.set_shape([None, None, None, 1])
        # gt_img.set_shape([None, None, None, 1])

        # Calculate PSNR
        psnr_values = tf.image.psnr(gt_img, oracle_pred, max_val=self.max_val)
        self.tracker.update_state(psnr_values)

    def result(self): return self.tracker.result()
    def reset_states(self): self.tracker.reset_states()


class OracleSSIM(tf.keras.metrics.Metric):
    def __init__(self, name="oracle_ssim", max_val=1.0, **kwargs):
        super().__init__(name=name, **kwargs)
        self.max_val = tf.cast(max_val, tf.float32)
        self.tracker = tf.keras.metrics.Mean()

    def update_state(self, y_true, y_pred, sample_weight=None):
        # 1. Cast everything to float32 (SSIM is very sensitive to this)
        gt_img = tf.cast(y_true[..., 0:1], tf.float32)
        oracle_pred = tf.cast(get_oracle_prediction(y_true, y_pred), tf.float32)

        # 2. THE DYNAMIC RESHAPE (No hardcoding)
        # We fetch the dynamic shape and force the channel to 1.
        # This acts as a "hard reset" for the graph compiler.
        dyn_shape = tf.shape(oracle_pred)
        gt_img = tf.reshape(gt_img, [dyn_shape[0], dyn_shape[1], dyn_shape[2], 1])
        oracle_pred = tf.reshape(oracle_pred, [dyn_shape[0], dyn_shape[1],
                                               dyn_shape[2], 1])

        # 3. Calculate SSIM
        ssim_values = tf.image.ssim(gt_img, oracle_pred, max_val=self.max_val)

        self.tracker.update_state(ssim_values)

    def result(self):
        return self.tracker.result()

    def reset_states(self):
        self.tracker.reset_states()

In [ ]:
class Evaluator:
    """
    Handles calculation of metrics and plotting of results.
    """

    @staticmethod
    def calculate_region_metrics(y_true: np.ndarray, y_pred: np.ndarray, mask: np.ndarray):
        """
        Calculates MAE, MSE, PSNR, SSIM for a specific region defined by 'mask'.
        """
        mask_bool = mask.astype(bool)
        if not np.any(mask_bool):
            return {'mae': 0.0, 'mse': 0.0, 'psnr': 0.0, 'ssim': 0.0}

        true_region = y_true[mask_bool]
        pred_region = y_pred[mask_bool]

        mae = np.mean(np.abs(true_region - pred_region))
        mse = np.mean(np.square(true_region - pred_region))

        # PSNR
        psnr_val = psnr(true_region, pred_region, data_range=1.0) if mse > 0 else 100.0

        # SSIM (Calculated on bounding box of the region to be valid)
        rows, cols = np.where(mask_bool)
        r_min, r_max = np.min(rows), np.max(rows)
        c_min, c_max = np.min(cols), np.max(cols)

        if (r_max - r_min < 7) or (c_max - c_min < 7):
            ssim_val = 0.0
        else:
            # Crop to bbox
            bbox_true = y_true[r_min:r_max, c_min:c_max]
            bbox_pred = y_pred[r_min:r_max, c_min:c_max]
            ssim_val = ssim(bbox_true, bbox_pred, data_range=1.0)

        return {'mae': mae, 'mse': mse, 'psnr': psnr_val, 'ssim': ssim_val}

    @staticmethod
    def plot_ortho_slices(volume: np.ndarray, title: str = "Orthogonal Views"):
        """Plots Center slices for Axial, Coronal, and Sagittal planes."""
        c_x, c_y, c_z = np.array(volume.shape) // 2

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        fig.suptitle(title, fontsize=16)

        # Axial (XY)
        axes[0].imshow(volume[c_x, :, :], cmap='bone')
        axes[0].set_title(f"Axial (Slice {c_x})")

        # Coronal (XZ) -> In numpy (Z, Y, X), this is (:, Y, :)
        axes[1].imshow(np.rot90(volume[:, c_y, :]), cmap='bone')
        axes[1].set_title(f"Coronal (Slice {c_y})")

        # Sagittal (YZ) -> (:, :, X)
        axes[2].imshow(np.rot90(volume[:, :, c_z]), cmap='bone')
        axes[2].set_title(f"Sagittal (Slice {c_z})")

        for ax in axes: ax.axis('off')
        plt.show()

    @staticmethod
    def visualize_restoration(gt, pred, mask=None, slice_idx=None):
        """Side-by-side comparison of GT, Pred, and Difference."""
        if slice_idx is None:
            slice_idx = gt.shape[0] // 2

        g_slice = gt[slice_idx]
        p_slice = pred[slice_idx]
        diff = np.abs(g_slice - p_slice)

        cols = 4 if mask is not None else 3
        fig, axes = plt.subplots(1, cols, figsize=(4*cols, 4))

        axes[0].imshow(g_slice, cmap='bone')
        axes[0].set_title("Ground Truth")

        axes[1].imshow(p_slice, cmap='bone')
        axes[1].set_title("Restoration")

        axes[2].imshow(diff, cmap='inferno', vmin=0, vmax=0.3)
        axes[2].set_title("Error Map")

        if mask is not None:
            # Overlay mask on GT
            axes[3].imshow(g_slice, cmap='bone')
            axes[3].contour(mask[slice_idx], levels=[0.5], colors='red')
            axes[3].set_title("Tumor Mask Overlay")

        for ax in axes: ax.axis('off')
        plt.show()

# Cell 13: The Unified Reconstructor
- **Target File:** src/inference/reconstructor.py
- **Content:**
    - VolumeReconstructor class
    - Inference logic: _predict_slice, autoregressive_restore, bidirectional_restore

In [ ]:
from collections import deque


class VolumeReconstructor:
    """
    Engine for applying the trained model to full 3D volumes.
    Supports Autoregressive and Bidirectional strategies.
    """
    def __init__(self, model, config: Config):
        self.model = model
        self.cfg = config

    def _predict_slice(self, input_stack_native, target_slice_native,
                       start_idx, target_idx, axis_size, direction,
                       mask_volume=None):

        void_mask = (
            mask_volume[target_idx] > 0
        ).astype(np.float32) if mask_volume is not None else None

        model_inputs, scale, pads = InputProcessor.prepare_model_input(
            input_stack_native, target_slice_native,
            start_idx, target_idx, axis_size, direction,
            void_mask_native=void_mask, config=self.cfg
        )

        hist_pad, scale, pads = GeometryOps.resize_and_pad(
            tf.convert_to_tensor(model_inputs['history_input']),
            self.cfg.data.padded_size, 'bicubic'
        )
        mask_pad, _, _ = GeometryOps.resize_and_pad(
            tf.convert_to_tensor(model_inputs['mask_input']),
            self.cfg.data.padded_size, 'nearest'
        )
        model_inputs['history_input'] = hist_pad
        model_inputs['mask_input'] = mask_pad

        x_batch = {k: tf.expand_dims(v, 0) for k, v in model_inputs.items()}

        # Returns shape (1, H_pad, W_pad, M)
        pred_padded_raw = self.model(x_batch, training=False)[0]

        native_h, native_w = target_slice_native.shape

        # Inverse resize ALL M hypotheses at once!
        # Returns shape (H_native, W_native, M)
        pred_restored = GeometryOps.inverse_resize_pad(
            pred_padded_raw, (native_h, native_w), scale, pads
        ).numpy()

        # --- DIAGNOSTIC ---
        if np.max(pred_restored) < 1e-6:
             logger.warning(f"⚠️ Reconstructor output is EMPTY! Input max: {np.max(input_stack_native):.4f}")
        # ------------------

        return pred_restored

    def autoregressive_restore(self,
                               volume: np.ndarray,
                               start_idx: int,
                               end_idx: int,
                               direction: str = 'forward',
                               mask_volume: np.ndarray = None) -> np.ndarray:
        """
        Restores slices. If mask_volume is provided, uses 'Teacher Forcing'
        to keep healthy tissue (outside mask) perfect, inpainting only inside mask.
        mask_volume: (Z, H, W) binary mask where 1=Tumor/Void, 0=Healthy.
        """
        axis_size = volume.shape[0]
        recon_vol = np.zeros_like(volume)
        N = self.cfg.data.neighborhood

        if direction == 'forward':
            iter_range = range(start_idx, end_idx)
            # Context: [t-N ... t-1]
            # Transpose to (H, W, N) for InputProcessor consistency
            initial_context = [volume[i] for i in range(start_idx - N, start_idx)]
            recon_vol[:start_idx] = volume[:start_idx]
        else:
            iter_range = range(end_idx - 1, start_idx - 1, -1)
            # Context: [t+N ... t+1] (Spatial High to Low)
            # For backward prediction at 't', we need t+1, t+2, t+3.
            # Generator flip logic: t+1 becomes "index 0", t+3 "index 2"?
            # Let's trust the queue order
            initial_context = [volume[i] for i in range(end_idx, end_idx + N)]
            # If we iterate backwards (decreasing index), the "immediate previous" is index+1.
            # Queue append logic handles the shift.
            # Important: Reverse initial context so popleft/append works linearly?
            # Let's keep it simple: List order [Closest ... Farthest] or [Farthest ... Closest]?
            # Standard: [t-3, t-2, t-1]. Append t. New: [t-2, t-1, t].
            # Backward: [t+3, t+2, t+1]. Append t. New: [t+2, t+1, t].
            # So we load [t+3, t+2, t+1] naturally
            initial_context.reverse() # [t+1, t+2, t+3] -> [t+3, t+2, t+1] order in RAM?
            # Actually, standard range gives [100, 101, 102].
            # We want buffer to end with 100 (t+1).
            # So input [102, 101, 100]
            recon_vol[end_idx:] = volume[end_idx:]

        buffer = deque(initial_context, maxlen=N)

        # Safety: Ensure iter_range is valid for the buffer size N
        valid_iter = []
        for i in iter_range:
            # For forward: need [i-N...i-1]. So i-N must be >= 0
            # For backward: need [i+1...i+N]. So i+N must be < axis_size
            if direction == 'forward' and (i - N) < 0: continue
            if direction == 'backward' and (i + N) >= axis_size: continue
            valid_iter.append(i)

        for i in tqdm(valid_iter, desc=f"{direction.title()} Pass", leave=False,
                      disable=self.cfg.batch_mode): # use valid_iter instead of iter_range
            # Stack from buffer -> (N, H, W)
            stack_n_h_w = np.stack(list(buffer), axis=0)

            # ----- Zero out the tumor in the context stack! -----
            if mask_volume is not None:
                ctx_indices =[i - N + k for k in range(N)] if direction == 'forward' else [i + N - k for k in range(N)]
                for buf_idx, z_idx in enumerate(ctx_indices):
                    if 0 <= z_idx < axis_size:
                        m = (mask_volume[z_idx] > 0).astype(np.float32)
                        stack_n_h_w[buf_idx] *= (1.0 - m)
            # -----------------------------------------------------

            # InputProcessor expects (H, W, N), we transpose
            stack_h_w_n = np.transpose(stack_n_h_w, (1, 2, 0))

            # Target Slice (Ground Truth Geometry)
            target_slice = volume[i]

            # Calculate true start index for positional encoding
            # Forward: Context is [i-N, ..., i-1]. Start is i-N.
            # Backward: Context is [i+N, ..., i+1]. Start is i+N.
            true_start_idx = (i - N) if direction == 'forward' else (i + N)

            # Predict returns (H, W, M)
            preds_all = self._predict_slice(
                stack_h_w_n, target_slice, true_start_idx, i, axis_size,
                direction, mask_volume
            )

            # FAST GREEDY SELECTION: Just take Hypothesis 0
            # (In MHP, the heads specialize, so sticking to Head 0 ensures consistent style)
            pred_slice = preds_all[..., 0] if self.cfg.model.num_hypotheses > 1 else preds_all[..., 0]

            # --- TEACHER FORCING LOGIC ---
            if mask_volume is not None:
                # Get tumor mask for current slice 'i'
                # Ensure mask is binary (0/1)
                m = (mask_volume[i] > 0).astype(np.float32)

                # Blend: Keep GT where mask is 0, Use Pred where mask is 1
                # Inpaint ONLY inside the tumor
                final_slice = (pred_slice * m) + (target_slice * (1.0 - m))
            else:
                # Pure Autoregression
                final_slice = pred_slice

            recon_vol[i] = final_slice
            buffer.append(final_slice)

        return recon_vol

    def beam_search_restore(self, volume: np.ndarray, start_idx: int, end_idx: int,
                            direction: str = 'forward', mask_volume: np.ndarray = None,
                            beam_width: int = 3) -> np.ndarray:
        axis_size = volume.shape[0]
        N = self.cfg.data.neighborhood
        M = self.cfg.model.num_hypotheses

        # Fallback to greedy if M=1
        if M == 1:
            return self.autoregressive_restore(volume, start_idx, end_idx,
                                               direction, mask_volume)

        if direction == 'forward':
            iter_range = range(start_idx, end_idx)
            initial_context = [volume[i] for i in range(start_idx - N, start_idx)]
        else:
            iter_range = range(end_idx - 1, start_idx - 1, -1)
            initial_context = [volume[i] for i in range(end_idx, end_idx + N)]
            initial_context.reverse()

        # A beam is: (cumulative_cost, buffer_list, generated_slices_list)
        beams =[(0.0, initial_context, [])]

        for i in tqdm(iter_range, desc=f"{direction.title()} Beam Search",
                      leave=False, disable=self.cfg.batch_mode):
            new_beams = []
            target_slice = volume[i]
            true_start_idx = (i - N) if direction == 'forward' else (i + N)

            # Next index to look ahead (for cost calculation)
            next_i = (i + 1) if direction == 'forward' else (i - 1)
            can_lookahead = (direction == 'forward' and next_i < end_idx) or \
                            (direction == 'backward' and next_i >= start_idx)

            # 1. Expand each active beam
            for cost, buffer, gen_slices in beams:
                stack_h_w_n = np.transpose(np.stack(buffer, axis=0), (1, 2, 0))

                # Get M hypotheses for current step
                preds_all = self._predict_slice(
                    stack_h_w_n, target_slice, true_start_idx, i, axis_size,
                    direction, mask_volume
                )

                # 2. Score each hypothesis
                for m in range(M):
                    hyp_slice = preds_all[..., m]

                    # Apply Teacher Forcing if mask exists
                    if mask_volume is not None:
                        mask_m = (mask_volume[i] > 0).astype(np.float32)
                        final_slice = (
                            hyp_slice * mask_m
                        ) + (
                            target_slice * (1.0 - mask_m)
                        )
                    else:
                        final_slice = hyp_slice

                    step_cost = 0.0

                    # --- THE RL LOOKAHEAD (Cost Calculation) ---
                    if can_lookahead:
                        temp_buffer = buffer[1:] + [final_slice]
                        temp_stack = np.transpose(np.stack(temp_buffer, axis=0),
                                                  (1, 2, 0))
                        next_target = volume[next_i]
                        next_start_idx = (next_i - N) if direction == 'forward' else (next_i + N)

                        # Peek at t+1
                        future_preds = self._predict_slice(
                            temp_stack, next_target, next_start_idx, next_i,
                            axis_size, direction, mask_volume
                        )
                        # Variance across the M heads for t+1
                        # High variance = the model is confused by our `final_slice` choice!
                        future_variance = np.var(future_preds, axis=-1)
                        step_cost = float(np.mean(future_variance))

                    new_cost = cost + step_cost
                    new_beams.append((new_cost, buffer[1:] + [final_slice],
                                      gen_slices + [final_slice]))

            # 3. Prune the beams
            new_beams.sort(key=lambda x: x[0])
            beams = new_beams[:beam_width]

        # Reconstruction is complete. The best trajectory is the one with the lowest cost.
        best_trajectory = beams[0][2]

        recon_vol = volume.copy()
        if direction == 'forward':
            recon_vol[start_idx:end_idx] = best_trajectory
        else:
            # For backward, the generated slices were appended in reverse order
            recon_vol[start_idx:end_idx] = list(reversed(best_trajectory))

        return recon_vol

    def bidirectional_restore(self, volume: np.ndarray,
                              mask: np.ndarray = None, masked_inference: bool = False,
                              use_beam_search: bool = True) -> np.ndarray:
        """
        masked_inference: If True, uses the mask to enforce ground truth outside the tumor.
        """
        N = self.cfg.data.neighborhood

        if mask is not None:
             has_tumor = np.any(np.isin(mask, [1, 2, 4]), axis=(1, 2))
             indices = np.where(has_tumor)[0]
             if len(indices) == 0: return volume

             # CRITICAL FIX: Clamp to ensure N context slices always exist
             start = max(N, indices[0] - 2)
             end = min(volume.shape[0] - N, indices[-1] + 3)

             # Safety check: if tumor is so huge/close to edges that start >= end
             if start >= end:
                 print("Warning: Tumor extends too far to edges for context window. Skipping.")
                 return volume
        else:
             start = N
             end = volume.shape[0] - N

        # Pass 'mask' to autoregressive_restore only if masked_inference is True
        inference_mask = mask if masked_inference else None

        # Choose the restoration engine
        restore_fn = self.beam_search_restore if use_beam_search else self.autoregressive_restore

        recon_fwd = restore_fn(volume, start, end, 'forward',
                               mask_volume=inference_mask)
        recon_bwd = restore_fn(volume, start, end, 'backward',
                               mask_volume=inference_mask)

        recon_merged = volume.copy()
        recon_merged[start:end] = (recon_fwd[start:end] + recon_bwd[start:end]) / 2.0

        return recon_merged

# Cell 14 (Unused): Execution Check
- **Target:** Notebook cell only (or tests/test_inference.py)
- **Content:**
    - Sanity checks for the Reconstructor on dummy data

# Cell 15: The Visualization Suite
- **Target File:** src/evaluation/visualizer.py
- **Content:**
    - VisualizationSuite class
    - All plotting logic: plot_augmentations, plot_training_history, plot_hallucination_buffer, analyze_best_worst, plot_autoregressive_performance, plot_bidirectional

In [ ]:
import os
import math

import numpy as np
import matplotlib.cm as cm

from PIL import Image, ImageDraw
from scipy.ndimage import binary_dilation


class VolumeDashboard:
    def __init__(self, volume_path, axis_size, axis, dataset_dir, downsample=2):
        self.volume_path = volume_path
        self.axis_size = axis_size
        self.axis = axis
        self.dataset_dir = dataset_dir

        # Grid parameters
        self.ds = downsample
        self.cell_sz = 160 // self.ds
        self.cols = 12
        self.rows = math.ceil(self.axis_size / self.cols)

        # Preallocate entire grid as a black RGB image
        self.grid_w = self.cols * self.cell_sz
        self.grid_h = self.rows * self.cell_sz
        self.image = Image.new("RGB", (self.grid_w, self.grid_h), (0, 0, 0))

        self.counts = {i: 0 for i in range(axis_size)}
        self.active_indices = set()

        # NEW: Track how many sequences have been stamped onto this brain
        self.z_counter = 0

    def update(self, slice_idx, img_arr, role, info, z_index, is_target=False,
               tumor_mask=None):
        if slice_idx < 0 or slice_idx >= self.axis_size: return

        self.counts[slice_idx] += 1
        # count = self.counts[slice_idx]
        self.active_indices.add(slice_idx)

        # Normalize image to 0-255 uint8 RGB
        img_ds = img_arr[::self.ds, ::self.ds]
        img_u8 = np.clip(img_ds * 255.0, 0, 255).astype(np.uint8)

        # Create base cell image
        cell_img = Image.fromarray(img_u8).convert("RGB")
        cell_draw = ImageDraw.Draw(cell_img)

        # 1. Bevels (Inner Border)
        bevel_color = (0, 255, 0) # Green (Healthy)
        if not is_target and info.get('hallucination_proc'):
            bevel_color = (128, 0, 128) # Purple
        elif info.get('glioma_applied'):
            bevel_color = (255, 0, 0) # Red

        cell_draw.rectangle([0, 0, self.cell_sz-1, self.cell_sz-1],
                            outline=bevel_color, width=max(1, 2//self.ds))

        # 2. Contours (Using scipy to find boundary)
        if tumor_mask is not None and np.any(tumor_mask):
            draw_contour = False
            c_color = (0, 0, 255) # Blue

            if is_target:
                # Target slices NEVER get white contours. Only red if it's a real tumor
                if info['mode'] == 'tumor':
                    draw_contour = True
                    c_color = (255, 0, 0) # Red
            else:
                # Input slices get contours if they were augmented
                if info['glioma_applied']:
                    draw_contour = True
                    c_color = (
                        0, 0, 255
                    ) if info['mode'] == 'clean' else (
                        255, 0, 0
                    )

            if draw_contour:
                mask_ds = tumor_mask[::self.ds, ::self.ds]
                mask_bool = mask_ds > 0.5
                boundary = mask_bool ^ binary_dilation(mask_bool)

                # Stamp contour using RGBA overlay
                overlay = np.zeros((self.cell_sz, self.cell_sz, 4),
                                   dtype=np.uint8)
                overlay[boundary] =[*c_color, 255]
                cell_img.paste(Image.fromarray(overlay), (0,0),
                               Image.fromarray(overlay))

        # 3. Dots for Blur/Noise (Bottom Left)
        dot_color = None
        is_blur = info.get('blur_applied', False) if not is_target else False
        is_noise = info.get('noise_applied', False) if not is_target else False

        if is_blur and is_noise: dot_color = (255, 255, 255) # White
        elif is_blur: dot_color = (0, 255, 255) # Cyan
        elif is_noise: dot_color = (255, 255, 0) # Yellow

        if dot_color:
            r = max(2, 4 // self.ds)
            x, y = int(self.cell_sz * 0.1), int(self.cell_sz * 0.9)
            cell_draw.ellipse([x-r, y-r, x+r, y+r], fill=dot_color)

        # 4. Text Overlays
        # Top-Left: Norm Pos
        norm_pos = f"{slice_idx / max(1, self.axis_size - 1):.3f}"
        cell_draw.text((2, 2), norm_pos, fill=(255, 255, 255))

        # Top-Right: Role + Direction (e.g., "T-1 ->")
        dir_arrow = "<" if info['direction'] == 'forward' else ">"
        role_str = f"{role} {dir_arrow}"
        role_color = (0, 255, 255) if is_target else (255, 165, 0)

        # Right align text
        try: tw = cell_draw.textlength(role_str)
        except AttributeError: tw = cell_draw.textsize(role_str)[0] # legacy compat
        cell_draw.text((self.cell_sz - tw - 2, 2), role_str, fill=role_color)

        # Bottom-Right: Z-Index (Sequence ID)
        z_str = f"z={z_index}"
        try: tw = cell_draw.textlength(z_str)
        except AttributeError: tw = cell_draw.textsize(z_str)[0]
        cell_draw.text((self.cell_sz - tw - 2, self.cell_sz - 12), z_str,
                       fill=(0, 255, 0))

        # 5. Paste directly into Master Grid
        col = slice_idx % self.cols
        row = slice_idx // self.cols
        self.image.paste(cell_img, (col * self.cell_sz, row * self.cell_sz))

    def render(self, input_only=False):
        # Draw placeholder text on untouched indices
        draw = ImageDraw.Draw(self.image)

        # Get global max count for normalization
        max_count = max(self.counts.values()) if self.counts else 1
        max_count = max(1, max_count)

        for i in range(self.axis_size):
            col = i % self.cols
            row = i // self.cols
            x0 = col * self.cell_sz
            y0 = row * self.cell_sz

            if i not in self.active_indices:
                # col = i % self.cols
                # row = i // self.cols
                draw.text((col * self.cell_sz + 2, row * self.cell_sz + 2),
                          f"{i / max(1, self.axis_size - 1):.3f}",
                          fill=(100, 100, 100))
            else:
                # --- INFERNO HEATMAP PATCH (per-slice bottom right corner of each) ---
                count = self.counts[i]
                norm_val = count / max_count

                # Get RGB from colormap
                rgba = cm.inferno(norm_val)
                r, g, b = int(rgba[0] * 255), int(rgba[1] * 255), int(rgba[2] * 255)

                rect_w, rect_h = 32, 14
                rx0 = x0 + self.cell_sz - rect_w - 2
                ry0 = y0 + self.cell_sz - rect_h - 2

                # High contrast text (black text on bright colors, white on dark)
                text_col = (0, 0, 0) if norm_val > 0.7 else (255, 255, 255)
                count_str = f"n={count}"

                try: tw = draw.textlength(count_str)
                except AttributeError: tw = draw.textsize(count_str)[0]

        # 2. --- GLOBAL BIRD'S-EYE HEATMAP IN THE LAST CELL ---
        # Find the absolute last cell of the entire dashboard grid
        last_cell_idx = (self.rows * self.cols) - 1
        lc_col = last_cell_idx % self.cols
        lc_row = last_cell_idx // self.cols

        lx0 = lc_col * self.cell_sz
        ly0 = lc_row * self.cell_sz

        # Draw a subtle border and title for the mini-heatmap
        draw.rectangle([lx0, ly0, lx0 + self.cell_sz - 1, ly0 + self.cell_sz - 1],
                       outline=(100, 100, 100))
        draw.text((lx0 + 4, ly0 + 2), "Global Sampling\nHeatmap",
                  fill=(255, 255, 255))

        # Calculate miniature block sizes
        # We map the mini-heatmap to the exact same column/row structure as the main dashboard
        y_offset = 28 # leave room for the title text
        mini_w = self.cell_sz / self.cols
        mini_h = (self.cell_sz - y_offset) / self.rows

        for i in range(self.axis_size):
            m_col = i % self.cols
            m_row = i // self.cols

            # Coordinates for this specific miniature block
            mx0 = lx0 + (m_col * mini_w)
            my0 = ly0 + y_offset + (m_row * mini_h)
            mx1 = mx0 + mini_w
            my1 = my0 + mini_h

            if i in self.active_indices:
                count = self.counts[i]
                norm_val = count / max_count
                rgba = cm.inferno(norm_val)
                r, g, b = int(rgba[0] * 255), int(rgba[1] * 255), int(rgba[2] * 255)
                draw.rectangle([mx0, my0, mx1, my1], fill=(r, g, b))
            else:
                # If not sampled, draw it dark gray
                draw.rectangle(
                    [mx0, my0, mx1, my1], fill=(30, 30, 30), outline=(50, 50, 50)
                )

        # Save to disk
        axis_names = ['axial', 'coronal', 'sagittal']
        filename = os.path.basename(self.volume_path)
        base_name = filename.replace('.nii.gz', '').replace('.nii', '')

        save_path = os.path.join(self.dataset_dir, axis_names[self.axis],
                                 f"{base_name}.jpg")
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        self.image.save(save_path)

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import nibabel as nib

In [ ]:
class VisualizationSuite:
    @staticmethod
    def _load_vol_direct(path):
        """Deprecated: Use VolumeLoader.load instead."""
        # This wrapper keeps API compatibility for internal calls if needed,
        # but calls the canonical loader
        vol = VolumeLoader.load(path)
        return vol.t1 if vol else None

    @staticmethod
    def _sample_real_brats_mask(target_volume: np.ndarray, brats_pool: list, target_z: int = None) -> np.ndarray:
        """
        Extracts a real 3D tumor shape from a random BraTS volume and injects it 
        into the target IXI volume. If target_z is provided, centers the tumor at that slice.
        """
        mask = np.zeros_like(target_volume, dtype=np.float32)

        # 1. Find valid brain tissue coordinates in the target IXI volume
        target_coords = np.argwhere(target_volume > 0.1)
        if len(target_coords) == 0:
            return mask

        # 2. Pick a random BraTS volume that actually has a segmentation mask
        valid_brats =[v for v in brats_pool if v.seg is not None]
        if not valid_brats:
            logger.warning("No BraTS volumes with masks found. Falling back to empty mask.")
            return mask

        source_brats = random.choice(valid_brats)

        # 3. Extract the real tumor (labels 1, 2, 4)
        tumor_mask = np.isin(source_brats.seg, [1, 2, 4]).astype(np.float32)
        tumor_coords = np.argwhere(tumor_mask > 0)

        if len(tumor_coords) == 0:
            return mask

        # 4. Crop the real tumor to its 3D bounding box
        z_min, y_min, x_min = tumor_coords.min(axis=0)
        z_max, y_max, x_max = tumor_coords.max(axis=0) + 1
        tumor_crop = tumor_mask[z_min:z_max, y_min:y_max, x_min:x_max]

        t_z, t_y, t_x = tumor_crop.shape

        # 5. Inject into target volume
        # To avoid unnatural centering, we force the tumor into the left or right hemisphere.
        # Assuming shape is (Z, Y, X), index 2 is the Sagittal (Left/Right) axis.
        mid_x = target_volume.shape[2] // 2

        # Pick a hemisphere and filter valid coordinates (adding a 10-pixel margin from the exact center)
        if random.random() < 0.5:
            hemisphere_coords =[c for c in target_coords if c[2] < mid_x - 10]
        else:
            hemisphere_coords = [c for c in target_coords if c[2] > mid_x + 10]

        # Fallback to all coords if the tumor is too big for the strict hemisphere split
        if len(hemisphere_coords) == 0:
            hemisphere_coords = target_coords

        # Try up to 50 times to find a center point where the tumor fits inside the bounds
        for _ in range(50):
            # cz, cy, cx = target_coords[random.randint(0, len(target_coords) - 1)]
            cz, cy, cx = hemisphere_coords[random.randint(0, len(hemisphere_coords) - 1)]

            if target_z is not None:
                cz = target_z  # force Z-center

            # Calculate injection bounds
            sz = cz - t_z // 2
            sy = cy - t_y // 2
            sx = cx - t_x // 2

            ez = sz + t_z
            ey = sy + t_y
            ex = sx + t_x

            # Check bounds
            if (sz >= 0 and sy >= 0 and sx >= 0 and 
                ez < target_volume.shape[0] and 
                ey < target_volume.shape[1] and 
                ex < target_volume.shape[2]):

                # Inject the real shape
                mask[sz:ez, sy:ey, sx:ex] = tumor_crop
                break

        # 6. Safety clip: Ensure the tumor doesn't spill into the background/air
        brain_mask = (target_volume > 0.01).astype(np.float32)
        mask = mask * brain_mask

        return mask

    @staticmethod
    def _generate_synthetic_mask(volume: np.ndarray, max_radius=20) -> np.ndarray:
        """Creates a realistic 3D Gaussian blob (simulated glioma) strictly inside the brain mass."""
        mask = np.zeros_like(volume, dtype=np.float32)

        # 1. Find valid brain tissue coordinates to place the tumor
        # Use > 0.1 to avoid placing the center on the extreme edges/skull
        coords = np.argwhere(volume > 0.1)
        if len(coords) == 0:
            return mask

        # 2. Pick a random center inside the brain
        center_idx = random.randint(0, len(coords) - 1)
        cz, cy, cx = coords[center_idx]

        # 3. Randomize elliptical shape (Gaussian covariance proxies)
        rz = random.uniform(max_radius * 0.4, max_radius * 0.8)
        ry = random.uniform(max_radius * 0.6, max_radius * 1.0)
        rx = random.uniform(max_radius * 0.6, max_radius * 1.0)

        # 4. Generate 3D grid
        z, y, x = np.ogrid[:volume.shape[0], :volume.shape[1], :volume.shape[2]]

        # 5. Gaussian falloff equation
        dist = (
            ((z - cz) ** 2 / rz ** 2) +
            ((y - cy) ** 2 / ry ** 2) +
            ((x - cx) ** 2 / rx ** 2)
        )
        blob = np.exp(-dist)

        # 6. Threshold to create a solid mask (simulating an enhancing core)
        mask[blob > 0.3] = 1.0 # TODO: sum and clip

        # 7. Safety clip: Ensure the tumor doesn't spill into the background/air
        brain_mask = (volume > 0.01).astype(np.float32)
        mask = mask * brain_mask

        return mask

    @staticmethod
    def plot_augmentations(dataset, num_groups=2, batch_mode=False):
        """
        Point 1 & 6: Hunts for specific augmentation types (Glioma, Blur, Noise).
        """
        print("\n--- [1] Data Augmentation Visualization (Hunting Mode) ---")

        samples = {
            "glioma_only": [],
            "blur_only": [],
            "noise_only": [],
            "blur_and_noise": []
        }

        criteria = {
            "glioma_only": lambda s: s["glioma_applied"] and not s["blur_applied"] and not s["noise_applied"],
            "blur_only": lambda s: not s["glioma_applied"] and s["blur_applied"] and not s["noise_applied"],
            "noise_only": lambda s: not s["glioma_applied"] and not s["blur_applied"] and s["noise_applied"],
            "blur_and_noise": lambda s: not s["glioma_applied"] and s["blur_applied"] and s["noise_applied"],
        }

        print("Searching dataset for augmentation examples...")
        count = 0
        stop = False
        counts = Counter()

        # Limit search to avoid hanging forever
        for x_batch, y_batch, info_batch in tqdm(dataset.take(1000), total=1000,
                                                 disable=batch_mode):
            if stop: break

            # Robust unbatching for info dict
            i_batch = [
                dict(zip(info_batch, map(lambda a: a.numpy(), v)))
                for v in zip(*info_batch.values())
            ]

            batch_size = y_batch.shape[0]
            for b in range(batch_size):
                # Extract dictionary of tensors for this specific batch element
                x = {k: v[b] for k, v in x_batch.items()}
                y = y_batch[b]
                info = i_batch[b]
                # Only cast the specific boolean augmentation flags to integers
                aug_flags =['glioma_applied', 'blur_applied', 'noise_applied']
                counts.update({k: int(info[k]) for k in aug_flags if k in info})
                count += 1

                for key, func in criteria.items():
                    if len(samples[key]) < num_groups and func(info):
                        samples[key].append({'x': x, 'y': y})

                if all(len(v) >= num_groups for v in samples.values()):
                    stop = True
                    break

        # --- Plotting ---
        for k in range(num_groups):
            print(f"\n--- Augmentation Group {k + 1} ---")

            # Determine grid size based on first available sample
            first_sample = next((v[0] for v in samples.values() if v), None)
            first_sample = next((v[0] for v in samples.values() if v), None)
            if not first_sample: continue

            # Access 'history_input' from the dictionary
            num_inputs = first_sample['x']['history_input'].shape[-1]
            # num_inputs = first_sample['x'].shape[-1]
            num_rows = len(samples)
            fig, axes = plt.subplots(num_rows, num_inputs + 1, figsize=(15, 3 * num_rows))
            if num_rows == 1: axes = [axes]

            for i, (key, sample_list) in enumerate(samples.items()):
                if k >= len(sample_list):
                    # Placeholder if missing
                    ax = axes[i, 0] if num_rows > 1 else axes[0]
                    ax.text(0.5, 0.5, f"Missing: {key}", ha='center', color='red')
                    for j in range(num_inputs + 1):
                        (axes[i, j] if num_rows > 1 else axes[j]).axis('off')
                    continue

                s = sample_list[k]
                x_data, y_data = s['x'], s['y']

                # Row Title
                ax_first = axes[i, 0] if num_rows > 1 else axes[0]
                ax_first.set_ylabel(key.replace('_', ' ').title(), fontsize=12,
                                    weight='bold')

                # Plot Inputs
                for j in range(num_inputs):
                    ax = axes[i, j] if num_rows > 1 else axes[j]
                    # ax.imshow(x_data[:, :, j], cmap='bone', vmin=0.0, vmax=1.0)
                    ax.imshow(x_data['history_input'][:, :, j], cmap='bone',
                              vmin=0.0, vmax=1.0)
                    # if i == 0: ax.set_title(f"Input {j} (t - {num_inputs - j})")
                    if i == 0: ax.set_title(f"Input {j}")
                    ax.set_xticks([]); ax.set_yticks([])

                # Plot Target
                ax = axes[i, num_inputs] if num_rows > 1 else axes[num_inputs]
                ax.imshow(y_data[:, :, 0], cmap='bone', vmin=0.0, vmax=1.0)
                # if i == 0: ax.set_title("Target (t)")
                if i == 0: ax.set_title("Target")
                ax.axis('off')

            plt.tight_layout()
            plt.show()

    @staticmethod
    def plot_training_history(history):
        print("\n--- [3] Training History ---")
        metrics = history.history
        # Find any valid metric to determine epoch count
        any_key = next(iter(metrics.keys()))
        epochs = range(1, len(metrics[any_key]) + 1)

        # Identify base metrics (exclude 'val_' prefix)
        base_metrics = [k for k in metrics.keys() if not k.startswith('val_')]

        # Determine grid
        n_metrics = len(base_metrics)
        cols = min(n_metrics, 3)
        rows = (n_metrics + cols - 1) // cols

        fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
        if n_metrics == 1: axes = [axes]
        axes = np.array(axes).flatten()

        for i, metric in enumerate(base_metrics):
            ax = axes[i]

            # Train curve
            ax.plot(epochs, metrics[metric], 'b-o', label=f'Train {metric}')

            # Val curve
            val_key = f'val_{metric}'
            if val_key in metrics:
                ax.plot(epochs, metrics[val_key], 'r--s', label=f'Valid {metric}')

            ax.set_title(metric.upper())
            ax.set_xlabel("Epochs")
            ax.legend()
            ax.grid(True, alpha=0.3)

        # Hide unused subplots
        for j in range(i + 1, len(axes)):
            axes[j].axis('off')

        plt.tight_layout()
        plt.show()

    @staticmethod
    def plot_hallucination_buffer(manager, num_samples=3):
        print("\n--- [4] Hallucination Buffer ---")
        buffer = manager.hallucination_buffer
        if not buffer:
            print("Buffer Empty.")
            return

        keys = random.sample(list(buffer.keys()), min(len(buffer), num_samples))
        fig, axes = plt.subplots(len(keys), 3, figsize=(12, 4 * len(keys)))
        if len(keys) == 1: axes = [axes]

        for i, key in enumerate(keys):
            path, axis, idx = key
            pred = buffer[key]

            # Try to load GT
            vol = VisualizationSuite._load_vol_direct(path)

            # Handle index logic:
            # If loaded directly, index is absolute. Buffer usually stores absolute.
            # But check bounds
            if vol is not None and idx < vol.shape[0]:
                gt = vol[idx]
            else:
                gt = np.zeros_like(pred)

            axes[i][0].imshow(gt, cmap='bone'); axes[i][0].set_title("Ground Truth")
            axes[i][1].imshow(pred, cmap='bone'); axes[i][1].set_title("Prediction")
            axes[i][2].imshow(np.abs(gt-pred), cmap='inferno'); axes[i][2].set_title("Error")
            for ax in axes[i]: ax.axis('off')
        plt.show()

    @staticmethod
    def analyze_best_worst(model, val_dataset, num_samples=3):
        print("\n--- [5] Best & Worst Predictions ---")
        best_samples = []
        worst_samples = []

        # Scan dataset
        # Limit to 50 batches for speed
        # --- Support both tf.data.Dataset and Keras Sequence ---
        if hasattr(val_dataset, 'take'):
            # tf.data.Dataset
            iterator = val_dataset.take(50)
        else:
            # Keras Sequence: slice the first 50 batches
            max_batches = min(50, len(val_dataset))
            iterator = [val_dataset[i] for i in range(max_batches)]
        # for x_batch, y_batch in val_dataset.take(50):
        for x_batch, y_batch in iterator:
            preds = model(x_batch, training=False)
            inputs = x_batch.numpy() if hasattr(x_batch, 'numpy') else x_batch
            inputs = x_batch['history_input'].numpy(
            ) if hasattr(x_batch['history_input'], 'numpy') else x_batch['history_input']
            targets = y_batch.numpy() if hasattr(y_batch, 'numpy') else y_batch

            # Calculate MSE per sample in batch
            batch_mse = np.mean(np.square(targets[..., 0] - preds[..., 0]),
                                axis=(1, 2))

            for i in range(targets.shape[0]):
                sample = {
                    'mse': batch_mse[i],
                    'input': inputs[i],
                    'gt': targets[i, ..., 0],
                    'pred': preds[i, ..., 0]
                }

                # Update Best
                best_samples.append(sample)
                best_samples.sort(key=lambda s: s['mse'])
                if len(best_samples) > num_samples: best_samples.pop()

                # Update Worst
                worst_samples.append(sample)
                worst_samples.sort(key=lambda s: s['mse'], reverse=True)
                if len(worst_samples) > num_samples: worst_samples.pop()

        # Sort final lists
        worst_samples.sort(key=lambda s: s['mse'])

        # --- Helper Plotter ---
        def visualize_group(samples, title):
            if not samples: return

            n_vis = len(samples)

            # Determine how many columns needed:
            # Inputs (N) + GT + Pred + Diff(GT) + Diff(Input)
            # Input shape: (H, W, C).
            # If mask attached, C = N+1.
            # We want to show first N channels as slices

            # Assuming standard config where C=3 or 4
            num_input_channels = samples[0]['input'].shape[-1]
            # Heuristic: If mask channel exists, it's the last one.
            # We only show image slices.
            # Let's show ALL input channels for completeness, or just N.
            # Let's show up to 3 input slices to keep plot sane
            num_input_show = min(num_input_channels, 3)

            cols = num_input_show + 4

            fig, axes = plt.subplots(n_vis, cols, figsize=(3 * cols, 3.75 * n_vis))
            fig.suptitle(title, fontsize=16, y=0.98)
            if n_vis == 1: axes = [axes]

            for i, s in enumerate(samples):
                ax_row = axes[i] if n_vis > 1 else axes

                # 1. Inputs
                for j in range(num_input_show):
                    ax = ax_row[j]
                    ax.imshow(s['input'][..., j], cmap='bone')
                    ax.set_title(f"Input {j} (t - {num_input_channels - j})")
                    ax.axis('off')

                # 2. GT
                ax = ax_row[num_input_show]
                ax.imshow(s['gt'], cmap='bone')
                ax.set_title("Target (t)")
                ax.axis('off')

                # 3. Pred
                ax = ax_row[num_input_show + 1]
                ax.imshow(s['pred'], cmap='bone')
                ax.set_title(f"Pred (MSE: {s['mse']:.5f})")
                ax.axis('off')

                # 4. Diff vs GT
                ax = ax_row[num_input_show + 2]
                diff_gt = np.abs(s['gt'] - s['pred'])
                ax.imshow(diff_gt, cmap='inferno', vmin=0, vmax=0.3)
                ax.set_title("Error |Pred - Target|")
                ax.axis('off')

                # 5. Diff vs Last Input (Temporal Change)
                # Last input slice is at index: num_input_channels - 1 (or -2 if mask)
                # Let's assume input image slice is at index 2 (if N=3)
                idx_last_input = 2 if num_input_channels >= 3 else 0
                last_input = s['input'][..., idx_last_input]

                ax = ax_row[num_input_show + 3]
                diff_in = np.abs(s['pred'] - last_input)
                ax.imshow(diff_in, cmap='inferno', vmin=0, vmax=0.3)
                ax.set_title(f"Diff |Pred - Input {j}|")
                ax.axis('off')

            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.show()

        if best_samples: visualize_group(best_samples, "Best Predictions (Lowest Error)")
        if worst_samples: visualize_group(worst_samples, "Worst Predictions (Highest Error)")

    @staticmethod
    def plot_autoregressive_performance(reconstructor, file_info, manager, masked_inference=False):
        # filename = os.path.basename(volume_path)
        filename = os.path.basename(file_info['t1'])
        print(f"\n--- [7] Autoregressive Analysis: {filename} ---")

        # 1. Load
        # vol = VisualizationSuite._load_vol_direct(volume_path)
        vol_obj = VolumeLoader.load(file_info['t1'], file_info['seg'])
        # if vol is None: return
        if vol_obj is None: return

        # Get volume and mask
        vol = vol_obj.t1
        mask_vol = vol_obj.seg
        if mask_vol is None and masked_inference:
            print(f"WARNING: masked inference is impossible, NO MASK!")
            masked_inference = False
        if not masked_inference:
            # Reset the mask if we want pure autoregressive
            mask_vol = None

        # 2. Ranges
        # Load Mask if provided (for BraTS)
        # If IXI, mask_vol is None, so we just run pure AR.
        # But user requested "Masked Autoregressive" visualization.
        # For IXI (Healthy), we don't have a tumor mask to hide behind.
        # So "Masked AR" only makes sense for BraTS or if we inject a fake mask
        center = vol.shape[0] // 2
        span = 40
        fwd_end = min(vol.shape[0], center + span)
        bwd_start = max(0, center - span)

        if fwd_end - bwd_start < 20: return

        # 3. Reconstruct
        if mask_vol is not None:
            print("Running Forward Masked Inference (Teacher Forcing)...")
        else:
            print("Running Forward Pure Autoregression Inference...")
        recon_fwd = reconstructor.autoregressive_restore(vol, center, fwd_end,
                                                         'forward',
                                                         mask_volume=mask_vol)
        if mask_vol is not None:
            print("Running Backward Masked Inference (Teacher Forcing)...")
        else:
            print("Running Backward Pure Autoregression Inference...")
        recon_bwd = reconstructor.autoregressive_restore(vol, bwd_start, center,
                                                         'backward',
                                                         mask_volume=mask_vol)

        # 4. Error Curves
        fwd_mse = np.mean(np.square(vol[center:fwd_end] - recon_fwd[center:fwd_end]), axis=(1, 2))
        bwd_mse = np.mean(np.square(vol[bwd_start:center] - recon_bwd[bwd_start:center]), axis=(1, 2))

        plt.figure(figsize=(14, 6))
        plt.plot(np.arange(bwd_start, center), bwd_mse, 'g-o', label='Backward (<-)')
        plt.plot(np.arange(center, fwd_end), fwd_mse, 'r-o', label='Forward (->)')
        plt.axvline(x=center, color='b', linestyle='--', label='Start (Center Slice)')

        if mask_vol is not None:
            plt.title(f"Bidirectional (Masked) Error Accumulation: {filename}")
        else:
            plt.title(f"Bidirectional Autoregressive Error Accumulation: {filename}")

        plt.xlabel("Slice Index")
        plt.ylabel("Mean Squared Error")
        plt.legend(); plt.grid(True, alpha=0.3); plt.show()

        # 5. Detailed Visuals (Input Stack + Mask + Pred)
        # We show 3 points: Backward Tail, Center, Forward Tail
        idx_show = [
            # bwd_start + 5,
            center - (center - bwd_start) // 2,
            center - 3,
            center - 1,
            center,
            center + 1,
            center + 3,
            center + (fwd_end - center) // 2,
            # fwd_end - 5
        ]
        # labels = ["Backward Tail", "Center (Anchor)", "Forward Tail"]

        N = reconstructor.cfg.data.neighborhood
        cols = N + 4 # inputs(N) + Mask + GT + Pred + Diff
        rows = len(idx_show)

        fig, axes = plt.subplots(rows, cols, figsize=(2.5 * cols, 4.2 * rows)) # increase height
        if mask_vol is not None:
            fig.suptitle(f"Reconstruction Details (Masked): {filename}", y=0.96,
                         fontsize=16)
        else:
            fig.suptitle(f"Reconstruction Details: {filename}", y=0.96,
                         fontsize=16)

        for i, idx in enumerate(idx_show):
            idx = int(max(0, min(idx, vol.shape[0] - 1)))

            # Select Source
            if idx < center:
                img_r = recon_bwd[idx]; name="Backward"
                context_vol = recon_bwd
                # Backward Model Input Order: [t+3, t+2, t+1]
                ctx_indices = [idx + N - k for k in range(N)] # [idx+3, idx+2, idx+1]
            elif idx > center:
                img_r = recon_fwd[idx]; name="Forward"
                context_vol = recon_fwd
                # Forward Model Input Order: [t-3, t-2, t-1]
                ctx_indices = [idx - N + k for k in range(N)] # [idx-3, idx-2, idx-1]
            else:
                img_r = vol[idx]; name="Anchor"
                context_vol = vol
                ctx_indices = [idx - N + k for k in range(N)]

            # 1. Plot Context (Model Input Order)
            for j in range(N):
                c_idx = ctx_indices[j]
                ax = axes[i][j]
                if 0 <= c_idx < vol.shape[0]:
                    ax.imshow(context_vol[c_idx], cmap='bone')
                    ax.set_title(f"In[{j}]\nSlice {c_idx}")
                else:
                    ax.set_title("OOB")
                ax.axis('off')

            # 2. Mask
            ax = axes[i][N]
            mask = (vol[idx] > 0.01).astype(np.float32)
            ax.imshow(mask, cmap='gray')
            ax.set_title("Mask")
            ax.axis('off')

            # 3. GT
            ax = axes[i][N+1]
            ax.imshow(vol[idx], cmap='bone')
            ax.set_title(f"GT {idx}")
            ax.axis('off')

            # 4. Pred
            ax = axes[i][N+2]
            ax.imshow(img_r, cmap='bone')
            ax.set_title("Pred")
            ax.axis('off')

            # 5. Diff
            ax = axes[i][N+3]
            mse = np.mean((vol[idx]-img_r)**2)
            ax.imshow(np.abs(vol[idx]-img_r), cmap='inferno', vmin=0, vmax=0.3)
            ax.set_title(f"Err {mse:.4f}")
            ax.axis('off')

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()

    @staticmethod
    def plot_bidirectional(reconstructor, brats_file, manager, num_vis_slices=5,
                           masked_inference=False):
        """
        Visualizes bidirectional autoregressive inference on a BraTS tumor volume.
        Matches the original pipeline's visualization style.
        """
        print(f"\n--- [9] Bidirectional Inpainting: {brats_file['id']} ---")

        # 1. Load Data using Universal Loader (Canonical Orientation & Crop)
        vol_obj = VolumeLoader.load(brats_file['t1'], brats_file['seg'])
        if vol_obj is None:
            print("Failed to load volume!")
            return

        t1 = vol_obj.t1
        seg = vol_obj.seg

        if masked_inference and seg is None:
            print("WARNING: No real mask found. Generating synthetic spherical mask for IXI evaluation.")
            seg = VisualizationSuite._generate_synthetic_mask(t1, radius=16)

        # 2. Determine Range based on Tumor
        # Assuming Axis 0 is Axial (Slice) after VolumeLoader processing
        start, end = 0, t1.shape[0]

        if seg is not None:
            # Find slices containing tumor labels (1, 2, 4)
            # Sum spatial dims (1, 2) to get tumor pixels per slice
            tumor_presence = np.any(np.isin(seg, [1, 2, 4]), axis=(1, 2))
            indices = np.where(tumor_presence)[0]

            if len(indices) == 0:
                print("No tumor found in segmentation mask.")
                return

            # Add padding context
            pad = 2
            start = max(0, indices[0] - pad)
            end = min(t1.shape[0], indices[-1] + pad + 1)
        else:
            print("No segmentation mask available. Using center volume.")
            center = t1.shape[0] // 2
            start = center - num_vis_slices * 3
            end = center + num_vis_slices * 3

        print(f"Tumor Range found: Slice {start} to {end} (Total {end-start} slices)")

        # Run masked on demand, otherwise as possible
        mask = seg if masked_inference else None

        # 3. Run Forward Pass (Bottom -> Top)
        # Context comes from [start-N ... start]
        recon_fwd = reconstructor.autoregressive_restore(t1, start, end, 'forward',
                                                         mask_volume=mask)

        # 4. Run Backward Pass (Top -> Bottom)
        # Context comes from [end ... end+N]
        recon_bwd = reconstructor.autoregressive_restore(t1, start, end, 'backward',
                                                         mask_volume=mask)

        # 5. Visualization Selection
        # Select N indices evenly spaced within the range
        if end - start < num_vis_slices:
            vis_indices = range(start, end)
        else:
            vis_indices = np.linspace(start, end-1, num=num_vis_slices, dtype=int)

        # Setup Plot
        n_rows = len(vis_indices)
        fig, axes = plt.subplots(n_rows, 5, figsize=(20, 4 * n_rows))
        if not masked_inference:
            fig.suptitle(f"Bidirectional Autoregressive Inpainting: {brats_file['id']}",
                         fontsize=16, y=0.98)
        else:
            fig.suptitle(f"Bidirectional (Masked) Inpainting: {brats_file['id']}",
                         fontsize=16, y=0.98)

        cols = [
            "Ground Truth",
            "Forward (Bot->Top)",
            "Backward (Top->Bot)",
            "Fwd Diff",
            "Bwd Diff"
        ]

        # Handle single row case
        if n_rows == 1:
            axes = [axes]
            header_ax = axes[0]
        else:
            header_ax = axes[0]

        # Set Column Titles
        for ax, col in zip(header_ax, cols):
            ax.set_title(col, fontsize=12, weight='bold')

        for i, idx in enumerate(vis_indices):
            row_axes = axes[i]

            # Data
            gt_slice = t1[idx]
            seg_slice = seg[idx] if seg is not None else np.zeros_like(gt_slice)
            fwd_slice = recon_fwd[idx]
            bwd_slice = recon_bwd[idx]

            # 1. Ground Truth + Red Tumor Outline
            row_axes[0].imshow(gt_slice, cmap='bone')
            if np.sum(seg_slice) > 0:
                # Overlay tumor contour
                row_axes[0].contour(seg_slice, levels=[0.5], colors='red',
                                    linewidths=0.5)
            row_axes[0].set_ylabel(f"Slice {idx}", fontsize=12)

            # 2. Forward Prediction
            row_axes[1].imshow(fwd_slice, cmap='bone')

            # 3. Backward Prediction
            row_axes[2].imshow(bwd_slice, cmap='bone')

            # 4. Forward Difference (Heatmap)
            diff_fwd = np.abs(gt_slice - fwd_slice)
            row_axes[3].imshow(diff_fwd, cmap='inferno', vmin=0, vmax=0.3)

            # 5. Backward Difference (Heatmap)
            diff_bwd = np.abs(gt_slice - bwd_slice)
            row_axes[4].imshow(diff_bwd, cmap='inferno', vmin=0, vmax=0.3)

            # Remove ticks
            for ax in row_axes:
                ax.set_xticks([])
                ax.set_yticks([])

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()

    @staticmethod
    def plot_sampling_statistics(manager):
        print("\n--- [Stats] Data Sampling Distribution ---")
        stats = manager.stats.hits
        if not stats:
            print("No statistics recorded.")
            return

        # Organize data
        # Structure: data[dataset][axis] = {slice_idx: count}
        # Aggregate per volume for global check

        # 1. Volume Hits (Histogram)
        vol_counts = Counter()
        for (d, v, a, s, _), count in stats.items():
            vol_counts[f"{d}_{v}"] += count

        # Plot Top 50 volumes
        # FIX: Ensure we don't request more samples than actually exist
        total_hits = sum(vol_counts.values())
        if total_hits == 0:
            print("No sampling statistics to plot.")
            return

        k_samples = min(50, total_hits)

        # Plot Random 50 Sampled Volumes (Weighted by frequency)
        top_vols = random.sample(tuple(vol_counts.keys()),
                                 counts=tuple(vol_counts.values()), k=k_samples)

        labels, values = zip(*Counter(top_vols).items())
        # top_vols = vol_counts.most_common(50)
        # top_vols = random.sample(tuple(vol_counts.keys()),
        #                          counts=tuple(vol_counts.values()), k=50)
        # labels, values = zip(*top_vols)
        # labels, values = zip(*Counter(top_vols).items())

        plt.figure(figsize=(15, 4))
        plt.bar(labels, values)
        plt.xticks(rotation=90, fontsize=8)
        plt.title("50 Random Sampled Volumes")
        plt.show()

        # 2. Slice Distribution per Dataset (The "Fence" check)
        # We aggregate all volumes together to see global slice preference
        slice_counts = {'ixi': {0: Counter(), 1: Counter(), 2: Counter()},
                        'brats': {0: Counter(), 1: Counter(), 2: Counter()}}

        for (d, v, a, s, _), count in stats.items():
            slice_counts[d][a][s] += count

        # Plot
        for dataset in ['ixi', 'brats']:
            fig, axes = plt.subplots(3, 1, figsize=(15, 9))
            fig.suptitle(f"{dataset.upper()} Slice Distribution (Aggregated)",
                         fontsize=16)

            axes_names = ['Axial (0)', 'Coronal (1)', 'Sagittal (2)']
            for axis in [0, 1, 2]:
                data = slice_counts[dataset][axis]
                if not data:
                    axes[axis].text(0.5, 0.5, "No Data", ha='center')
                    axes[axis].set_title(axes_names[axis])
                    continue

                # Fill curve
                max_slice = max(data.keys())
                x = range(max_slice + 1)
                y = [data[i] for i in x]

                axes[axis].bar(x, y, width=1.0)
                axes[axis].set_title(axes_names[axis])
                axes[axis].set_xlabel("Slice Index")
                axes[axis].set_ylabel("Hits")

            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.show()

        # 3. Direction Balance
        dir_counts = {'ixi': Counter(), 'brats': Counter()}
        for (d, v, a, s, direction), count in stats.items():
            dir_counts[d][direction] += count

        fig, axes = plt.subplots(1, 2, figsize=(15, 7))
        fig.suptitle("Direction Balance (Forward vs Backward)", fontsize=14)

        for i, ds in enumerate(['ixi', 'brats']):
            counts = dir_counts[ds]
            if counts:
                axes[i].pie(counts.values(), labels=counts.keys(),
                            colors=plt.cm.Accent.colors, autopct='%1.1f%%')  # Pastel1, Set1, Dark2
                axes[i].set_title(ds.upper())
            else:
                axes[i].text(0.5, 0.5, "No Data")

        plt.show()

# Cell 16: The Main Execution Block (`main.py`)
- **Target File:** main.py
- **Content:**
    - run_pipeline() function
    - Orchestration of Config -> Data -> Model -> Trainer -> Visualization

In [ ]:
# %rm -rfv *.json databases

In [ ]:
import pandas as pd

from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr


def get_ablation_configs(base_cfg):
    """Generates the 4 specific states for the CVPR ablation study."""
    import copy
    configs = {}

    # 1. Baseline U-Net (No SPADE, No Buffer, No PE)
    cfg_base = copy.deepcopy(base_cfg)
    cfg_base.model.architecture = 'unet'
    cfg_base.aug.prob_hallucination_max = 0.0
    cfg_base.aug.prob_hallucination_replay = 0.0
    cfg_base.model.use_positional_encoding = False
    configs['baseline'] = cfg_base

    # 2. + SPADE (SPADE on, No Buffer, No PE)
    cfg_spade = copy.deepcopy(base_cfg)
    cfg_spade.model.architecture = 'spade'
    cfg_spade.aug.prob_hallucination_max = 0.0
    cfg_spade.aug.prob_hallucination_replay = 0.0
    cfg_spade.model.use_positional_encoding = False
    configs['spade'] = cfg_spade

    # 3. + Buffer (SPADE on, Buffer on, No PE)
    cfg_buffer = copy.deepcopy(base_cfg)
    cfg_buffer.model.architecture = 'spade'
    cfg_buffer.aug.prob_hallucination_max = 0.65
    cfg_buffer.model.use_positional_encoding = False
    configs['buffer'] = cfg_buffer

    # 4. Full Model (+ PE)
    cfg_full = copy.deepcopy(base_cfg)
    cfg_full.model.architecture = 'spade'
    cfg_full.aug.prob_hallucination_max = 0.65
    cfg_full.model.use_positional_encoding = True
    configs['full'] = cfg_full

    return configs


def calc_focal_frequency_error(gt_slice: np.ndarray,
                               pred_slice: np.ndarray,
                               alpha: float = 1.0) -> float:
    """
    Computes the Focal Frequency Loss (FFL) as an evaluation metric.
    Numpy equivalent of the TF implementation to prevent memory leaks during inference.
    """
    # 1. Compute 2D FFT (Complex domain)
    fft_gt = np.fft.fft2(gt_slice)
    fft_pr = np.fft.fft2(pred_slice)

    # 2. Shift zero-frequency component to center
    fft_gt = np.fft.fftshift(fft_gt)
    fft_pr = np.fft.fftshift(fft_pr)

    # 3. Calculate complex difference matrix d(u,v) = |F_r - F_f|^2
    diff_matrix = np.abs(fft_gt - fft_pr) ** 2

    # 4. Normalize difference matrix to [0, 1] for focal weighting
    max_diff = np.max(diff_matrix)
    if max_diff < 1e-8:
        return 0.0

    diff_norm = diff_matrix / max_diff

    # 5. Calculate focal weights w(u,v) = d_norm(u,v)^alpha
    weight_matrix = diff_norm ** alpha

    # 6. Compute final weighted frequency loss
    ffl = np.mean(weight_matrix * diff_matrix)

    return float(ffl)


def calc_gradient_sharpness_error(gt_slice, pred_slice):
    """Numpy equivalent of your custom GradientSharpnessMetric"""
    dy_gt, dx_gt = np.gradient(gt_slice)
    dy_pr, dx_pr = np.gradient(pred_slice)
    grad_diff = np.abs(dy_gt - dy_pr) + np.abs(dx_gt - dx_pr)
    return float(np.mean(grad_diff))


def calculate_step_metrics(gt_vol, pred_vol, start_idx, end_idx, direction, ablation_name, vol_id):
    """Evaluates an autoregressive rollout step-by-step (k)."""
    metrics_list =[]
    iter_range = range(
        start_idx, end_idx
    ) if direction == 'forward' else range(
        end_idx - 1, start_idx - 1, -1
    )

    for i, z_idx in enumerate(iter_range):
        k = i + 1  # rollout step (1, 2, 3...)
        gt_slice = gt_vol[z_idx]
        pr_slice = pred_vol[z_idx]

        mae = float(np.mean(np.abs(gt_slice - pr_slice)))
        mse = float(np.mean(np.square(gt_slice - pr_slice)))
        psnr_val = float(psnr(gt_slice, pr_slice, data_range=1.0)) if mse > 0 else 100.0
        ssim_val = float(ssim(gt_slice, pr_slice, data_range=1.0))
        grad_err = calc_gradient_sharpness_error(gt_slice, pr_slice)
        ffl_err = calc_focal_frequency_error(gt_slice, pr_slice, alpha=1.0)

        metrics_list.append({
            'Ablation': ablation_name,
            'Volume_ID': vol_id,
            'Direction': direction,
            'Rollout_Step_k': k,
            'Z_Index': z_idx,
            'SSIM': ssim_val,
            'PSNR': psnr_val,
            'MAE': mae,
            'Grad_Error': grad_err,
            'FFL': ffl_err  # <--- NEW LOGIC
        })
    return metrics_list


def run_eda_mode(config, train_files, train_brats, input_only=False, downsample=2):
    logger.info(">>> STARTING EDA DASHBOARD MODE (PIL Optimized) <<<")

    manager = ActiveLoader(config, train_files, train_brats)
    manager.start()

    gen_ixi = iter(IXIActiveGenerator(manager)())
    gen_brats_clean = iter(BraTSActiveGenerator(manager, mode='clean')())
    gen_brats_tumor = iter(BraTSActiveGenerator(manager, mode='tumor')())

    dashboards = {}

    w_ixi = max(0.1, min(0.9, config.data.ixi_sampling_weight))
    w_brats_tumor = 0.20 # FIXME: 20% slices with tumor assumption

    batch_size = config.train.batch_size
    # FIXME: average 130 slices contain brain tissue assumption
    train_steps = max(100, (len(train_files) * 130) // batch_size)
    # Fetch actual epochs from config
    epochs = config.train.epochs
    total_samples = train_steps * batch_size * epochs

    logger.info(f"Simulating {total_samples} discrete samples...")

    for _ in tqdm(range(total_samples), desc="Simulating Epoch Flow"):

        # 1. Pipeline Probability Flow
        if random.random() < w_ixi:
            dir_name = 'eda_ixi_clean'
            model_inputs, y, info = next(gen_ixi)
        else:
            if random.random() < w_brats_tumor:
                dir_name = 'eda_brats_tumor'
                model_inputs, y, info = next(gen_brats_tumor)
            else:
                dir_name = 'eda_brats_clean'
                model_inputs, y, info = next(gen_brats_clean)

        # 2. Enforce Strict 160x160 Padding (Fixes Slice Mismatches)
        hist_pad, _, _ = GeometryOps.resize_and_pad(
            tf.convert_to_tensor(model_inputs['history_input']),
            config.data.padded_size, 'bicubic'
        )
        y_pad, _, _ = GeometryOps.resize_and_pad(
            tf.convert_to_tensor(y),
            config.data.padded_size, 'nearest'
        )

        hist_np = hist_pad.numpy()
        y_np = y_pad.numpy()

        # 3. Dashboard Routing
        path = info['volume_path']
        axis = info['axis']
        dash_key = (dir_name, path, axis)

        if dash_key not in dashboards:
            dashboards[dash_key] = VolumeDashboard(path, info['axis_size'],
                                                   axis, dir_name,
                                                   downsample=downsample)
        dash = dashboards[dash_key]

        # NEW: Increment the sequence Z-index for this specific brain
        dash.z_counter += 1
        current_z = dash.z_counter

        N = config.data.neighborhood
        spatial_indices = info['spatial_indices']

        # Target tumor mask is channel 1 of padded Y
        padded_tumor_mask = y_np[:, :, 1]

        # Update Input Slices
        for i in range(N):
            img_slice = hist_np[:, :, i]
            s_idx = spatial_indices[i]
            role = f"T-{N-i}"
            dash.update(s_idx, img_slice, role, info, z_index=current_z,
                        is_target=False, tumor_mask=padded_tumor_mask)

        # Update Target Slice
        if not input_only:
            target_img = y_np[:, :, 0]
            target_s_idx = spatial_indices[-1]
            dash.update(target_s_idx, target_img, "T", info, z_index=current_z,
                        is_target=True, tumor_mask=padded_tumor_mask)

    VisualizationSuite.plot_sampling_statistics(manager)

    manager.stop()

    logger.info(f"Rendering {len(dashboards)} unique volume dashboards to disk...")
    for dash in tqdm(dashboards.values(), desc="Rendering Images"):
        dash.render(input_only=input_only)

    logger.info("EDA Dashboard Generation Complete!")

In [ ]:
import gc


def run_pipeline(mode='train', test_mode=False):
    config = CFG

    # Override for Quick Kaggle Test
    test_mode = config.run_type == 'Interactive'
    if test_mode:
        logger.info(">>> TEST MODE ENABLED: Reducing HPO parameters for sanity check.")
        # config.train.epochs = 2
        config.hpo.n_trials = 2
        config.hpo.train_steps_per_trial = 10
        config.hpo.val_steps_per_trial = 5
        # config.data.ixi_cache_size = 4 # tiny pool

    # 0. Load Overrides
    if mode == 'train' and os.path.exists("best_hyperparams.json"):
        logger.info("Loading optimized hyperparameters...")
        config = Config.load("best_hyperparams.json")

    with PipelineTimer("1. File Discovery & Split"):
        # 1. Discover Files (Required before Manager init)
        t1_files = find_t1_files(config.data.data_root_ixi)
        brats_list = get_brats_subjects(config.data.data_root_brats)

        print(">>> Checking IXI:")
        analyze_dataset_geometry(t1_files, num_samples=2)

        print("\n>>> Checking BraTS:")
        analyze_dataset_geometry(brats_list, num_samples=2)

        # 2. Split
        random.shuffle(t1_files)
        # split = int(0.9 * len(t1_files))
        # train_files, val_files = t1_files[:split], t1_files[split:]
        random.shuffle(t1_files)
        ixi_split = int(0.9 * len(t1_files))
        train_files, val_files = t1_files[:ixi_split], t1_files[ixi_split:]

        # 2b. Split BraTS (Fixing the blind validation bug)
        random.shuffle(brats_list)
        brats_split = int(0.9 * len(brats_list))
        train_brats, val_brats = brats_list[:brats_split], brats_list[brats_split:]

        # Calculate Steps (Heuristic based on volume size ~130 slices)
        # Since we use random sampling, we define an "epoch" arbitrarily to match dataset size
        avg_slices = 130
        train_steps = max(
            100, (len(train_files) * avg_slices) // config.train.batch_size
        )
        val_steps = max(
            50, (len(val_files) * avg_slices) // config.train.batch_size
        )

        if mode == 'train':
            logger.info(f"Train Steps: {train_steps}, Val Steps: {val_steps}")

    if mode == 'hpo':
        # logger.info(">>> STARTING HPO MODE")
        # run_hpo_session(20, config, t1_files, brats_list)
        # return
        logger.info(">>> STARTING HPO MODE (Autoregressive Metric)")
        # run_hpo_session(20, config, train_files, brats_list)
        # Instantiate Engine
        hpo_engine = HPMEngine(config, train_files, brats_list)
        hpo_engine.run()
        # FIXME: decouple HPO merge and analyze, do not run in parallel
        merge_hpo_databases(config.hpo.storage_dir, config.hpo.study_name,
                            config.hpo.database_name)
        analyze_hpo_results(f"sqlite:///{config.hpo.database_name}") # "sqlite:///hpo_final.db"
        return

    if mode == 'eda':
        run_eda_mode(config, train_files, train_brats, input_only=False)
        return

    # --- Validation Loader (Static) ---
    # We load validation data into RAM once.
    # If val set is too big, use a smaller subset or switch to ActiveLoader logic for Val too.
    # For safety with 240 volumes, loading 10% (24 vols) is fine
    # val_loader = StaticLoader(config, val_files, [])
    val_loader = StaticLoader(config, val_files, val_brats)

    # --- Training Loader (Active) ---
    # Context manager ensures thread cleanup
    # with ActiveLoader(config, train_files, brats_list) as train_loader:
    # --- Training Loader (Active) ---
    with ActiveLoader(config, train_files, train_brats) as train_loader:

        # 3. Setup Generators
        with PipelineTimer("2. Generator & Dataset Setup"):
            # Training Generators (Infinite)
            # Note: We use the new class names 'IXIActiveGenerator'
            train_ixi = IXIActiveGenerator(train_loader)
            train_brats = BraTSActiveGenerator(train_loader, mode='clean')

            train_ds = get_training_dataset(train_ixi, train_brats, config)

            # Validation Generator (Reuse ActiveGen class but with StaticLoader)
            # It works because they share the 'get_volume' interface
            # val_gen = IXIActiveGenerator(val_loader)
            # val_ds = create_tf_dataset(val_gen, config, is_training=False).take(val_steps)
            # Use the Unbiased, Deterministic Validation Generator
            val_gen = SequentialValidationGenerator(val_loader,
                                                    max_slices_per_vol=15)
            # Remove .take() because it is natively finite now!
            val_ds = create_tf_dataset(val_gen, config, is_training=False)
            # Since val_ds is finite, tell Trainer to iterate until exhaustion
            val_steps = None  # FIXME: refine logic flow

            # Build Model
            model = ModelBuilder.build(config)

            # FIX: Pass the model to the loader so the background thread can
            # generate hallucinations without pausing the training loop!
            train_loader.model = model

            # Trainer
            trainer = Trainer(config, model, train_ds, val_ds, train_loader,
                              train_steps=train_steps, val_steps=val_steps)

        # 4. Pre-Training Visualization
        with PipelineTimer("3. visualization (Data Augmentation Check)"):
            # We create a specific visualization dataset using the Training Generators
            # This enables "hunting" for rare augmentations

            # Create finite datasets for sampling
            viz_ixi = create_tf_dataset(train_ixi, config, is_training=True,
                                        include_info=True)
            viz_brats = create_tf_dataset(train_brats, config, is_training=True,
                                          include_info=True)

            # Mix them (Weighted)
            # Use Configurable Weights
            w_ixi = config.data.ixi_sampling_weight
            # Clip to safety
            w_ixi = max(0.1, min(0.9, w_ixi))
            w_brats = 1.0 - w_ixi
            # Use repeat() to allow 'hunting' through many samples
            viz_ds = tf.data.Dataset.sample_from_datasets(
                [viz_ixi.repeat(), viz_brats.repeat()],
                weights=[w_ixi, w_brats]
            )

            # CRITICAL: Batch the viz dataset, otherwise plotter crashes on scalars
            viz_ds = viz_ds.batch(1)

            VisualizationSuite.plot_augmentations(viz_ds, num_groups=2,
                                                  batch_mode=config.batch_mode)

        # 5. Training Loop
        name_model = f"{config.model.name}_best.keras"
        if mode == 'viz' and os.path.exists(name_model):
            logger.info(">>> STARTING VISUALIZATION/ABLATION MODE")
            logger.info(f"Loading Pretrained Model [{name_model}]...")
            model.load_weights(name_model)
            history = None
        else:
            logger.info(">>> STARTING LONG-RUN TRAINING MODE")

            # --- RESTORED: RESUME TRAINING LOGIC ---
            if config.train.resume_training:
                if os.path.exists(config.train.resume_training):
                    logger.info(f"♻️ Resuming training from '{config.train.resume_training}'...")
                    # Load weights into the generator before the Trainer potentially wraps it in GAN mode
                    model.load_weights(config.train.resume_training)
                else:
                    logger.warning(f"⚠️ Resume path '{config.train.resume_training}' not found. Starting from scratch...")
            # ---------------------------------------

            history = trainer.train()

        # 6. Post-Training Analysis
        with PipelineTimer("4. Post-Training Analysis"):
            # Necessary visualizations for 'train' and 'vis' modes
            if history:
                VisualizationSuite.plot_training_history(history)

            VisualizationSuite.plot_hallucination_buffer(train_loader)

            # Use Val DS for best/worst
            VisualizationSuite.analyze_best_worst(model, val_ds, num_samples=5)

            # Reconstructions
            reconstructor = VolumeReconstructor(model, config)

            # Autoregressive on a real validation file
            # Note: val_loader has 'pool'. We can grab a path from there
            if val_files:
                VisualizationSuite.plot_autoregressive_performance(
                    reconstructor, {'id': None, 't1': val_files[0], 'seg': None},
                    val_loader
                )
            # Autoregressive on BraTS
            if brats_list:
                VisualizationSuite.plot_autoregressive_performance(
                    reconstructor, brats_list[0], train_loader
                )
                VisualizationSuite.plot_autoregressive_performance(
                    reconstructor, brats_list[0], train_loader,
                    masked_inference=True
                )

            # Bidirectional on a real validation file
            # Note: val_loader has 'pool'. We can grab a path from there
            if val_files:
                VisualizationSuite.plot_bidirectional(
                    reconstructor, {'id': None, 't1': val_files[0], 'seg': None},
                    val_loader
                )
            # Bidirectional on BraTS
            if brats_list:
                # We need a BraTS file. train_loader has them
                VisualizationSuite.plot_bidirectional(
                    reconstructor, brats_list[0], train_loader
                )
                VisualizationSuite.plot_bidirectional(
                    reconstructor, brats_list[0], train_loader,
                    masked_inference=True
                )

            # Data sampling statistics
            VisualizationSuite.plot_sampling_statistics(train_loader)
            if mode == 'viz':
                # TODO: extensive visualizations
                pass


def run_ablation_study_(base_config, train_files, val_files, train_brats, val_brats):
    logger.info(">>> STARTING CVPR ABLATION STUDY <<<")
    configs = get_ablation_configs(base_config)

    weights_dir = base_config.data.weights_dir
    results_dir = base_config.data.results_dir
    os.makedirs(results_dir, exist_ok=True)

    all_metrics =[]

    # --- KAGGLE INTERACTIVE PROTECTIONS ---
    is_interactive = base_config.run_type == 'Interactive'
    n_eval_vols = 2 if is_interactive else 10
    rollout_span = 5 if is_interactive else 20  # 10 steps total interactively vs 40 steps for prod

    if KAGGLE:
        # Prevent System RAM OOM by drastically shrinking the active caching pools
        base_config.data.ixi_cache_size = 100
        base_config.data.brats_cache_size = 100
    # --------------------------------------

    # 1. Lock in a fixed set of Test Volumes for fair comparison
    test_ixi = val_files[:n_eval_vols]
    test_brats = val_brats[:n_eval_vols]

    # 2. Prevent Kaggle Batch OOM: Cap static validation pool globally for Kaggle
    max_val_vols = 20 if KAGGLE else 50
    val_loader = StaticLoader(base_config, val_files[:max_val_vols], val_brats[:max_val_vols])

    for ab_name, cfg in configs.items():
        logger.info(f"=============================================")
        logger.info(f"   ABLATION PHASE: {ab_name.upper()}")
        logger.info(f"=============================================")

        # model_weights_path = f"{ABLATION_DIR}/model_{ab_name}.keras"
        primary_weights_path = os.path.join(weights_dir, f"model_{ab_name}.keras")
        local_weights_path = os.path.join(results_dir, f"model_{ab_name}.keras")

        # We must rebuild the active loader because the config dictates hallucination probabilities
        with ActiveLoader(cfg, train_files, train_brats) as train_loader:

            # --- MODEL BUILDING & TRAINING ---
            model = ModelBuilder.build(cfg)
            train_loader.model = model # attach for Buffer Updates

            if os.path.exists(primary_weights_path):
                logger.info(f"✅ Found existing weights in dataset for {ab_name}. Skipping training!")
                logger.debug(f"Model weights = {primary_weights_path}")
                model.load_weights(primary_weights_path)
            elif os.path.exists(local_weights_path):
                logger.info(f"✅ Found locally trained weights for {ab_name}. Skipping training!")
                logger.debug(f"Model weights = {local_weights_path}")
                model.load_weights(local_weights_path)
            else:
                logger.info(f"⚙️ Training {ab_name} model from scratch...")
                # Setup Training pipeline natively
                train_ixi = IXIActiveGenerator(train_loader)
                train_brats_gen = BraTSActiveGenerator(train_loader, mode='clean')
                train_ds = get_training_dataset(train_ixi, train_brats_gen, cfg)

                val_gen = SequentialValidationGenerator(val_loader, max_slices_per_vol=15)
                val_ds = create_tf_dataset(val_gen, cfg, is_training=False)

                # Heuristic steps based on dataset size (Scale down for Interactive)
                train_steps = max(
                    10, (len(train_files) * 130) // cfg.train.batch_size
                ) if is_interactive else max(
                    50, (len(train_files) * 130) // cfg.train.batch_size
                )

                trainer = Trainer(cfg, model, train_ds, val_ds, train_loader,
                                  train_steps=train_steps, val_steps=None)
                trainer.train(compile_model=True)

                logger.info(f"💾 Saving {ab_name} model weights...")
                model.save(local_weights_path)  # save full model to local writable dir

            # --- FIX: Kill the loader's background thread before inference ---
            train_loader.stop()

            # --- INFERENCE & ARTIFACT GENERATION (Phase 3) ---
            logger.info(f"🔍 Running Autoregressive Inference for {ab_name}...")
            reconstructor = VolumeReconstructor(model, cfg)
            reconstructor.cfg.batch_mode = True  # <--- FIX: silences the internal TQDM spam!

            # Helper to run eval on a specific pool
            def evaluate_pool(pool, dataset_name):
                profiler = InferenceProfiler(ab_name, results_dir)

                for vol_obj in tqdm(pool, desc=f"Evaluating {dataset_name} ({ab_name})"):
                    vol = vol_obj.t1
                    vol_id = os.path.basename(vol_obj.path).replace('.nii.gz', '').replace('.nii', '')

                    # Target a rollout near the center of the brain
                    center = vol.shape[0] // 2
                    start, end = center - rollout_span, center + rollout_span

                    if end - start < (rollout_span * 2) or start < 0: continue

                    # 1. Forward Pass (No mask -> Pure Autoregression)
                    profiler.start()
                    recon_vol = reconstructor.autoregressive_restore(
                        vol, start, end, 'forward', mask_volume=None
                    )
                    profiler.stop_and_log(vol_id)

                    # 2. Extract Per-Step Metrics
                    metrics = calculate_step_metrics(vol, recon_vol, start, end,
                                                     'forward', ab_name, vol_id)
                    all_metrics.extend(metrics)

                    # 3. Save Raw NPY Volume (For Phase 4 Image Matrices)
                    np.save(os.path.join(results_dir, f"{ab_name}_{vol_id}.npy"), recon_vol)

                    # Save Ground Truth (Only needs to be done once per volume, but overwriting is safe/fast)
                    np.save(os.path.join(results_dir, f"gt_{vol_id}.npy"), vol)

            # Evaluate on IXI and BraTS pools loaded in the static val_loader
            # --- EVALUATION ROUTING (Respecting Dataset Weights) ---
            if cfg.data.ixi_sampling_weight > 1e-8:
                evaluate_pool([
                    v for v in val_loader.pool['ixi'] if v.path in test_ixi
                ], "IXI")

            if cfg.data.ixi_sampling_weight < 1.0 - 1e-8:
                evaluate_pool([
                    v for v in val_loader.pool['brats']
                    if v.path in [b['t1'] for b in test_brats]
                ], "BraTS")

            # --- CRITICAL LEAK FIX: Wipe the RAM cache for the next ablation ---
            train_loader.pool.clear()
            train_loader.keys.clear()
            train_loader.files.clear()
            # -------------------------------------------------------

        # --- CRITICAL OOM FIX: Deep Memory Cleanup ---
        # 1. Delete local references to destroy the datasets and iterators
        if 'model' in locals(): del model
        if 'trainer' in locals(): del trainer
        if 'train_ds' in locals(): del train_ds
        if 'val_ds' in locals(): del val_ds
        if 'reconstructor' in locals(): del reconstructor

        # 2. Destroy the Perceptual Loss Singleton so it doesn't bleed into the next model
        global _GLOBAL_PERCEPTUAL_LOSS
        _GLOBAL_PERCEPTUAL_LOSS = None

        # 3. Force garbage collection and purge Keras backend
        gc.collect()
        tf.keras.backend.clear_session()
        gc.collect()
        # ---------------------------------------------

        # Clean up TF Graph before next ablation to prevent OOM
        tf.keras.backend.clear_session()

    # Save all metrics to a single CSV for Phase 4 plotting
    df = pd.DataFrame(all_metrics)
    # csv_path = f"{ABLATION_DIR}/ablation_metrics.csv"
    csv_path = os.path.join(results_dir, "ablation_metrics.csv")
    df.to_csv(csv_path, index=False)
    logger.info(f"✅ Ablation study complete! Metrics saved to {csv_path}")


def run_ablation_study(CFG):
    # Setup files (reusing existing logic from run_pipeline)
    t1_files = find_t1_files(CFG.data.data_root_ixi)
    brats_list = get_brats_subjects(CFG.data.data_root_brats)

    random.seed(42)
    random.shuffle(t1_files)
    ixi_split = int(0.9 * len(t1_files))
    t_files, v_files = t1_files[:ixi_split], t1_files[ixi_split:]

    random.shuffle(brats_list)
    brats_split = int(0.9 * len(brats_list))
    t_brats, v_brats = brats_list[:brats_split], brats_list[brats_split:]

    # Run the study
    run_ablation_study_(CFG, t_files, v_files, t_brats, v_brats)

* **Target File**: cvpr_plots.py

In [ ]:
import os
import glob
import shutil
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# 1. Config-Driven Pathing & Auto-Unzip
# Safely pull the paths from the global config
weights_dir = CFG.data.weights_dir
results_dir = CFG.data.results_dir
figures_dir = CFG.data.figures_dir

# Handle zipped datasets (common when uploading artifacts to Kaggle)
if os.path.exists(weights_dir + '.zip'):
    print(f"Extracting {weights_dir}.zip to local './ablations'...")
    with zipfile.ZipFile(weights_dir + '.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    weights_dir = 'ablations'
elif not os.path.exists(weights_dir):
    print(f"Warning: {weights_dir} not found. Defaulting to local 'ablations'.")
    weights_dir = 'ablations'

print(f"Using weights_dir: {weights_dir}")
print(f"Using results_dir: {results_dir}")
os.makedirs(figures_dir, exist_ok=True)

In [ ]:
# 2. Strict CVPR Aesthetics (No System LaTeX Dependency)
plt.rcParams.update({
    "text.usetex": False,          # Bypasses the 'latex not found' crash
    "font.family": "serif",        # Serif to match Times
    "mathtext.fontset": "stix",    # STIX matches Times New Roman math rendering perfectly
    "font.size": 8,                # Base size
    "axes.titlesize": 8,           # Title size
    "axes.labelsize": 8,           # Axis label size
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 7,          # Compact legend
    "figure.titlesize": 9,
    "pdf.fonttype": 42,            # Embed fonts
    "ps.fonttype": 42
})

# Standardized color palette for consistency across all charts
PALETTE = {
    # Ablation Track
    'baseline': '#E63946', # Red
    'spade': '#F4A261',    # Orange
    'buffer': '#2A9D8F',   # Green
    'full': '#264653',     # Dark Blue
    # SOTA Track
    'ours_full_masked': '#1D3557', # Navy Blue
    'monai_3d_ldm': '#8E44AD',     # Purple
    'monai_2d_ldm': '#E67E22',     # Carrot Orange
    'monai_vqgan': '#7F8C8D'       # Gray (Upper Bound)
}

LABELS = {
    # Ablation Track
    'baseline': 'Baseline U-Net',
    'spade': '+ SPADE',
    'buffer': '+ SPADE + Buffer',
    'full': '+ SPADE + Buffer + PE (Final)',
    # SOTA Track
    'ours_full_masked': 'Ours (2.5D SPADE AR)',
    'monai_3d_ldm': 'MONAI 3D LDM',
    'monai_2d_ldm': 'MONAI 2D LDM',
    # 'monai_vqgan': 'MONAI 3D VQ-GAN (Upper Bound)'
    'monai_vqgan': 'MONAI 3D VQ-GAN'
}


def generate_figure_1_drift_curve():
    """Generates Figure 1: The Quantitative Drift Curves (SSIM and FFL)."""
    print("Generating Figure 1: Drift Curves (SSIM & FFL)...")

    # csv_path = os.path.join(ABLATION_DIR, 'ablation_metrics.csv')
    csv_path = os.path.join(results_dir, 'ablation_metrics.csv')
    if not os.path.exists(csv_path):
        print(f"Error: Could not find {csv_path}")
        return

    df = pd.read_csv(csv_path)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # Only look at forward generation for simplicity in the chart
    df = df[df['Direction'] == 'forward']

    is_interactive = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Interactive') == 'Interactive'
    max_k = 10 if is_interactive else 40

    # --- PLOT 1: SSIM (Spatial Degradation) ---
    # CVPR Single Column width = 3.25 inches
    fig_ssim, ax_ssim = plt.subplots(figsize=(3.25, 2.5))

    sns.lineplot(
        data=df, x='Rollout_Step_k', y='SSIM', hue='Ablation',
        palette=PALETTE, linewidth=1.5, ax=ax_ssim
    )

    ax_ssim.set_xlim(1, max_k)
    ax_ssim.set_ylim(0.2, 1.0)
    ax_ssim.set_xlabel(r'Autoregressive Rollout Step ($k$)')
    ax_ssim.set_ylabel(r'Structural Similarity (SSIM) $\uparrow$')
    ax_ssim.grid(True, linestyle='--', alpha=0.6)

    handles, labels = ax_ssim.get_legend_handles_labels()
    safe_labels =[LABELS.get(l, l) for l in labels]
    ax_ssim.legend(handles, safe_labels, title="", loc='lower left',
                   fontsize=6, framealpha=0.75)

    plt.tight_layout()
    plt.savefig(os.path.join(figures_dir, 'Fig1_DriftCurve_SSIM.pdf'),
                dpi=300, bbox_inches='tight')
    plt.close(fig_ssim)
    print("  -> Saved Fig1_DriftCurve_SSIM.pdf")

    # --- PLOT 2: FFL (Frequency/Texture Degradation) ---
    # We only plot FFL if it exists in the dataframe (backwards compatibility check)
    if 'FFL' in df.columns:
        fig_ffl, ax_ffl = plt.subplots(figsize=(3.25, 2.5))

        sns.lineplot(
            data=df, x='Rollout_Step_k', y='FFL', hue='Ablation',
            palette=PALETTE, linewidth=1.5, ax=ax_ffl
        )

        ax_ffl.set_xlim(1, max_k)
        # FFL is unscaled, let Seaborn auto-scale the Y-axis but force bottom to 0
        ax_ffl.set_ylim(bottom=0.0)
        ax_ffl.set_xlabel(r'Autoregressive Rollout Step ($k$)')
        ax_ffl.set_ylabel(r'Focal Frequency Loss (FFL) $\downarrow$')
        ax_ffl.grid(True, linestyle='--', alpha=0.6)

        handles, labels = ax_ffl.get_legend_handles_labels()
        safe_labels =[LABELS.get(l, l) for l in labels]
        ax_ffl.legend(handles, safe_labels, title="", loc='upper left',
                      fontsize=6, framealpha=0.75)

        plt.tight_layout()
        plt.savefig(os.path.join(figures_dir, 'Fig1_DriftCurve_FFL.pdf'),
                    dpi=300, bbox_inches='tight')
        plt.close(fig_ffl)
        print("  -> Saved Fig1_DriftCurve_FFL.pdf")
    else:
        print("  -> Skipping FFL plot (FFL metric not found in CSV. Re-run ablation inference).")


def generate_figure_2_evolution_matrix():
    """Generates Figure 2: The Qualitative Evolution Matrix."""
    print("Generating Figure 2: Evolution Matrix...")

    # Find a target volume to visualize (preferably an IXI file for pure brain drift)
    npy_files = glob.glob(os.path.join(results_dir, 'full_*.npy'))
    if not npy_files:
        # print("Error: No ablation numpy arrays found.")
        print(f"Error: No ablation numpy arrays found in '{results_dir}'.")
        return

    # Extract Vol ID from the first 'full_...' file
    is_interactive = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Interactive') == 'Interactive'
    # Loop over up to 5 volumes to ensure we can verify the phenomenon isn't an anomaly
    for vol_idx, npy_file in enumerate(npy_files[:5 if is_interactive else None]):
        vol_id = os.path.basename(npy_file).replace('full_', '').replace('.npy', '')

        # Load all models and Ground Truth for this volume
        vols = {}
        for ab in['baseline', 'spade', 'buffer', 'full']:
            # path = os.path.join(ABLATION_DIR, f'{ab}_{vol_id}.npy')
            path = os.path.join(results_dir, f'{ab}_{vol_id}.npy')
            if os.path.exists(path):
                vols[ab] = np.load(path)

        # Locate the original NIfTI file to extract real Ground Truth
        gt_vol = None
        search_paths = glob.glob(
            f"/kaggle/input/**/{vol_id}", recursive=True
        ) + glob.glob(
            f"data/**/{vol_id}", recursive=True
        )
        if search_paths:
            gt_obj = VolumeLoader.load(search_paths[0])
            if gt_obj:
                gt_vol = gt_obj.t1

        row_keys = ['baseline', 'spade', 'buffer', 'full']
        if gt_vol is not None:
            vols['gt'] = gt_vol
            LABELS['gt'] = 'Ground Truth'
            row_keys.append('gt')

        # THE FIX: Dynamically adapt sequence steps to Interactive vs Batch mode limits
        is_interactive = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Interactive') == 'Interactive'
        rollout_span = 5 if is_interactive else 20
        start_idx = vols['full'].shape[0] // 2 - rollout_span

        # Let's slice at relative k steps:
        k_steps =[1, 4, 7, 9] if is_interactive else [1, 10, 20, 39]

        # 3.25" width, height scales with rows. Hspace adjusted to fit labels above images.
        fig, axes = plt.subplots(len(row_keys), len(k_steps),
                                 figsize=(3.25, 1.365 * len(row_keys)),
                                 gridspec_kw={'wspace': 0.035, 'hspace': 0.035})

        for r, ab in enumerate(row_keys):
            for c, k in enumerate(k_steps):
                ax = axes[r, c]
                z_idx = start_idx + k

                if ab in vols:
                    # Use vmin/vmax to prevent normalization clipping
                    ax.imshow(vols[ab][start_idx + k], cmap='bone', vmin=0, vmax=1)
                ax.axis('off')

                # Move Ablation label ABOVE the row (Left aligned) for compactness
                if c == 0:
                    safe_label = LABELS.get(ab, ab)#.replace('_', r'\_')
                    ax.text(0.1, 1.175, safe_label, transform=ax.transAxes,
                            ha='left', va='bottom', fontsize=8, weight='bold')

                # Show t-steps on every image to save vertical space
                ax.set_title(f"$t={k}$", fontsize=8, pad=2)

        plt.savefig(
            os.path.join(figures_dir, f"Fig2_EvolutionMatrix_vol{vol_idx + 1:03d}.pdf"),
            dpi=300, bbox_inches='tight', pad_inches=0.01
        )
        plt.close()
        print(f"  -> Saved Fig2_EvolutionMatrix_vol{vol_idx + 1:03d}.pdf")


def generate_figure_3_pe_proof():
    """Generates Figure 3: Z-Axis Proof showing ventricles appearing/disappearing."""
    print("Generating Figure 3: Positional Encoding Proof...")

    # We want to compare 'buffer' (No PE) vs 'full' (+PE)
    npy_files = glob.glob(os.path.join(results_dir, 'full_*.npy'))
    if not npy_files: return

    is_interactive = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Interactive') == 'Interactive'
    # Loop over up to 5 volumes to ensure we can verify the phenomenon isn't an anomaly
    for vol_idx, npy_file in enumerate(npy_files[:5 if is_interactive else None]):
        vol_id = os.path.basename(npy_file).replace('full_', '').replace('.npy', '')

        try:
            vol_buffer = np.load(os.path.join(results_dir, f'buffer_{vol_id}.npy'))
            vol_full = np.load(os.path.join(results_dir, f'full_{vol_id}.npy'))
        except Exception as e:
            print(f"Error loading arrays for Fig 3: {e}")
            return

        # Choose a slice late in the rollout near the top of the head
        # where ventricles should NOT exist but 'buffer' hallucinates them.
        # Safely target the very last generated slice instead of an out-of-bounds hardcoded index
        rollout_span = 5 if is_interactive else 20
        z_idx = (vol_full.shape[0] // 2) + (rollout_span - 1)

        # 1. Debugging: Check for NaNs/Infs which cause black screens
        # print(f"Debug: Slice {z_idx} Max Value: {vol_buffer[z_idx].max():.4f}")

        # Load Real Ground Truth
        gt_slice = vol_full[z_idx] # fallback
        search_paths = glob.glob(
            f"/kaggle/input/**/{vol_id}", recursive=True
        ) + glob.glob(
            f"data/**/{vol_id}", recursive=True
        )
        if search_paths:
            gt_obj = VolumeLoader.load(search_paths[0])
            if gt_obj:
                gt_slice = gt_obj.t1[z_idx]

        # Calculate MSEs
        mse_buffer = np.mean((gt_slice - vol_buffer[z_idx]) ** 2)
        mse_full = np.mean((gt_slice - vol_full[z_idx]) ** 2)

        # Use consistent vmin/vmax
        # 2x2 Grid for 3.25 inches (Much more readable)
        fig, axes = plt.subplots(2, 2, figsize=(3.25, 3.25 * (1.0 * gt_slice.shape[0] /
                                                              gt_slice.shape[1])),
                                 gridspec_kw={'wspace': 0.15,
                                              'hspace': 0.15})
        axes = axes.flatten()

        # 1. Model w/out PE
        axes[0].imshow(vol_buffer[z_idx], cmap='bone', vmin=0, vmax=1)
        axes[0].set_title("No Pos. Encoding", pad=2)

        # 2. Model w/ PE
        axes[1].imshow(vol_full[z_idx], cmap='bone', vmin=0, vmax=1)
        axes[1].set_title("With Pos. Encoding", pad=2)

        # 3. Error Map (No PE)
        err_buffer = np.abs(gt_slice - vol_buffer[z_idx])
        axes[2].imshow(err_buffer, cmap='inferno', vmin=0, vmax=0.3)
        axes[2].set_title(f"Error MSE: {mse_buffer:.4f}", pad=2)

        # 4. Error Map (PE)
        err_full = np.abs(gt_slice - vol_full[z_idx])
        axes[3].imshow(err_full, cmap='inferno', vmin=0, vmax=0.3)
        axes[3].set_title(f"Error MSE: {mse_full:.4f}", pad=2)

        for ax in axes:
            # ax.axis('off')
            ax.set_xticks([]); ax.set_yticks([])

        plt.savefig(
            os.path.join(figures_dir, f"Fig3_PE_Proof_vol{vol_idx + 1:03d}.pdf"),
            dpi=300, bbox_inches='tight', pad_inches=0.01
        )
        plt.close()
        print(f"  -> Saved Fig3_PE_Proof_vol{vol_idx + 1:03d}.pdf")


def generate_figure_4_anatomical_dsc():
    """Generates Figure 4: Downstream Anatomical Validation (DSC Bar Chart)."""
    print("Generating Figure 4: Anatomical Validation (SynthSeg DSC)...")

    csv_path = os.path.join(results_dir, 'anatomical_metrics.csv')
    if not os.path.exists(csv_path):
        print(f"Error: Could not find {csv_path}. Did you run the 'dsc' mode?")
        return

    df = pd.read_csv(csv_path)

    # Double column width = 6.875 inches
    fig, ax = plt.subplots(figsize=(6.875, 2.5))

    # Grouped bar chart
    sns.barplot(
        data=df,
        x='Region',
        y='DSC',
        hue='Ablation',
        palette=PALETTE,
        errorbar='ci', # shows 95% confidence interval
        capsize=0.05,
        errwidth=1.0,
        ax=ax
    )

    ax.set_ylim(0.0, 1.0)
    ax.set_xlabel('Anatomical Macro-Region')
    ax.set_ylabel(r'Dice Similarity Coefficient (DSC) $\uparrow$')
    ax.grid(True, axis='y', linestyle='--', alpha=0.6)

    # Fix legend
    handles, labels = ax.get_legend_handles_labels()
    safe_labels =[LABELS.get(l, l) for l in labels]
    ax.legend(handles, safe_labels, title="", loc='upper left',
              fontsize=7, framealpha=0.85, ncol=2)

    plt.tight_layout()
    plt.savefig(os.path.join(figures_dir, 'Fig4_Anatomical_DSC.pdf'),
                dpi=300, bbox_inches='tight')
    plt.close(fig)
    print("  -> Saved Fig4_Anatomical_DSC.pdf")


def generate_figure_5_frechet_distances():
    """Generates Figure 5: 3D-FID and FVD Metrics."""
    print("Generating Figure 5: 3D-FID and FVD...")

    csv_path = os.path.join(results_dir, 'frechet_metrics.csv')
    if not os.path.exists(csv_path):
        print(f"Error: Could not find {csv_path}. Did you run the 'frechet' mode?")
        return

    df = pd.read_csv(csv_path)
    
    # Double column width = 6.875 inches
    fig, axes = plt.subplots(1, 2, figsize=(6.875, 2.5),
                             gridspec_kw={'wspace': 0.35})

    sns.barplot(
        data=df, x='Ablation', y='3D-FID', palette=PALETTE, ax=axes[0]
    )
    axes[0].set_xlabel('')
    axes[0].set_ylabel(r'3D-FID $\downarrow$')
    axes[0].set_title('3D Structural Realism', fontsize=9)
    axes[0].grid(True, axis='y', linestyle='--', alpha=0.6)
    axes[0].set_xticklabels([LABELS.get(l.get_text(), l.get_text())
                             for l in axes[0].get_xticklabels()], 
                            rotation=45, ha='right', fontsize=7)

    # Plot 2: FVD
    sns.barplot(
        data=df, x='Ablation', y='FVD', palette=PALETTE, ax=axes[1]
    )
    axes[1].set_xlabel('')
    axes[1].set_ylabel(r'Fréchet Video Distance (FVD) $\downarrow$')
    axes[1].set_title('Z-Axis Sequential Continuity', fontsize=9)
    axes[1].grid(True, axis='y', linestyle='--', alpha=0.6)
    axes[1].set_xticklabels([LABELS.get(l.get_text(), l.get_text())
                             for l in axes[1].get_xticklabels()], 
                            rotation=45, ha='right', fontsize=7)

    plt.tight_layout()
    # plt.tight_layout(pad=1.0)
    plt.savefig(os.path.join(figures_dir, 'Fig5_Frechet_Distances.pdf'),
                dpi=300, bbox_inches='tight')
    plt.close(fig)
    print("  -> Saved Fig5_Frechet_Distances.pdf")


def generate_figure_6_pareto(combined=True):
    """Generates Figure 6: Pareto Fronts (Efficiency vs Fidelity)."""
    print("Generating Figure 6: Pareto Fronts...")

    perf_csv = os.path.join(results_dir, 'inference_performance.csv')
    fid_csv = os.path.join(results_dir, 'frechet_metrics.csv')
    dsc_csv = os.path.join(results_dir, 'anatomical_metrics.csv')
    abl_csv = os.path.join(results_dir, 'ablation_metrics.csv')

    if not os.path.exists(perf_csv):
        print("Error: Missing performance CSV.")
        return

    # 1. Load and aggregate Performance Data (Mean Time and VRAM per model)
    df_perf = pd.read_csv(perf_csv)
    df_master = df_perf.groupby('Model').agg({
        'Time_Seconds': 'mean',
        'Peak_VRAM_GB': 'mean'
    }).reset_index()

    # 2. Load and aggregate Fréchet Data
    if os.path.exists(fid_csv):
        df_fid = pd.read_csv(fid_csv).groupby('Ablation').mean(numeric_only=True).reset_index()
        df_master = pd.merge(df_master, df_fid, left_on='Model',
                             right_on='Ablation', how='left').drop(columns=['Ablation'])

    # 3. Load and aggregate Anatomical DSC Data
    if os.path.exists(dsc_csv):
        df_dsc = pd.read_csv(dsc_csv).groupby('Ablation').agg({'DSC': 'mean'}).reset_index()
        df_master = pd.merge(df_master, df_dsc, left_on='Model',
                             right_on='Ablation', how='left').drop(columns=['Ablation'])

    # 4. Load and aggregate Spatial Metrics (SSIM, MAE)
    if os.path.exists(abl_csv):
        df_abl = pd.read_csv(abl_csv).groupby('Ablation').agg({'SSIM': 'mean',
                                                               'MAE': 'mean'}).reset_index()
        df_master = pd.merge(df_master, df_abl, left_on='Model',
                             right_on='Ablation', how='left').drop(columns=['Ablation'])

    # Filter to only include the SOTA comparison track
    sota_models =['ours_full_masked', 'monai_3d_ldm', 'monai_2d_ldm', 'monai_vqgan']
    df_merged = df_master[df_master['Model'].isin(sota_models)].copy()

    if df_merged.empty:
        print("ERROR: df_merged is empty! Check if the Model names in inference_performance.csv match Ablation names.")
        return

    # Update palette to include MONAI
    local_palette = PALETTE.copy()
    # local_palette['monai_3d_ldm'] = '#8E44AD' # Purple for SOTA

    local_labels = LABELS.copy()
    # local_labels['monai_3d_ldm'] = 'MONAI 3D LDM (SOTA)'

    # Define the metrics we want to plot and their Y-axis labels
    metrics_to_plot =[
        ('DSC', r'Mean Anatomical DSC $\uparrow$'),
        ('3D-FID', r'3D-FID (Structural Error) $\downarrow$'),
        ('FVD', r'Fréchet Video Distance (FVD) $\downarrow$'),
        # ('SSIM', r'Structural Similarity (SSIM) $\uparrow$'),
        # ('MAE', r'Mean Absolute Error (MAE) $\downarrow$')
    ]

    available_metrics = [(col, lbl) for col, lbl in metrics_to_plot
                         if col in df_merged.columns]
    # logger.debug(f"merged dataframe columns = {df_merged.columns}")
    # logger.debug(f"available metrics = {available_metrics}")

    if combined and available_metrics:
        n_rows = len(available_metrics)
        # fig, axes = plt.subplots(n_rows, 2, figsize=(6.875, 2.5 * n_rows),
        #                          gridspec_kw={'wspace': 0.35, 'hspace': 0.4})
        # FIX: Increased height multiplier from 2.5 to 3.2 for square aspect ratio
        fig, axes = plt.subplots(n_rows, 2, figsize=(6.875, 3.25 * n_rows),
                                 gridspec_kw={'wspace': 0.35, 'hspace': 0.4})
        if n_rows == 1: axes = np.expand_dims(axes, axis=0)

        for i, (metric_col, metric_label) in enumerate(available_metrics):
            df_plot = df_merged.dropna(subset=[metric_col])
            
            # Plot 1: Time vs Metric
            sns.scatterplot(
                data=df_plot, x='Time_Seconds', y=metric_col, hue='Model', 
                palette=local_palette, s=150, edgecolor='black',
                ax=axes[i, 0], legend=False
            )
            axes[i, 0].set_xlabel(r'Inference Time per Volume (Seconds) $\downarrow$')
            axes[i, 0].set_ylabel(metric_label)
            if i == 0: axes[i, 0].set_title('Compute Time vs. Fidelity', fontsize=9)
            axes[i, 0].grid(True, linestyle='--', alpha=0.6)

            # Plot 2: VRAM vs Metric
            sns.scatterplot(
                data=df_plot, x='Peak_VRAM_GB', y=metric_col, hue='Model', 
                palette=local_palette, s=150, edgecolor='black', ax=axes[i, 1]
            )
            axes[i, 1].set_xlabel(r'Peak VRAM (GB) $\downarrow$')
            axes[i, 1].set_ylabel(metric_label)
            if i == 0: axes[i, 1].set_title('Memory Footprint vs. Fidelity', fontsize=9)
            axes[i, 1].grid(True, linestyle='--', alpha=0.6)

            # Remove individual legends
            if axes[i, 1].get_legend() is not None:
                axes[i, 1].legend().remove()

        # Add single legend at bottom
        handles, labels = (axes[0, 1].get_legend_handles_labels()
                           if axes[0, 1].get_legend_handles_labels()[0]
                           else axes[0, 0].get_legend_handles_labels())
        safe_labels =[local_labels.get(l, l) for l in labels]
        fig.legend(handles, safe_labels, loc='lower center', bbox_to_anchor=(0.5, 0.02),
                   fontsize=7, framealpha=0.85, ncol=2)

        plt.savefig(os.path.join(figures_dir, 'Fig6_Pareto_Combined.pdf'),
                    dpi=300, bbox_inches='tight')
        plt.close(fig)
        print("  -> Saved Fig6_Pareto_Combined.pdf")
    else:
        # Fallback to individual plots if combined is False
        for metric_col, metric_label in available_metrics:
            # if metric_col not in df_merged.columns:
            #     continue
    
            # Drop models that don't have this specific metric calculated
            df_plot = df_merged.dropna(subset=[metric_col])
            # if df_plot.empty:
            #     continue

            # Double column width, increased wspace to prevent Y-axis overlap
            fig, axes = plt.subplots(1, 2, figsize=(6.875, 2.75),
                                     gridspec_kw={'wspace': 0.35})
            # FIX: Increased height multiplier from 2.75 to 3.2 for square aspect ratio
            # fig, axes = plt.subplots(n_rows, 2, figsize=(6.875, 3.2),
            #                          gridspec_kw={'wspace': 0.35, 'hspace': 0.4})

            # Plot 1: Time vs Metric
            sns.scatterplot(data=df_plot, x='Time_Seconds',
                            y=metric_col, hue='Model', palette=local_palette,
                            s=150, edgecolor='black', ax=axes[0], legend=False)
            axes[0].set_xlabel(r'Inference Time per Volume (Seconds) $\downarrow$');
            axes[0].set_ylabel(metric_label);
            axes[0].set_title(f'Compute Time vs. {metric_col}', fontsize=9);
            axes[0].grid(True, linestyle='--', alpha=0.6)

            # Plot 2: VRAM vs Metric
            sns.scatterplot(data=df_plot, x='Peak_VRAM_GB',
                            y=metric_col, hue='Model', palette=local_palette,
                            s=150, edgecolor='black', ax=axes[1])
            axes[1].set_xlabel(r'Peak VRAM (GB) $\downarrow$');
            axes[1].set_ylabel(metric_label);
            axes[1].set_title(f'Memory Footprint vs. {metric_col}', fontsize=9);
            axes[1].grid(True, linestyle='--', alpha=0.6)

            # Remove the default legend from the second axis
            handles, labels = axes[1].get_legend_handles_labels()
            axes[1].legend().remove()

            # Create a single, centered legend BELOW the plots
            safe_labels =[local_labels.get(l, l) for l in labels]
            fig.legend(handles, safe_labels, loc='lower center',
                       bbox_to_anchor=(0.5, -0.15), fontsize=7,
                       framealpha=0.85, ncol=2)

            # Use pad to ensure the figure boundaries respect the external legend
            plt.tight_layout()
            plt.savefig(os.path.join(figures_dir, f'Fig6_Pareto_{metric_col}.pdf'),
                        dpi=300, bbox_inches='tight')
            plt.close(fig)
            print(f"  -> Saved Fig6_Pareto_{metric_col}.pdf")


def generate_supp_bidirectional():
    print("Generating Supp: Bidirectional Inpainting...")

    # Check both weights_dir and results_dir for epoch weights
    weight_files = glob.glob(os.path.join(weights_dir, '..', '*-???e.keras'))

    if not weight_files:
        weight_files = glob.glob(os.path.join(results_dir, '..', '*-???e.keras'))

    if not weight_files:
        print("  -> No epoch specific weights found. Skipping Supp Bidirectional.")
        return

    brats_list = get_brats_subjects(CFG.data.data_root_brats)
    if not brats_list: return
    test_brats = brats_list[0]

    vol_obj = VolumeLoader.load(test_brats['t1'], test_brats['seg'])
    if not vol_obj: return
    t1 = vol_obj.t1

    # Fallback ID parsing to fix the "None" title bug
    vol_id = test_brats.get('id')
    if not vol_id:
        vol_id = os.path.basename(test_brats['t1']).replace('_t1.nii.gz', '').replace('.nii', '')

    num_vis_slices = 5
    center = t1.shape[0] // 2
    start = center - num_vis_slices * 3
    end = center + num_vis_slices * 3
    vis_indices = np.linspace(start, end-1, num=num_vis_slices, dtype=int)

    model = ModelBuilder.build(CFG)
    reconstructor = VolumeReconstructor(model, CFG)
    reconstructor.cfg.batch_mode = True # silence TQDM

    for weight_path in sorted(weight_files):
        epoch_id = os.path.basename(weight_path).replace('.keras', '')
        print(f"  -> Processing {epoch_id}...")

        try:
            model.load_weights(weight_path)
        except:
            print(f"FAILED to load weights: {os.path.abspath(weight_path)}...")
            continue

        recon_fwd = reconstructor.autoregressive_restore(t1, start, end, 'forward')
        recon_bwd = reconstructor.autoregressive_restore(t1, start, end, 'backward')

        # Double Column width = 6.875 inches
        fig, axes = plt.subplots(num_vis_slices, 5, figsize=(6.875, 1.2 * num_vis_slices),
                                 gridspec_kw={'wspace': 0.02, 'hspace': 0.05})

        safe_title = f"Bidirectional Autoregressive Inpainting: {vol_id} ({epoch_id})"#.replace('_', r'\_')
        fig.suptitle(safe_title, fontsize=11, weight='bold', y=0.98)

        cols =["Ground Truth", "Forward", "Backward", "Fwd Diff", "Bwd Diff"]
        for ax, col in zip(axes[0], cols):
            ax.set_title(col, fontsize=9)

        for i, idx in enumerate(vis_indices):
            gt_slice = t1[idx]
            axes[i, 0].imshow(gt_slice, cmap='bone', vmin=0, vmax=1)
            axes[i, 0].set_ylabel(f"Slice {idx}", fontsize=9)

            axes[i, 1].imshow(recon_fwd[idx], cmap='bone', vmin=0, vmax=1)
            axes[i, 2].imshow(recon_bwd[idx], cmap='bone', vmin=0, vmax=1)
            axes[i, 3].imshow(np.abs(gt_slice - recon_fwd[idx]), cmap='inferno', vmin=0, vmax=0.3)
            axes[i, 4].imshow(np.abs(gt_slice - recon_bwd[idx]), cmap='inferno', vmin=0, vmax=0.3)

            for ax in axes[i]:
                ax.set_xticks([]); ax.set_yticks([])

        # Note: dpi=150 is used to keep supplementary PDF file size low for Overleaf
        plt.savefig(os.path.join(figures_dir, f'Supp_Bidirectional_{epoch_id}.pdf'),
                    dpi=150, bbox_inches='tight')
        plt.close()


def generate_supp_ablation_masked_ar():
    print("Generating Supp: Ablation Masked AR...")
    ixi_files = find_t1_files(CFG.data.data_root_ixi)
    if not ixi_files: return

    vol_obj = VolumeLoader.load(ixi_files[0])
    if not vol_obj: return

    vol = vol_obj.t1
    vol_id = os.path.basename(ixi_files[0]).replace('.nii.gz', '').replace('.nii', '')

    # Generate synthetic Glioma
    # synthetic_mask = VisualizationSuite._generate_synthetic_mask(vol, max_radius=18)

    # Generate realistic Glioma from BraTS pool
    # We need access to the BraTS pool. Since this function is called inside the script,
    # we can rebuild a quick static loader pool for BraTS, or pass it in.
    # For now, let's load a few BraTS files to sample from
    brats_list = get_brats_subjects(CFG.data.data_root_brats)
    brats_pool =[]
    for b in brats_list[:10]: # load 10 for variety
        v = VolumeLoader.load(b['t1'], b['seg'])
        if v: brats_pool.append(v)

    center = vol.shape[0] // 2
    # FORCE TUMOR TO CENTER
    synthetic_mask = VisualizationSuite._sample_real_brats_mask(vol, brats_pool, target_z=center)
    # synthetic_mask = VisualizationSuite._sample_real_brats_mask(vol, brats_pool)

    is_interactive = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Interactive') == 'Interactive'

    rollout_span = 10 if is_interactive else 30
    start = max(0, center - rollout_span)
    end = min(vol.shape[0], center + rollout_span)

    configs = get_ablation_configs(CFG)

    for ab_name, cfg in configs.items():
        # weight_path = os.path.join(ABLATION_DIR, f'model_{ab_name}.keras')
        weight_path = os.path.join(weights_dir, f'model_{ab_name}.keras')
        if not os.path.exists(weight_path):
            weight_path = os.path.join(results_dir, f'model_{ab_name}.keras')

        if not os.path.exists(weight_path): continue
        print(f"  -> Processing Masked AR for {os.path.abspath(ab_name)}...")

        model = ModelBuilder.build(cfg)
        # model.load_weights(weight_path)
        try:
            model.load_weights(weight_path)
        except:
            print(f"FAILED to load weights: {weight_path}...")
            continue

        reconstructor = VolumeReconstructor(model, cfg)
        reconstructor.cfg.batch_mode = True # silence TQDM

        recon_fwd = reconstructor.autoregressive_restore(
            vol, start, end, 'forward', mask_volume=synthetic_mask
        )
        recon_bwd = reconstructor.autoregressive_restore(
            vol, start, end, 'backward', mask_volume=synthetic_mask
        )

        # 1. Error Curves (Double column)
        fwd_mse = np.mean(np.square(vol[start:end] - recon_fwd[start:end]), axis=(1, 2))
        bwd_mse = np.mean(np.square(vol[start:end] - recon_bwd[start:end]), axis=(1, 2))

        fig, ax = plt.subplots(figsize=(6.875, 3.0))
        ax.plot(np.arange(start, end), bwd_mse, 'g-o', label='Backward (<-)', markersize=3)
        ax.plot(np.arange(start, end), fwd_mse, 'r-o', label='Forward (->)', markersize=3)
        # ax.axvline(x=center, color='b', linestyle='--', label='Start (Center Slice)')
        ax.axvline(x=center, color='b', linestyle='--', label='Tumor Center')

        safe_ab = LABELS.get(ab_name, ab_name).replace('_', r'\_')
        safe_vol_id = vol_id.replace('_', r'\_') 

        ax.set_title(f"Masked Error Accumulation [{safe_ab}] - {safe_vol_id}")
        ax.set_xlabel("Slice Index")
        ax.set_ylabel("Mean Squared Error")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.savefig(os.path.join(figures_dir, f'Supp_Masked_Curve_{ab_name}.pdf'),
                    dpi=150, bbox_inches='tight')
        plt.close()

        # 2. Reconstruction Details (Fixing the Anchor BUG)
        idx_show =[
            start + 2,                      # Near start
            center - 1,                     # Just before center
            center,                         # Center (First predicted slice going forward)
            center + 1,                     # Just after center
            end - 3                         # Near end
        ]

        N = cfg.data.neighborhood
        cols = N + 4
        rows = len(idx_show)

        fig, axes = plt.subplots(rows, cols,
                                 figsize=(6.875, 1.2 * rows),
                                 gridspec_kw={'wspace': 0.02,
                                              'hspace': 0.05})
        fig.suptitle(f"Reconstruction Details [{safe_ab}]",
                     fontsize=11, weight='bold', y=0.98)

        for i, idx in enumerate(idx_show):
            idx = int(max(0, min(idx, vol.shape[0] - 1)))

            # If it's the center, it IS a predicted slice in the forward pass
            if idx < center:
                img_r = recon_bwd[idx]
                context_vol = recon_bwd
                ctx_indices =[idx + N - k for k in range(N)]
            else:
                # Includes idx == center
                img_r = recon_fwd[idx]
                context_vol = recon_fwd
                ctx_indices =[idx - N + k for k in range(N)]

            # Context
            for j in range(N):
                c_idx = ctx_indices[j]
                ax = axes[i, j]
                if 0 <= c_idx < vol.shape[0]:
                    ctx_slice = context_vol[c_idx].copy()
                    # Apply the void visually so the plot reflects what the model actually saw
                    if synthetic_mask is not None:
                        m = (synthetic_mask[c_idx] > 0).astype(np.float32)
                        ctx_slice *= (1.0 - m)
                    ax.imshow(ctx_slice, cmap='bone', vmin=0, vmax=1)
                    if i == 0: ax.set_title(f"Ctx {j + 1}", fontsize=8)
                ax.axis('off')

            # Mask
            mask_slice = (synthetic_mask[idx] > 0.01).astype(np.float32)
            axes[i, N].imshow(mask_slice, cmap='gray')
            if i == 0: axes[i, N].set_title("Mask", fontsize=8)
            axes[i, N].axis('off')

            # GT
            axes[i, N + 1].imshow(vol[idx], cmap='bone', vmin=0, vmax=1)
            if i == 0: axes[i, N+1].set_title("GT", fontsize=8)
            axes[i, N + 1].axis('off')

            # Pred
            axes[i, N + 2].imshow(img_r, cmap='bone', vmin=0, vmax=1)
            if i == 0: axes[i, N+2].set_title("Pred", fontsize=8)
            axes[i, N + 2].axis('off')

            # Diff
            mse = np.mean((vol[idx] - img_r)**2)
            axes[i, N + 3].imshow(np.abs(vol[idx] - img_r), cmap='inferno',
                                  vmin=0, vmax=0.2)
            if i == 0: axes[i, N+3].set_title("Error", fontsize=8)
            axes[i, N + 3].axis('off')

            # Row label
            label = "Center (Pred)" if idx == center else f"Slice {idx}"
            axes[i, 0].text(-0.1, 0.5, label, va='center', ha='right', rotation=90,
                            transform=axes[i, 0].transAxes, fontsize=8)

        plt.savefig(os.path.join(figures_dir, f'Supp_Masked_Details_{ab_name}.pdf'),
                    dpi=150, bbox_inches='tight')
        plt.close()


import copy


def generate_supp_scaling_failure(render_errors=True):
    print("Generating Supp: Scaling Failure (B0 vs B1)...")

    # Resolve paths to the two different ablation directories
    b0_weights = os.path.abspath(os.path.join(weights_dir, '..', 'ablations-B0', 'model_full.keras'))
    b1_weights = os.path.abspath(os.path.join(weights_dir, '..', 'ablations-B1', 'model_full.keras'))

    if not os.path.exists(b0_weights) or not os.path.exists(b1_weights):
        print(f"  -> B0 or B1 weights not found. Skipping scaling failure plot.\n"
              f"     Checked: {b0_weights}\n     and: {b1_weights}")
        return

    # brats_list = get_brats_subjects(CFG.data.data_root_brats)
    # if not brats_list: return
    # test_brats = brats_list[0]

    # Use IXI dataset for pure healthy tissue drift evaluation
    ixi_files = find_t1_files(CFG.data.data_root_ixi)
    if not ixi_files: return
    # test_ixi = ixi_files[0]

    is_interactive = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Interactive') == 'Interactive'
    num_eval_vols = 5 if is_interactive else 10

    # vol_obj = VolumeLoader.load(test_brats['t1'], test_brats['seg'])
    for vol_idx, test_ixi in enumerate(ixi_files[:num_eval_vols]):
        vol_obj = VolumeLoader.load(test_ixi)
        if not vol_obj: continue
        vol = vol_obj.t1
        # vol_id = test_brats.get('id', 'Unknown')
        vol_id = os.path.basename(test_ixi).replace('.nii.gz', '').replace('.nii', '')
    
        center = vol.shape[0] // 2
        rollout_span = 10
        start = max(0, center - rollout_span)
        end = min(vol.shape[0], center + rollout_span)
    
        # Setup B0 Reconstructor
        cfg_b0 = copy.deepcopy(CFG)
        cfg_b0.model.backbone = 'B0'
        model_b0 = ModelBuilder.build(cfg_b0)
        model_b0.load_weights(b0_weights)
        recon_b0 = VolumeReconstructor(model_b0, cfg_b0)
        recon_b0.cfg.batch_mode = True
    
        # Setup B1 Reconstructor
        cfg_b1 = copy.deepcopy(CFG)
        cfg_b1.model.backbone = 'B1'
        model_b1 = ModelBuilder.build(cfg_b1)
        model_b1.load_weights(b1_weights)
        recon_b1 = VolumeReconstructor(model_b1, cfg_b1)
        recon_b1.cfg.batch_mode = True

        print("  -> Running B0 and B1 inference on CPU...")
        # Force execution on CPU to avoid allocating VRAM during plotting phase
        with tf.device('/CPU:0'):
            pred_b0 = recon_b0.autoregressive_restore(vol, start, end, 'forward')
            pred_b1 = recon_b1.autoregressive_restore(vol, start, end, 'forward')

        # Plotting logic
        num_vis_slices = 5
        vis_indices = np.linspace(start + 2, end - 2, num=num_vis_slices, dtype=int)

        # fig, axes = plt.subplots(num_vis_slices, 5, figsize=(6.875, 1.2 * num_vis_slices),
        #                          gridspec_kw={'wspace': 0.02, 'hspace': 0.05})
        # FIX: Single CVPR column width (3.25 inches). Tighter layout
        num_cols = 3 + 2 * int(render_errors)
        fig, axes = plt.subplots(num_vis_slices, num_cols,
                                 figsize=(3.25, 0.75 * num_vis_slices), 
                                 gridspec_kw={'wspace': 0.05, 'hspace': 0.05})

        # fig.suptitle(f"Scaling Failure Mode (B0 vs B1): {vol_id}",
        #              fontsize=11, weight='bold', y=0.98)
        safe_vol_id = vol_id.replace('_', r'\_')
        fig.suptitle(f"Scaling Failure (B0 vs B1):\n{safe_vol_id}",
                     fontsize=8, weight='bold', y=1.0)

        # cols = ["Ground Truth", "B0 Pred", "B1 Pred", "B0 Error", "B1 Error"]
        # Abbreviated columns to fit 3.25 inches
        cols = ["GT", "B0", "B1", "|B0-GT|", "|B1-GT|"]
        for ax, col in zip(axes[0], cols[:num_cols]):
            ax.set_title(col, fontsize=9)
    
        for i, idx in enumerate(vis_indices):
            gt_slice = vol[idx]
            p_b0 = pred_b0[idx]
            p_b1 = pred_b1[idx]
    
            axes[i, 0].imshow(gt_slice, cmap='bone', vmin=0, vmax=1)
            axes[i, 0].set_ylabel(f"Slice {idx}", fontsize=9)
    
            axes[i, 1].imshow(p_b0, cmap='bone', vmin=0, vmax=1)
            axes[i, 2].imshow(p_b1, cmap='bone', vmin=0, vmax=1)
    
            if render_errors:
                axes[i, 3].imshow(np.abs(gt_slice - p_b0), cmap='inferno', vmin=0, vmax=0.3)
                axes[i, 4].imshow(np.abs(gt_slice - p_b1), cmap='inferno', vmin=0, vmax=0.3)
    
            for ax in axes[i]:
                ax.set_xticks([]); ax.set_yticks([])
    
        plt.savefig(os.path.join(figures_dir, f'Supp_Scaling_Failure_vol{vol_idx + 1:03d}.pdf'),
                    dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"  -> Saved Supp_Scaling_Failure_vol{vol_idx + 1:03d}.pdf")


def run_cvpr_rendering(render_supp=True, render_bidirectional=False):
    print("--- Starting CVPR Visualization Generation ---")
    generate_figure_1_drift_curve()
    generate_figure_2_evolution_matrix()
    generate_figure_3_pe_proof()
    generate_figure_4_anatomical_dsc()
    generate_figure_5_frechet_distances()
    generate_figure_6_pareto()
    if render_supp:
        if render_bidirectional:
            generate_supp_bidirectional()
        generate_supp_ablation_masked_ar()
        generate_supp_scaling_failure(render_errors=True)
    else:
        print("Skipping Supplementary Figures to save time (render_supp=False).")
    print("--- All Figures Generated Successfully! ---")

# Cell 17: The Optuna Interface

In [ ]:
import gc

In [ ]:
def compute_autoregressive_score(model, config, val_loader, ixi_files,
                                 brats_files, n_steps=5):
    """
    Calculates the mean MAE over a multi-step autoregressive rollout.
    This is the 'True' metric we want to optimize.
    """
    reconstructor = VolumeReconstructor(model, config)

    # 1. Grab pre-loaded MedicalVolume objects directly from RAM
    test_volumes = []

    for path in ixi_files:
        # Match the path string to the loaded object in the list
        vol = next((v for v in val_loader.pool['ixi'] if v.path == path), None)
        if vol: test_volumes.append(vol)

    for d in brats_files:
        vol = next((v for v in val_loader.pool['brats'] if v.path == d['t1']),
                   None)
        if vol: test_volumes.append(vol)

    total_mae = 0.0
    total_slices = 0

    for vol_obj in test_volumes:
        # 2. Bypass disk entirely
        vol = vol_obj.t1

        # Determine Range (Middle of brain)
        # We need n_steps. Let's start slightly before center
        center = vol.shape[0] // 2
        start = center - (n_steps // 2)
        end = start + n_steps

        # Bounds check
        if start < 0 or end > vol.shape[0]: continue

        # Run Pure Autoregression (Hardest task)
        # We don't use Teacher Forcing (mask) here because we want to measure stability
        recon = reconstructor.autoregressive_restore(vol, start, end, 'forward',
                                                     mask_volume=None)

        # Calculate MAE over the generated sequence
        # GT vs Prediction
        gt_seq = vol[start:end]
        pred_seq = recon[start:end]

        mae = np.mean(np.abs(gt_seq - pred_seq))

        total_mae += mae
        total_slices += 1

    if total_slices == 0: return float('inf')

    return total_mae / total_slices

## Static version

In [ ]:
import copy
import gc
import os
import uuid
import random
import shutil
import tracemalloc

import numpy as np
import optuna
import tensorflow as tf

In [ ]:
class HPMEngine:
    """
    Ultra-Stable HPO Engine.
    Uses Static RAM allocation and a Single-Dataset paradigm.
    Zero disk I/O, Zero background threads, Zero memory fragmentation.
    """
    def __init__(self, config: Config, train_files: list, brats_files: list):
        self.cfg = config
        self.tracemem = self.cfg.hpo.tracemalloc

        # FIXED REPRESENTATIVE SUBSETS (e.g., 150 IXI, 150 BraTS)
        # Random seed ensures consistent subset across runs
        random.seed(42)

        # Shuffle before slicing
        random.shuffle(train_files)
        random.shuffle(brats_files)

        # 1. Calculate dynamic proportional splits based on global generator weights
        w_ixi = self.cfg.data.ixi_sampling_weight
        w_ixi = max(0.1, min(0.9, w_ixi)) # safety clip
        w_brats = 1.0 - w_ixi

        # Number of volumes per subset
        n_train_ixi = int(self.cfg.hpo.base_train_volumes * w_ixi)
        n_train_brats = int(self.cfg.hpo.base_train_volumes * w_brats)

        n_valid_ixi = int(self.cfg.hpo.base_val_volumes * w_ixi)
        n_valid_brats = int(self.cfg.hpo.base_val_volumes * w_brats)

        n_eval_ixi = int(self.cfg.hpo.base_eval_volumes * w_ixi)
        n_eval_brats = int(self.cfg.hpo.base_eval_volumes * w_brats)

        # 2. Slice datasets safely
        self.train_ixi = train_files[:n_train_ixi]
        self.train_brats = brats_files[:n_train_brats]

        self.val_ixi = train_files[n_train_ixi : n_train_ixi + n_valid_ixi]
        self.val_brats = brats_files[n_train_brats : n_train_brats + n_valid_brats]

        # Evaluation subset for the AR Metric (sliced from Val pool so it's unseen in training)
        self.eval_ixi = self.val_ixi[:n_eval_ixi]
        self.eval_brats = self.val_brats[:n_eval_brats]

        logger.info(
            f"HPO Volumes Loaded -> Train: {len(self.train_ixi)} IXI / {len(self.train_brats)} BraTS"
            f" | Valid: {len(self.val_ixi)} IXI / {len(self.val_brats)} BraTS"
        )

    def _objective(self, trial):
        # -1. Start the memory profiler at the very beginning of the trial
        if self.tracemem:
            tracemalloc.start()
        # TODO: refactor global telemetry instance
        telemetry.log("TRIALSTART", trial.number)

        logger.info(f">>> Starting trial #{trial.number:03d}...")
        # 0. Clean Slate Keras Graph
        # tf.keras.backend.clear_session()
        # gc.collect()

        # A. RESTORE PRISTINE MODEL WEIGHTS
        self.model.set_weights(self.initial_weights)

        # B. ZERO OUT OPTIMIZER MOMENTUM (Prevents contamination between trials)
        if hasattr(self.model, 'optimizer') and self.model.optimizer is not None:
            # for var in self.model.optimizer.variables():
            #     var.assign(tf.zeros_like(var))
            # FIX: In Keras 3, optimizer.variables is a list property, not a method!
            opt_vars = self.model.optimizer.variables
            if callable(opt_vars):
                # Fallback for Keras 2 just in case
                opt_vars = opt_vars()

            for var in opt_vars:
                var.assign(tf.zeros_like(var))

        # C. INJECT NEW HYPERPARAMETERS DYNAMICALLY (No recompilation!)
        if not self.cfg.train.gan_mode:
            lr_inline = trial.suggest_float('lr', 5e-5, 5e-4, log=True)
            # K.set_value(self.model.optimizer.learning_rate, lr_inline)
            # --- FIX: Keras 3 robust learning rate injection ---
            if hasattr(self.model.optimizer.learning_rate, 'assign'):
                # It is a tf.Variable (Standard Keras 3 behavior)
                self.model.optimizer.learning_rate.assign(lr_inline)
            else:
                # It is a raw property (Fallback)
                self.model.optimizer.learning_rate = lr_inline
        else:
            # (GAN logic remains the same, updating both optimizers)
            lr_g_inline = trial.suggest_float('lr_g', 5e-5, 5e-4, log=True)
            lr_d_inline = trial.suggest_float('lr_d', 5e-5, 5e-4, log=True)

            if hasattr(self.trainer.model.g_optimizer.learning_rate, 'assign'):
                self.trainer.model.g_optimizer.learning_rate.assign(lr_g_inline)
                self.trainer.model.d_optimizer.learning_rate.assign(lr_d_inline)
            else:
                self.trainer.model.g_optimizer.learning_rate = lr_g_inline
                self.trainer.model.d_optimizer.learning_rate = lr_d_inline

        # 1. Update self.cfg IN-PLACE
        if not self.cfg.train.gan_mode:
            # self.cfg.train.learning_rate = trial.suggest_float('lr', 5e-5, 5e-4, log=True)
            ...  # skip learning rate for deterministic approach
            if not self.cfg.train.use_spatial_loss:
                #     --- FIX: Update the actual TF Variables on the compiled loss function ---
                self.loss_fn.w_tumor.assign(trial.suggest_float('l_tumor', 0.0, 1.0))
                self.loss_fn.w_healthy.assign(trial.suggest_float('l_healthy', 0.0, 1.0))
                self.loss_fn.w_bg.assign(trial.suggest_float('l_background', 0.0, 1.0))
                self.loss_fn.w_grad.assign(trial.suggest_float('l_grad', 0.0, 1.0))
                self.loss_fn.w_perc.assign(trial.suggest_float('l_perceptual', 0.0, 1.0))
                self.loss_fn.w_spec.assign(trial.suggest_float('l_spectral', 0.0, 1.0))
        else:
            # self.cfg.train.learning_rate_g = trial.suggest_float('lr_g', 5e-5, 5e-4, log=True)
            # self.cfg.train.learning_rate_d = trial.suggest_float('lr_d', 5e-5, 5e-4, log=True)
            # --- FIX: Inject GAN Loss Weights dynamically ---
            # In GAN mode, self.trainer.model is the SPADEGANTrainer
            self.trainer.model.w_l1.assign(trial.suggest_float('w_l1', 0.0, 1.0))
            self.trainer.model.w_perc.assign(trial.suggest_float('w_perc', 0.0, 1.0))
            self.trainer.model.w_spec.assign(trial.suggest_float('w_spec', 0.0, 1.0))
            self.trainer.model.w_gan.assign(trial.suggest_float('w_gan', 0.0, 1.0))
            self.trainer.model.w_fm.assign(trial.suggest_float('w_fm', 0.0, 1.0))

            if self.cfg.model.architecture == 'vae':
                self.trainer.model.w_kl.assign(trial.suggest_float('w_kl', 0.0, 0.1))

        if not self.cfg.train.gan_mode and not self.cfg.train.use_spatial_loss:
            ...
        #     self.cfg.train.lambda_tumor = trial.suggest_float('l_tumor', 0.0, 1.0)
        #     self.cfg.train.lambda_healthy = trial.suggest_float('l_healthy', 0.0, 1.0)
        #     self.cfg.train.lambda_background = trial.suggest_float('l_background', 0.0, 1.0)
        #     self.cfg.train.lambda_grad = trial.suggest_float('l_grad', 0.0, 1.0)
        #     self.cfg.train.lambda_perceptual = trial.suggest_float('l_perceptual', 0.0, 1.0)
        #     self.cfg.train.lambda_spectral = trial.suggest_float('l_spectral', 0.0, 1.0)
        elif not self.cfg.train.use_spatial_loss:
            ...
        #     self.cfg.train.weight_l1 = trial.suggest_float('w_l1', 0.0, 1.0)
        #     self.cfg.train.weight_perceptual = trial.suggest_float('w_perc', 0.0, 1.0)
        #     self.cfg.train.weight_spectral = trial.suggest_float('w_spec', 0.0, 1.0)
        #     self.cfg.train.weight_gan = trial.suggest_float('w_gan', 0.0, 1.0)
        #     self.cfg.train.weight_fm = trial.suggest_float('w_fm', 0.0, 1.0)

        # Short training for HPO
        self.cfg.train.epochs = 5

        # 2. Create FINITE Training Dataset
        # FIX: The `.take()` forces the infinite generator to stop, destroying the C++ zombie threads
        # total_train_batches = self.cfg.hpo.train_steps_per_trial * self.cfg.train.epochs
        # finite_train_ds = self.train_ds.take(total_train_batches)

        # 2. Create Pure Python Training Dataset INSIDE the objective
        # This completely bypasses the C++ tf.data memory leak
        train_ds = HpoTrainSequence(
            self.train_gen_ixi,
            self.train_gen_brats,
            self.cfg,
            self.cfg.hpo.train_steps_per_trial
        )

        combined_score = float('inf')
        ar_score = float('inf')
        os_score = float('inf')

        try:
            # 3. Build & Train (Uses globally defined datasets)
            # model = ModelBuilder.build(self.cfg)

            # trainer = Trainer(self.cfg, model, finite_train_ds, self.val_ds, self.train_loader,
            #                   train_steps=self.cfg.hpo.train_steps_per_trial,
            #                   val_steps=None)
            # trainer = Trainer(self.cfg, model, train_ds, self.val_ds_sequence, self.train_loader,
            #                   train_steps=self.cfg.hpo.train_steps_per_trial,
            #                   val_steps=None)
            # --- FIX: Do NOT build the model here. Do NOT recreate the Trainer! ---
            # Just inject the new sequence into the global trainer.
            self.trainer.train_ds = train_ds
            self.trainer.train_steps = self.cfg.hpo.train_steps_per_trial

            # history = self.trainer.train(generator_ref=None)
            # Train the global model WITHOUT recompiling
            history = self.trainer.train(compile_model=False)

            # 4. Extract 1-Shot Validation Metric
            os_score = float('inf')
            if history and 'val_loss' in history.history:
                os_score = history.history['val_loss'][-1]
            elif history and 'val_l1_loss' in history.history: # fallback for GAN mode
                os_score = history.history['val_l1_loss'][-1]

            # 5. Score using RAM Pool (Zero Disk I/O)
            if np.isnan(history.history['loss'][-1]):
                ar_score = float('inf')
            else:
                # Calculate AR Score reading directly from val_loader.pool
                # --- FIX: Evaluate the global self.model! ---
                ar_score = compute_autoregressive_score(
                    self.model, self.cfg, self.val_loader, self.eval_ixi,
                    self.eval_brats, n_steps=self.cfg.hpo.ar_rollout_steps
                )

            # 6. Blend the Metrics
            w_ar = self.cfg.hpo.metric_ar_weight
            w_os = 1.0 - w_ar

            # Avoid nan propagation in multiplication
            if np.isnan(ar_score) or np.isnan(os_score):
                combined_score = float('inf')
            else:
                combined_score = (ar_score * w_ar) + (os_score * w_os)

        except Exception as e:
            logger.error(f"Trial failed: {e}")
            # ar_score = float('inf')
            combined_score = float('inf')

        finally:
            # Delete local references so Graph can be destroyed
            # del model
            # del trainer
            del train_ds
            if 'history' in locals(): del history
            tf.keras.backend.clear_session()
            gc.collect()

            # 2. Take a snapshot of RAM exactly before the trial ends
            if self.tracemem:
                snapshot = tracemalloc.take_snapshot()

                # 3. Print the top 10 memory-hogging lines of code
                top_stats = snapshot.statistics('lineno')
                print(f"\n--- [PROFILER] TOP 10 RAM ALLOCATIONS (TRIAL {trial.number}) ---")
                for stat in top_stats[:10]:
                    print(stat)

                tracemalloc.stop()

            telemetry.log("TRIALSTOP", trial.number)
            # logger.info(f"Trial #{trial.number:03d} finished with Score: {ar_score:.4f}")
            logger.info(
                f"Trial #{trial.number:03d} finished with Combined Score: {combined_score:.4f} (AR: {ar_score:.4f}, OS: {os_score:.4f})"
            )

        # return ar_score
        return combined_score

    def run(self):
        os.makedirs(self.cfg.hpo.storage_dir, exist_ok=True)

        # Define paths
        unique_id = str(uuid.uuid4())[:8]
        # db_name = f"{self.cfg.hpo.storage_dir}/hpo_{unique_id}.db"
        db_name = os.path.join(self.cfg.hpo.storage_dir, f"hpo_{unique_id}.db")
        # final_db_name = os.path.join(self.cfg.hpo.storage_dir, self.cfg.hpo.database_name)
        final_db_name = self.cfg.hpo.database_name
        # --- CRITICAL FIX: WARM START OPTUNA ---
        # If the merged database exists from previous runs, copy it to our worker's unique DB
        if os.path.isfile(self.cfg.hpo.database_preload) and not os.path.exists(final_db_name):
            logger.info(f"Found preload database: {self.cfg.hpo.database_preload}")
            shutil.copy(self.cfg.hpo.database_preload, final_db_name)
        # Optuna will load the history and continue the TPE search intelligently
        if os.path.exists(final_db_name):
            shutil.copy(final_db_name, db_name)
            logger.info(f"Loaded previous Optuna history from sqlite:///{final_db_name}")
        else:
            logger.info("No previous history found. Starting fresh.")

        storage_url = f"sqlite:///{db_name}"

        logger.info(f"Starting Static-RAM HPO Node. Storage: {storage_url}")

        # 1. LOAD ALL DATA INTO RAM ONCE
        logger.info(f"Allocating {len(self.train_ixi)} IXI and {len(self.train_brats)} BraTS volumes to RAM...")

        self.val_loader = StaticLoader(self.cfg, self.val_ixi, self.val_brats)
        self.train_loader = StaticLoader(self.cfg, self.train_ixi, self.train_brats)

        # Disable Hallucination updating during HPO since StaticLoader doesn't have a background thread
        self.cfg.aug.prob_hallucination_replay = 0.0
        # Disable hullucination buffer at all, since it's impact insignificant during HPO
        self.cfg.aug.prob_hallucination_max = 0.0
        # self.cfg.aug.prob_hallucination_replay = 0.0

        # 2. CREATE TF.DATA DATASETS EXACTLY ONCE
        # logger.info("Initializing global tf.data pipelines...")
        logger.info("Initializing python generators...")

        # These just yield numpy arrays from RAM. No TF logic
        self.train_gen_ixi = IXIActiveGenerator(self.train_loader)
        self.train_gen_brats = BraTSActiveGenerator(self.train_loader, mode='clean')

        # Convert all validation data into a static Sequence once.
        logger.info("Materializing Validation Sequence...")
        val_gen = SequentialValidationGenerator(self.val_loader, max_slices_per_vol=10)
        self.val_ds_sequence = HpoValidSequence(val_gen, self.cfg)

        # 3. PRE-WARM SINGLETON LOSS
        # Forces EfficientNet to download/build once in the global context
        # logger.info("Pre-warming Global Perceptual Loss Singleton...")
        # import src.models.losses as losses_module
        # losses_module.get_perceptual_loss(self.cfg)
        # get_perceptual_loss(self.cfg)
        # PRE-WARM SINGLETON LOSS (only if needed)
        if self.cfg.train.lambda_perceptual >= 0.01 or self.cfg.train.weight_perceptual >= 0.01:
            logger.info("Pre-warming Global Perceptual Loss Singleton...")
            get_perceptual_loss(self.cfg)
        else:
            logger.info("Perceptual Loss weight < 0.01. Bypassing EfficientNet initialization.")

        # --- THE FIX: COMPILE THE C++ GRAPH EXACTLY ONCE ---
        logger.info("Building and Compiling Model Graph...")
        self.model = ModelBuilder.build(self.cfg)

        # Save pristine weights
        self.initial_weights = self.model.get_weights()

        # Instantiate Loss and Trainer once globally
        self.loss_fn = CompositeLoss(self.cfg)

        # We manually compile the model here so the Trainer doesn't recreate the graph
        optimizer = tf.keras.optimizers.Adam(learning_rate=self.cfg.train.learning_rate)
        metrics = [
            # 'mae', 'mse', SSIMMetric(), PSNRMetric(), GradientSharpnessMetric()
            OracleMAE(), # OracleMAEMetric(),
            OracleMSE(), # OracleMSEMetric(),
            OracleSSIM(), # SSIMMetric(),
            OraclePSNR(), # PSNRMetric(),
            GradientSharpnessMetric()
        ]
        self.model.compile(optimizer=optimizer, loss=self.loss_fn, metrics=metrics)

        # --- FIX: Pass None for train_ds during global initialization ---
        self.trainer = Trainer(self.cfg, self.model, None, self.val_ds_sequence,
                               self.train_loader,
                               train_steps=self.cfg.hpo.train_steps_per_trial,
                               val_steps=None)

        # 4. RUN OPTUNA
        study = optuna.create_study(
            study_name=self.cfg.hpo.study_name,
            storage=storage_url,
            direction='minimize',
            load_if_exists=True
        )

        study.optimize(
            self._objective,
            n_trials=self.cfg.hpo.n_trials
        )

        logger.info(f"Best Params: {study.best_params}")
        self._save_best_config(study.best_params)

    def _save_best_config(self, best_params):
        # if 'lr' in best_params: self.cfg.train.learning_rate = best_params['lr']
        if 'l_tumor' in best_params: self.cfg.train.lambda_tumor = best_params['l_tumor']
        if 'l_healthy' in best_params: self.cfg.train.lambda_healthy = best_params['l_healthy']
        if 'l_background' in best_params: self.cfg.train.lambda_background = best_params['l_background']
        if 'l_grad' in best_params: self.cfg.train.lambda_grad = best_params['l_grad']
        if 'l_perceptual' in best_params: self.cfg.train.lambda_perceptual = best_params['l_perceptual']
        if 'l_spectral' in best_params: self.cfg.train.lambda_spectral = best_params['l_spectral']
        if 'lr_g' in best_params: self.cfg.train.learning_rate_g = best_params['lr_g']
        if 'lr_d' in best_params: self.cfg.train.learning_rate_d = best_params['lr_d']
        if 'w_l1' in best_params: self.cfg.train.weight_l1 = best_params['w_l1']
        if 'w_perc' in best_params: self.cfg.train.weight_perceptual = best_params['w_perc']
        if 'w_spec' in best_params: self.cfg.train.weight_spectral = best_params['w_spec']
        if 'w_gan' in best_params: self.cfg.train.weight_gan = best_params['w_gan']
        if 'w_fm' in best_params: self.cfg.train.weight_fm = best_params['w_fm']

        # FIX: Restore full RAM capacity for standard training before saving!
        # TODO: de-hardcode
        self.cfg.data.ixi_cache_size = 120
        self.cfg.data.brats_cache_size = 120

        path = self.cfg.hpo.best_params_file
        self.cfg.save(path)
        logger.info(f"Saved optimized config to {path}")

# Cell 18: The Merging Utility

In [ ]:
import glob
import os
import optuna

In [ ]:
def merge_hpo_databases(storage_dir, study_name, output_db="hpo_final.db"):
    """
    Merges multiple SQLite Optuna databases into one to find the global best parameters.
    """
    # 1. Find all DBs
    db_files = glob.glob(os.path.join(storage_dir, "*.db"))
    if not db_files:
        print("No databases found.")
        return

    print(f"Found {len(db_files)} databases. Merging into {output_db}...")

    # 2. Create Target Study
    if output_db.startswith('sqlite:///'):
        raise RuntimeError(f"'output_db' must be a path to file, got {output_db}...")
    target_storage = f"sqlite:///{output_db}"

    # Delete if exists to start fresh merge
    if os.path.exists(output_db):
        os.remove(output_db)

    target_study = optuna.create_study(study_name=study_name, storage=target_storage,
                                       direction='minimize')

    # 3. Smart Merging with Conflict Resolution
    seen_signatures = set()
    total_added = 0
    total_skipped = 0

    # Iterate and Copy Trials
    for db in db_files:
        print(f"Merging {db}...")
        try:
            source_storage = f"sqlite:///{db}"
            source_study = optuna.load_study(study_name=study_name,
                                             storage=source_storage)

            for trial in source_study.trials:
                # Only keep successful trials
                if trial.state != optuna.trial.TrialState.COMPLETE:
                    continue

                # Create a unique, hashable signature from the hyperparameters
                # e.g., (('l_grad', 0.5), ('lr', 0.001), ...)
                param_signature = tuple(sorted(trial.params.items()))

                if param_signature in seen_signatures:
                    total_skipped += 1
                    continue # skip duplicate!

                # Add to set and insert into target database
                seen_signatures.add(param_signature)
                target_study.add_trial(trial)
                total_added += 1
        except Exception as e:
            print(f"Failed to merge {db}: {e}")

    print(f"Merge Complete. Total Trials: {len(target_study.trials)}")
    # print(f"Global Best Params: {target_study.best_params}")
    # print(f"Global Best Value: {target_study.best_value}")
    # print("\n--- Merge Complete ---")
    print(f"Unique Trials Added: {total_added}")
    print(f"Duplicate Trials Skipped: {total_skipped}")
    print(f"Global Best Params: {target_study.best_params}")
    print(f"Global Best Value: {target_study.best_value}")

# Cell 19: HPO Visualization Analysis

In [ ]:
import optuna
import plotly.io as pio

from IPython.display import display, IFrame
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_param_importances
from optuna.visualization import plot_slice
from optuna.visualization import plot_contour
from optuna.visualization import plot_parallel_coordinate

In [ ]:
def analyze_hpo_results(storage_url, study_name="scratchnet", save_dir="reports"):
    """
    Loads a study from the DB and generates Plotly visualizations.
    Saves interactive HTML files to ensure results are viewable even if notebook rendering fails.
    """
    # 1. Setup Environment
    os.makedirs(save_dir, exist_ok=True)

    # Try to set a robust renderer for notebooks
    try:
        # 'iframe' is usually safest for Kaggle/Colab
        pio.renderers.default = "iframe"
    except:
        print(f"Failed to change Plotly default renderer")

    print(f"Loading study '{study_name}' from {storage_url}...")
    try:
        study = optuna.load_study(study_name=study_name, storage=storage_url)
    except Exception as e:
        print(f"Failed to load study: {e}")
        return

    if len(study.trials) == 0:
        print("Study has no trials.")
        return

    print(f"Total Trials: {len(study.trials)}")
    print(f"Best Value (AR-MAE): {study.best_value:.6f}")
    print("Best Params:")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")

    # Helper to save and show
    def process_plot(fig, filename, title):
        fig.update_layout(title=title)

        # Save HTML (Always works, can be downloaded and opened in browser)
        save_path = os.path.join(save_dir, filename)
        fig.write_html(save_path)
        print(f"Saved: {save_path}")

        # Try interactive show (might fail in some envs)
        # try:
        #     fig.show()
        # except Exception:
        #     print("Interactive plot failed to render inline (check saved HTML).")
        # Explicitly display the saved file in an IFrame
        # This bypasses Plotly's internal temp file caching
        try:
            display(IFrame(src=save_path, width="100%", height="500px"))
        except Exception as e:
            print(f"Could not display IFrame: {e}")

    # 1. Optimization History
    # Goal: See if the HPO actually found better models over time or if it flatlined.
    # If the curve goes down, HPO worked
    try:
        fig_hist = plot_optimization_history(study)
        process_plot(fig_hist, "history.html", "1. Optimization History (Convergence)")
    except Exception as e: print(f"Plot failed: {e}")

    # 2. Hyperparameter Importance
    # Goal: Identify which params drive performance.
    # For the paper: "We found that L1 weight and Learning Rate were the most critical..."
    try:
        if len(study.trials) > 1:
            fig_imp = plot_param_importances(study)
            # process_plot(fig, "importance.html", "Hyperparameter Importance")
            process_plot(fig_imp, "importance.html", "2. Hyperparameter Importance")
    except Exception as e:
        print(f"Importance plot failed: {e}")

    # 3. Slice Plot
    # Goal: See the individual relationship of each param to the error.
    # Look for "U-shapes" (optimum in middle) or "Linear slopes" (optimum at edge)
    try:
        fig_slice = plot_slice(study)
        # process_plot(fig, "slices.html", "Parameter Slices")
        process_plot(fig_slice, "slices.html", "3. Individual Parameter Slices")
    except Exception as e: print(f"Slice plot failed: {e}")

    # 4. Contour Plots (Pairwise Interactions)
    # Goal: See how two heavy-hitters interact.
    # Specifically useful for: L1 vs Spectral, or LR vs Batch Size

    # Let's pick the top 3 most important params to plot contours for
    try:
        # Get importance to find top params
        try:
            importance = optuna.importance.get_param_importances(study)
            top_params = list(importance.keys())[:3]
        except:
            # Fallback if importance calc fails (too few trials)
            top_params = list(study.best_params.keys())[:2]

        if len(top_params) >= 2:
            fig_land = plot_contour(study, params=top_params)
            # process_plot(fig, "contour.html", f"Contour Landscape ({top_params})")
            process_plot(fig_land, "contour.html", f"4. Contour Landscape ({top_params})")
    except Exception as e: print(f"Contour plot failed: {e}")

    # 5. Parallel Coordinates
    # Goal: Visualizing the "flow" of high-dimensional configs.
    # Useful to spot clusters of good runs (e.g., "All good runs have High L1 and Low LR")
    try:
        fig_para = plot_parallel_coordinate(study)
        # process_plot(fig, "parallel.html", "High-Dimensional Overview")
        process_plot(fig_para, "parallel.html", "5. Parallel Coordinates (High-Dim View)")
    except Exception as e: print(f"Parallel plot failed: {e}")

    print(f"\n✅ Analysis complete. All interactive plots saved to '{save_dir}/'")

# Cell 20: Extra evaluation metrics

In [ ]:
class SynthSegEvaluator:
    """
    Orchestrates the downstream anatomical validation using SynthSeg.
    Operates on the saved .npy files from the ablation study.
    """
    def __init__(self, config: Config):
        self.results_dir = config.data.results_dir
        self.nifti_dir = (os.path.join(self.results_dir, "nifti"))
        self.seg_dir = (os.path.join(self.results_dir, "synthseg_masks"))

        os.makedirs(self.nifti_dir, exist_ok=True)
        os.makedirs(self.seg_dir, exist_ok=True)

        # FreeSurfer / SynthSeg standard labels
        self.macro_regions = {
            'Ventricles': [4, 43], # Left/Right Lateral Ventricles
            'Deep_GM':[10, 49, 11, 50, 12, 51, 13, 52], # Thalamus, Caudate, Putamen, Pallidum
            'White_Matter': [2, 41], # Cerebral White Matter
            'Cortex': [3, 42] # Cerebral Cortex
        }

    def setup(self):
        # """Clones SynthSeg if not present."""
        # if not os.path.exists("SynthSeg"):
        #     logger.info("Cloning SynthSeg repository...")
        #     # subprocess.run(["git", "clone", "https://github.com/BBillot/SynthSeg.git"], check=True)
        #     subprocess.run(["git", "clone", "https://github.com/ValV/SynthSeg.git"], check=True)
        # else:
        #     logger.info("SynthSeg repository found.")
        """Prepares SynthSeg environment and ensures models/data are downloaded."""
        # 1. Ensure the package is installed
        # (Assuming you already ran %pip install git+https://github.com/ValV/SynthSeg.git)

        # 2. Trigger the model download logic provided by the library itself
        from SynthSeg.cli import get_model_dir

        logger.info("Verifying SynthSeg model weights...")
        try:
            get_model_dir() # This will download to user_cache_dir if missing
        except Exception as e:
            logger.error(f"Failed to download SynthSeg models: {e}")
            raise

        logger.info("SynthSeg environment ready.")

    def convert_to_nifti(self):
        """Converts ablation .npy files to .nii.gz for SynthSeg."""
        # Only convert if NIfTI folder is empty
        nifti_files = glob.glob(os.path.join(self.nifti_dir, "*.nii.gz"))
        if len(nifti_files) > 0:
            logger.info(f"NIfTI files already exist in {self.nifti_dir}. Skipping conversion.")
            return

        logger.info("Converting .npy volumes to .nii.gz...")
        npy_files = glob.glob(os.path.join(self.results_dir, "*.npy"))

        if not npy_files:
            raise FileNotFoundError(f"No .npy files found in {self.results_dir}. Did you run the ablation study?")

        for f in tqdm(npy_files, desc="NIfTI Conversion"):
            vol = np.load(f)
            # SynthSeg is resolution/contrast agnostic. An identity affine is perfectly fine.
            nii = nib.Nifti1Image(vol, np.eye(4))
            out_name = os.path.basename(f).replace('.npy', '.nii.gz')
            nib.save(nii, os.path.join(self.nifti_dir, out_name))

    def run_prediction(self):
        """Executes SynthSeg using the library import with explicit defaults."""
        logger.info("Running SynthSeg prediction (This may take a while)...")

        # A. Clear the main model from GPU to free up VRAM for SynthSeg
        if hasattr(self, 'model'):
            del self.model

        tf.keras.backend.clear_session()
        gc.collect()

        # B. Now run the library call
        from SynthSeg.predict_synthseg import predict
        from SynthSeg.cli import get_model_dir
        import SynthSeg # import the package to find its location
        
        # 1. Locate the package directory dynamically
        synthseg_pkg_path = os.path.dirname(os.path.dirname(SynthSeg.__file__))
        
        # 2. Construct the labels path relative to the package location
        # Based on your find command, the labels are in src/SynthSeg/data/labels_classes_priors/
        labels_path = os.path.join(
            synthseg_pkg_path, 
            'SynthSeg', "data", "labels_classes_priors", 
            "synthseg_segmentation_labels_2.0.npy"
        )
        
        # 3. Locate the model
        model_dir = get_model_dir()
        path_model_seg = os.path.join(model_dir, 'synthseg_2.0.h5')
        
        # if not os.path.exists(labels_path):
        #     raise FileNotFoundError(f"Labels file not found at {labels_path}. Check SynthSeg's data folder.")
        # 4. Verify
        if not os.path.exists(labels_path):
            logger.warning(f"'{labels_path}' NOT FOUND! Falling back to './SynthSeg'...")
            # Fallback: Check if it's in the cloned folder directly
            labels_path = "./SynthSeg/src/SynthSeg/data/labels_classes_priors/synthseg_segmentation_labels_2.0.npy"
            if not os.path.exists(labels_path):
                raise FileNotFoundError(f"Could not locate labels at {labels_path}")
        else:
            logger.info(f"Using '{labels_path}' as labels path.")

        logger.info(f"Running SynthSeg prediction...")

        # These are the values the library expects to see for standard segmentation
        predict(
            path_images=self.nifti_dir,
            path_segmentations=self.seg_dir,
            path_model_segmentation=path_model_seg,
            labels_segmentation=labels_path,
            robust=False,
            fast=True,
            v1=False,
            n_neutral_labels=None, # 19
            labels_denoiser=os.path.join(os.path.dirname(labels_path),
                                         'synthseg_denoiser_labels_2.0.npy'),
            path_posteriors=None,
            path_resampled=None,
            path_volumes=None,
            do_parcellation=False,
            path_model_parcellation=None,
            labels_parcellation=None,
            path_qc_scores=None,
            path_model_qc=None,
            labels_qc=None,
            cropping=None
        )

        logger.info("SynthSeg prediction complete!")

    def _dice_coef(self, mask1: np.ndarray, mask2: np.ndarray) -> float:
        intersection = np.sum(mask1 & mask2)
        vol1 = np.sum(mask1)
        vol2 = np.sum(mask2)
        if vol1 + vol2 == 0: return 1.0
        return float(2.0 * intersection / (vol1 + vol2))

    def calculate_dsc(self):
        """Computes DSC between Ground Truth and Ablation predictions."""
        logger.info("Calculating Anatomical DSC metrics...")

        # Find all Ground Truth segmentations
        gt_files = glob.glob(os.path.join(self.seg_dir, "gt_*.nii.gz"))
        ablations =['baseline', 'spade', 'buffer', 'full', 'ours_full_masked',
                    'monai_3d_ldm', 'monai_2d_ldm', 'monai_vqgan']

        results =[]

        for gt_path in tqdm(gt_files, desc="Scoring Volumes"):
            vol_id = os.path.basename(gt_path).replace('gt_', '').replace('.nii.gz', '')
            gt_seg = nib.load(gt_path).get_fdata()

            for ab in ablations:
                pred_path = os.path.join(self.seg_dir, f"{ab}_{vol_id}.nii.gz")
                if not os.path.exists(pred_path): continue

                pred_seg = nib.load(pred_path).get_fdata()

                for region_name, labels in self.macro_regions.items():
                    # Create binary masks for the macro region
                    gt_mask = np.isin(gt_seg, labels)
                    pred_mask = np.isin(pred_seg, labels)

                    dsc = self._dice_coef(gt_mask, pred_mask)

                    results.append({
                        'Volume_ID': vol_id,
                        'Ablation': ab,
                        'Region': region_name.replace('_', ' '),
                        'DSC': dsc
                    })

        df = pd.DataFrame(results)
        csv_path = os.path.join(self.results_dir, "anatomical_metrics.csv")
        df.to_csv(csv_path, index=False)
        logger.info(f"Anatomical metrics saved to {csv_path}")

In [ ]:
import torch
from scipy import linalg
import torchvision.models.video as video_models


class FrechetEvaluator:
    """
    Computes 3D-FID and FVD (Fréchet Video Distance) to measure 3D structural 
    realism and Z-axis continuity.
    """
    def __init__(self, config: Config):
        self.results_dir = config.data.results_dir
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        logger.info(f"Initializing 3D Feature Extractors on {self.device}...")
        
        # 1. FVD Extractor: I3D (Inception 3D) pre-trained on Kinetics-400
        # Standard for measuring sequential (Z-axis) smoothness
        self.i3d = video_models.mvit_v1_b(weights=video_models.MViT_V1_B_Weights.DEFAULT)
        self.i3d.head = torch.nn.Identity() # Strip classification head to get raw features
        self.i3d.eval().to(self.device)
        
        # 2. 3D-FID Extractor: 3D ResNet
        # Ideally MedicalNet, but TorchVision's R3D_18 is natively available and highly effective
        self.r3d = video_models.r3d_18(weights=video_models.R3D_18_Weights.DEFAULT)
        self.r3d.fc = torch.nn.Identity()
        self.r3d.eval().to(self.device)

    def _extract_features(self, vol: np.ndarray, model: torch.nn.Module,
                          target_size=(16, 224, 224)) -> np.ndarray:
        # """
        # Converts a (Z, H, W) volume to (B, C, T, H, W) and extracts 3D features.
        # """
        """
        Iterates over the full (Z, H, W) volume using a sliding window, 
        extracting 3D features for every 16-slice chunk.
        """
        # 1. Resize/Crop to standard 3D input (e.g., 16 frames/slices, 112x112 spatial)
        # We sample a block from the center to represent the core brain structure
        Z, H, W = vol.shape
        # z_start = max(0, Z // 2 - 8)
        chunk_size = 16
        stride = 8  # 50% overlap for dense Z-axis coverage
        features_list =[]

        for z_start in range(0, max(1, Z - chunk_size + 1), stride):
            # Extract 16 slices
            vol_clip = vol[z_start:z_start + 16]
            vol_clip = vol[z_start:z_start+chunk_size]

            # if vol_clip.shape[0] < 16:
            #     # Pad if too small
            #     pad_width = 16 - vol_clip.shape[0]
            #     vol_clip = np.pad(vol_clip, ((0, pad_width), (0, 0), (0, 0)), mode='edge')
            # Pad if we hit the end of the volume and it's smaller than 16 slices
            if vol_clip.shape[0] < chunk_size:
                pad_width = chunk_size - vol_clip.shape[0]
                vol_clip = np.pad(vol_clip, ((0, pad_width), (0, 0), (0, 0)), mode='edge')

            # Skip chunks that are pure background (air) to avoid polluting the distribution
            if np.mean(vol_clip) < 0.01:
                continue

            # 2. Format for PyTorch Video Models: (Batch, Channels, Time/Z, Height, Width)
            # Convert grayscale to RGB by repeating channels
            vol_tensor = torch.tensor(vol_clip, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            vol_tensor = vol_tensor.repeat(1, 3, 1, 1, 1) # (1, 3, 16, H, W)

            # Interpolate spatial dimensions to 112x112 (standard for R3D/I3D)
            # vol_tensor = torch.nn.functional.interpolate(vol_tensor, size=(16, 112, 112), mode='trilinear')
            # Interpolate spatial dimensions to the exact target size required by the model
            vol_tensor = torch.nn.functional.interpolate(vol_tensor, size=target_size, mode='trilinear')

            # Normalize to ImageNet/Kinetics stats
            vol_tensor = (vol_tensor - 0.45) / 0.225

            with torch.no_grad():
                features = model(vol_tensor.to(self.device))

        return features.cpu().numpy().flatten()

    def _calculate_frechet_distance(self, mu1, sigma1, mu2, sigma2, eps=1e-6):
        """Numpy implementation of the Fréchet Distance."""
        mu1 = np.atleast_1d(mu1)
        mu2 = np.atleast_1d(mu2)
        sigma1 = np.atleast_2d(sigma1)
        sigma2 = np.atleast_2d(sigma2)

        diff = mu1 - mu2
        
        # Product of covariances
        covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
        if not np.isfinite(covmean).all():
            offset = np.eye(sigma1.shape[0]) * eps
            covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))
            
        if np.iscomplexobj(covmean):
            covmean = covmean.real

        tr_covmean = np.trace(covmean)
        return diff.dot(diff) + np.trace(sigma1) + np.trace(sigma2) - 2 * tr_covmean

    def evaluate(self):
        logger.info("Calculating 3D-FID and FVD...")
        
        gt_files = glob.glob(os.path.join(self.results_dir, "gt_*.npy"))
        if not gt_files:
            logger.error(f"No Ground Truth (gt_*.npy) files found in {self.results_dir}. Aborting.")
            return
        ablations = ['baseline', 'spade', 'buffer', 'full', 'ours_full_masked',
                     'monai_3d_ldm', 'monai_2d_ldm', 'monai_vqgan']
        
        # Dictionaries to store feature vectors
        features_r3d = {ab:[] for ab in ablations}
        features_r3d['gt'] = []
        
        features_i3d = {ab:[] for ab in ablations}
        features_i3d['gt'] =[]
        
        # 1. Extract Features
        for gt_path in tqdm(gt_files, desc="Extracting 3D Features"):
            vol_id = os.path.basename(gt_path).replace('gt_', '').replace('.npy', '')
            gt_vol = np.load(gt_path)

            # Extract GT features
            # features_r3d['gt'].append(self._extract_features(gt_vol, self.r3d))
            # features_i3d['gt'].append(self._extract_features(gt_vol, self.i3d))
            # Extract GT features (R3D expects 112x112, MViT expects 224x224)
            # features_r3d['gt'].append(self._extract_features(
            #     gt_vol, self.r3d, target_size=(16, 112, 112)))
            # features_i3d['gt'].append(self._extract_features(
            #     gt_vol, self.i3d, target_size=(16, 224, 224)))
            # Use .extend() because _extract_features now returns a list of chunks
            features_r3d['gt'].extend(self._extract_features(
                gt_vol, self.r3d, target_size=(16, 112, 112)))
            features_i3d['gt'].extend(self._extract_features(
                gt_vol, self.i3d, target_size=(16, 224, 224)))

            for ab in ablations:
                pred_path = os.path.join(self.results_dir, f"{ab}_{vol_id}.npy")
                if not os.path.exists(pred_path): continue

                pred_vol = np.load(pred_path)
                # features_r3d[ab].append(self._extract_features(pred_vol, self.r3d))
                # features_i3d[ab].append(self._extract_features(pred_vol, self.i3d))
                # features_r3d[ab].append(self._extract_features(
                #     pred_vol, self.r3d, target_size=(16, 112, 112)))
                # features_i3d[ab].append(self._extract_features(
                #     pred_vol, self.i3d, target_size=(16, 224, 224)))
                features_r3d[ab].extend(self._extract_features(
                    pred_vol, self.r3d, target_size=(16, 112, 112)))
                features_i3d[ab].extend(self._extract_features(
                    pred_vol, self.i3d, target_size=(16, 224, 224)))

        # 2. Compute Statistics and Fréchet Distance
        results =[]
        
        # Ground Truth Stats
        mu_gt_r3d, sig_gt_r3d = np.mean(features_r3d['gt'], axis=0), np.cov(features_r3d['gt'], rowvar=False)
        mu_gt_i3d, sig_gt_i3d = np.mean(features_i3d['gt'], axis=0), np.cov(features_i3d['gt'], rowvar=False)
        
        for ab in ablations:
            if not features_r3d[ab]: continue
                
            mu_ab_r3d, sig_ab_r3d = np.mean(features_r3d[ab], axis=0), np.cov(features_r3d[ab], rowvar=False)
            mu_ab_i3d, sig_ab_i3d = np.mean(features_i3d[ab], axis=0), np.cov(features_i3d[ab], rowvar=False)
            
            fid_3d = self._calculate_frechet_distance(mu_gt_r3d, sig_gt_r3d, mu_ab_r3d, sig_ab_r3d)
            fvd = self._calculate_frechet_distance(mu_gt_i3d, sig_gt_i3d, mu_ab_i3d, sig_ab_i3d)
            
            results.append({
                'Ablation': ab,
                '3D-FID': fid_3d,
                'FVD': fvd
            })

        df = pd.DataFrame(results)
        csv_path = os.path.join(self.results_dir, "frechet_metrics.csv")
        df.to_csv(csv_path, index=False)
        logger.info(f"Fréchet metrics saved to {csv_path}")
        print(df.to_string(index=False))

In [ ]:
class FastSurferEvaluator:
    """
    Orchestrates downstream anatomical validation using FastSurfer.
    FastSurfer is a pure PyTorch pipeline that segments T1 MRIs into 95 classes.
    """
    def __init__(self, config: Config):
        self.results_dir = config.data.results_dir
        self.nifti_dir = os.path.join(self.results_dir, "nifti")
        self.seg_dir = os.path.join(self.results_dir, "fastsurfer_masks")

        os.makedirs(self.nifti_dir, exist_ok=True)
        os.makedirs(self.seg_dir, exist_ok=True)

        # FreeSurfer DKT Atlas Labels
        self.macro_regions = {
            'Ventricles': [4, 43], # Left/Right Lateral Ventricles
            'Deep_GM':[10, 49, 11, 50, 12, 51, 13, 52], # Thalamus, Caudate, Putamen, Pallidum
            'White_Matter': [2, 41], # Cerebral White Matter
            'Cortex': list(range(1000, 1036)) + list(range(2000, 2036)) # DKT Cortical regions
        }

    def setup(self):
        """Clones FastSurfer and installs PyTorch dependencies."""
        if not os.path.exists("FastSurfer"):
            logger.info("Cloning FastSurfer...")
            subprocess.run(["git", "clone", # "--branch", "v2.4.2",
                            "https://github.com/Deep-MI/FastSurfer.git"],
                           check=True)

            # logger.info("Installing FastSurfer dependencies...")
            # subprocess.run(["pip", "install", "-r", "FastSurfer/requirements.txt"], check=True)
        else:
            logger.info("FastSurfer repository found.")

        # logger.info("Installing FastSurfer...")
        # subprocess.run(["pip", "install", "-e", "FastSurfer"], check=True)
        logger.info("Installing FastSurfer dependencies...")
        subprocess.run(["pip", "install", "yacs"], check=True)

    def convert_to_nifti(self):
        """Converts ablation .npy files to .nii.gz for FastSurfer."""
        nifti_files = sorted(glob.glob(os.path.join(self.nifti_dir, "*.nii.gz")))
        if len(nifti_files) > 0:
            logger.info(f"NIfTI files already exist in {self.nifti_dir}. Skipping conversion.")
            return

        logger.info("Converting .npy volumes to .nii.gz...")
        npy_files = sorted(glob.glob(os.path.join(self.results_dir, "*.npy")))
        
        if not npy_files:
            raise FileNotFoundError(f"No .npy files found in {self.results_dir}. Run ablation mode first.")
            
        # for f in tqdm(npy_files, desc="NIfTI Conversion"):
        #     vol = np.load(f)
        #     # FastSurfer handles arbitrary resolutions via its conform module.
        #     nii = nib.Nifti1Image(vol, np.eye(4))
        #     out_name = os.path.basename(f).replace('.npy', '.nii.gz')
        #     nib.save(nii, os.path.join(self.nifti_dir, out_name))
        for f in tqdm(npy_files, desc="NIfTI Conversion"):
            vol = np.load(f) # shape is currently (Z, Y, X) and range is [0.0, 1.0]

            # 1. Restore Canonical RAS+ Orientation (X, Y, Z)
            vol_xyz = np.transpose(vol, (2, 1, 0))

            # 2. Restore MRI Intensity Range (0-255)
            # FastSurfer casts to UCHAR. If we pass [0, 1], it becomes a binary silhouette!
            vol_scaled = np.clip(vol_xyz * 255.0, 0, 255).astype(np.uint8)

            # 3. Save with Identity Affine
            nii = nib.Nifti1Image(vol_scaled, np.eye(4))
            out_name = os.path.basename(f).replace('.npy', '.nii.gz')
            nib.save(nii, os.path.join(self.nifti_dir, out_name))

    def run_prediction(self):
        """Executes FastSurfer segmentation."""
        logger.info("Running FastSurfer prediction...")
        nifti_files = sorted(glob.glob(os.path.join(self.nifti_dir, "*.nii.gz")))

        env = os.environ.copy()

        # Ensure FastSurfer's internal imports resolve correctly
        env["PYTHONPATH"] = os.path.abspath("FastSurfer") + ":" + env.get("PYTHONPATH", "")
        # The bash script relies on this environment variable
        env["FASTSURFER_HOME"] = os.path.abspath("FastSurfer")

        # Create a dummy FreeSurfer license to bypass the bash script's strict checks
        license_path = os.path.abspath("dummy_fs_license.txt")
        if not os.path.exists(license_path):
            with open(license_path, "w") as f:
                f.write("dummy@dummy.com\n12345\n12345\n12345\n")

        for f in tqdm(nifti_files, desc="FastSurfer Inference"):
            sid = os.path.basename(f).replace('.nii.gz', '')
            out_file = os.path.join(self.seg_dir, sid, "mri", "aparc.DKTatlas+aseg.deep.mgz")

            if os.path.exists(out_file):
                continue

            cmd =[
                "bash", "FastSurfer/run_fastsurfer.sh",
                "--allow_root",
                "--t1", os.path.abspath(f),
                "--seg_only",                           # skip surface reconstruction (no license needed)
                "--sd", os.path.abspath(self.seg_dir),  # subject directory root
                "--sid", sid,                           # subject ID
                "--device", "cuda",                     # force GPU
                "--batch", "16",                        # massive speedup for the neural network
                "--threads", "4"                        # speedup for CPU conforming steps
            ]
            
            # result = subprocess.run(cmd, env=env, capture_output=True, text=True)
            # if result.returncode != 0:
            #     logger.error(f"FastSurfer Failed for {sid}:\n{result.stderr}")

            result = subprocess.run(cmd, env=env, capture_output=True, text=True)
            
            if result.returncode != 0:
                # Log BOTH stdout and stderr to prevent silent failures
                logger.error(f"FastSurfer Failed for {sid}:"
                             f"\n--- STDERR ---\n{result.stderr}"
                             f"\n--- STDOUT ---\n{result.stdout}")

    def _dice_coef(self, mask1: np.ndarray, mask2: np.ndarray) -> float:
        intersection = np.sum(mask1 & mask2)
        vol1 = np.sum(mask1)
        vol2 = np.sum(mask2)
        if vol1 + vol2 == 0: return 1.0
        return float(2.0 * intersection / (vol1 + vol2))

    def calculate_dsc(self):
        """Computes DSC between Ground Truth and Ablation predictions."""
        logger.info("Calculating Anatomical DSC metrics...")
        
        # Find all Ground Truth segmentations
        # gt_sids =[os.path.basename(f).replace('.nii.gz', '') for f in glob.glob(os.path.join(self.nifti_dir, "gt_*.nii.gz"))]
        gt_sids =[os.path.basename(f).replace('.nii.gz', '') for f in glob.glob(os.path.join(self.nifti_dir, "gt_*.nii.gz"))]
        ablations = ['baseline', 'spade', 'buffer', 'full', 'ours_full_masked',
                     'monai_3d_ldm', 'monai_2d_ldm', 'monai_vqgan']
        
        results =[]
        
        for gt_sid in tqdm(gt_sids, desc="Scoring Volumes"):
            vol_id = gt_sid.replace('gt_', '')
            # FastSurfer saves outputs in .mgz format
            gt_path = os.path.join(self.seg_dir, gt_sid, "mri",
                                   "aparc.DKTatlas+aseg.deep.mgz")
            
            if not os.path.exists(gt_path):
                continue
                
            gt_seg = nib.load(gt_path).get_fdata()
            
            for ab in ablations:
                pred_sid = f"{ab}_{vol_id}"
                pred_path = os.path.join(self.seg_dir, pred_sid, "mri",
                                         "aparc.DKTatlas+aseg.deep.mgz")
                
                if not os.path.exists(pred_path): 
                    continue
                    
                pred_seg = nib.load(pred_path).get_fdata()
                
                # FastSurfer automatically conforms images to 256x256x256 1mm isotropic. 
                # The arrays match perfectly because they underwent the exact same spatial transformations
                for region_name, labels in self.macro_regions.items():
                    gt_mask = np.isin(gt_seg, labels)
                    pred_mask = np.isin(pred_seg, labels)
                    
                    dsc = self._dice_coef(gt_mask, pred_mask)
                    
                    results.append({
                        'Volume_ID': vol_id,
                        'Ablation': ab,
                        'Region': region_name.replace('_', ' '),
                        'DSC': dsc
                    })
                    
        df = pd.DataFrame(results)
        csv_path = os.path.join(self.results_dir, "anatomical_metrics.csv")
        df.to_csv(csv_path, index=False)
        logger.info(f"Anatomical metrics saved to {csv_path}")

In [ ]:
class MonaiSotaEvaluator:
    """
    Evaluates MONAI Generative SOTA models and Our Full Model on Masked Inpainting.
    """
    def __init__(self, config: Config):
        self.cfg = config
        self.results_dir = config.data.results_dir
        self.bundle_dir = os.path.join(self.results_dir, "monai_bundles")
        os.makedirs(self.bundle_dir, exist_ok=True)
        
        self.device = None
        self.autoencoder_3d = None
        self.unet_3d = None
        self.scheduler_3d = None
        
        self.unet_2d = None
        self.scheduler_2d = None

    def setup(self):
        logger.info("Setting up MONAI Generative ecosystem...")
        try:
            import monai
            import generative
        except ImportError:
            logger.info("Force-installing MONAI without touching Kaggle's CUDA/Torch deps...")
            subprocess.run([
                "pip", "install", "--no-deps", "monai", "monai-generative", "einops"
            ], check=True)
            
        import torch
        from monai.bundle import download
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # 1. Download 3D LDM Bundle
        self.bundle_3d = "brain_image_synthesis_latent_diffusion_model"
        if not os.path.exists(os.path.join(self.bundle_dir, self.bundle_3d)):
            logger.info("Downloading MONAI 3D LDM Bundle...")
            download(name=self.bundle_3d, bundle_dir=self.bundle_dir)
            
        # 2. Download 2D Diffusion Bundle (BraTS Axial)
        self.bundle_2d = "brats_mri_axial_slices_generative_diffusion"
        if not os.path.exists(os.path.join(self.bundle_dir, self.bundle_2d)):
            logger.info("Downloading MONAI 2D Diffusion Bundle...")
            download(name=self.bundle_2d, bundle_dir=self.bundle_dir)
            
        logger.info("MONAI SOTA bundles ready.")

    def _load_3d_models(self):
        if self.autoencoder_3d is not None: return
        import torch
        from monai.bundle import ConfigParser
        from generative.networks.schedulers import DDIMScheduler
        
        logger.info("Loading 3D LDM weights into VRAM...")
        bundle_path = os.path.join(self.bundle_dir, self.bundle_3d)
        parser = ConfigParser()
        parser.read_config(os.path.join(bundle_path, "configs", "inference.json"))
        
        try:
            self.autoencoder_3d = parser.get_parsed_content("autoencoder_def")
            self.unet_3d = parser.get_parsed_content("diffusion_def")
        except:
            self.autoencoder_3d = parser.get_parsed_content("autoencoder")
            self.unet_3d = parser.get_parsed_content("diffusion")
            
        self.scheduler_3d = DDIMScheduler(num_train_timesteps=1000, schedule="scaled_linear_beta", beta_start=0.0015, beta_end=0.0195, clip_sample=False)
        
        models_dir = os.path.join(bundle_path, "models")
        ae_sd = torch.load(os.path.join(models_dir, "autoencoder.pt"), map_location=self.device)
        
        unet_weights =[f for f in glob.glob(os.path.join(models_dir, "*.pt")) if "autoencoder" not in f][0]
        unet_sd = torch.load(unet_weights, map_location=self.device)
        
        translated_ae_sd = {k.replace(".conv.conv.", ".postconv.conv.") if "decoder." in k else k: v for k, v in ae_sd.items()}
        translated_unet_sd = {k.replace(".to_out.0.", ".out_proj."): v for k, v in unet_sd.items()}
        
        self.autoencoder_3d.load_state_dict(translated_ae_sd, strict=False)
        self.unet_3d.load_state_dict(translated_unet_sd, strict=False)
        self.autoencoder_3d.eval().to(self.device)
        self.unet_3d.eval().to(self.device)

    def _load_2d_models(self):
        if self.unet_2d is not None: return
        import torch
        from monai.bundle import ConfigParser
        from generative.networks.schedulers import DDIMScheduler
        
        logger.info("Loading 2D Diffusion weights into VRAM...")
        bundle_path = os.path.join(self.bundle_dir, self.bundle_2d)
        parser = ConfigParser()
        parser.read_config(os.path.join(bundle_path, "configs", "inference.json"))
        
        try:
            self.unet_2d = parser.get_parsed_content("network_def")
        except:
            self.unet_2d = parser.get_parsed_content("network")
            
        self.scheduler_2d = DDIMScheduler(num_train_timesteps=1000, schedule="linear_beta", beta_start=0.0015, beta_end=0.0195, clip_sample=False)
        
        models_dir = os.path.join(bundle_path, "models")
        unet_weights = glob.glob(os.path.join(models_dir, "*.pt"))[0]
        self.unet_2d.load_state_dict(torch.load(unet_weights, map_location=self.device), strict=False)
        self.unet_2d.eval().to(self.device)

    def _repaint_3d_latent(self, vol_tensor, mask_tensor, num_inference_steps=50, repaint_jumps=2):
        import torch
        import torch.nn.functional as F
        with torch.no_grad():
            latent_gt = self.autoencoder_3d.encode_stage_2_inputs(vol_tensor)
            
        latent_mask = F.interpolate(mask_tensor, size=latent_gt.shape[2:], mode='nearest')
        latent_pred = torch.randn_like(latent_gt).to(self.device)
        
        dummy_cond = torch.zeros((1, 4, *latent_gt.shape[2:]), dtype=latent_gt.dtype, device=self.device)
        dummy_context = torch.zeros((1, 1, 4), dtype=latent_gt.dtype, device=self.device)
        
        self.scheduler_3d.set_timesteps(num_inference_steps)
        
        with torch.cuda.amp.autocast():
            for t in self.scheduler_3d.timesteps:
                for jump in range(repaint_jumps):
                    with torch.no_grad():
                        unet_input = torch.cat([latent_pred, dummy_cond], dim=1)
                        step_out = self.unet_3d(x=unet_input, timesteps=torch.tensor([t]).to(self.device), context=dummy_context)
                        
                    noise_pred = step_out[0] if isinstance(step_out, tuple) else step_out
                    step_res = self.scheduler_3d.step(noise_pred, t, latent_pred)
                    latent_pred = step_res[0] if isinstance(step_res, tuple) else step_res
                    
                    noise = torch.randn_like(latent_gt).to(self.device)
                    known_latent_t = self.scheduler_3d.add_noise(latent_gt, noise, torch.tensor([t], dtype=torch.long, device=self.device))
                    latent_pred = known_latent_t * (1.0 - latent_mask) + latent_pred * latent_mask
                    
        with torch.no_grad():
            return self.autoencoder_3d.decode_stage_2_outputs(latent_pred)

    def _repaint_2d_pixel(self, slice_tensor, mask_tensor, num_inference_steps=50):
        import torch
        # 2D DDPM operates in pixel space (no autoencoder)
        img_pred = torch.randn_like(slice_tensor).to(self.device)
        self.scheduler_2d.set_timesteps(num_inference_steps)

        batch_size = slice_tensor.shape[0]

        with torch.cuda.amp.autocast():
            for t in self.scheduler_2d.timesteps:
                # Broadcast timestep to match batch size
                t_tensor = torch.full((batch_size,), t, dtype=torch.long, device=self.device)

                with torch.no_grad():
                    # noise_pred = self.unet_2d(x=img_pred, timesteps=torch.tensor([t]).to(self.device))
                    # FIX: Pass the broadcasted t_tensor to the UNet, not the scalar!
                    noise_pred = self.unet_2d(x=img_pred, timesteps=t_tensor)

                noise_pred = noise_pred[0] if isinstance(noise_pred, tuple) else noise_pred
                step_res = self.scheduler_2d.step(noise_pred, t, img_pred)
                img_pred = step_res[0] if isinstance(step_res, tuple) else step_res
                
                noise = torch.randn_like(slice_tensor).to(self.device)
                # known_img_t = self.scheduler_2d.add_noise(
                #     slice_tensor, noise, torch.tensor([t], dtype=torch.long, device=self.device)
                # )
                known_img_t = self.scheduler_2d.add_noise(
                    slice_tensor, noise, t_tensor
                )
                img_pred = known_img_t * (1.0 - mask_tensor) + img_pred * mask_tensor
                
        return img_pred

    def evaluate_sota_models(self, ixi_files: list, brats_pool: list, num_volumes: int = 10):
        import torch
        import torch.nn.functional as F
        
        test_files = ixi_files[:num_volumes]

        # Pre-generate and save shared masks and GT for all SOTA/Ours comparisons
        logger.info("Pre-generating shared evaluation masks...")
        eval_data =[]
        for f in test_files:
            vol_id = os.path.basename(f).replace('.nii.gz', '').replace('.nii', '')
            vol_obj = VolumeLoader.load(f)
            if not vol_obj: continue

            center = vol_obj.t1.shape[0] // 2
            mask_np = VisualizationSuite._sample_real_brats_mask(
                vol_obj.t1, brats_pool, target_z=center
            )

            # Ensure tumor is actually in the evaluation window
            rollout_span = 5 if self.cfg.run_type == 'Interactive' else 20
            eval_window_mask = mask_np[center - rollout_span : center + rollout_span]

            if np.sum(eval_window_mask) < 10: 
                logger.warning(f"Skipping {vol_id}: Tumor injection failed or missed evaluation window.")
                continue

            # Save GT and Mask so FrechetEvaluator and plots can use them
            np.save(os.path.join(self.results_dir, f"gt_{vol_id}.npy"), vol_obj.t1)
            np.save(os.path.join(self.results_dir, f"mask_{vol_id}.npy"), mask_np)

            eval_data.append((vol_id, vol_obj, mask_np))

        if not eval_data:
            logger.error("No valid volumes could be prepared for SOTA evaluation!")
            return

        # --- 1. OUR FULL MODEL (MASKED) ---
        logger.info("Running Ours (Full) in Masked Mode...")
        model_ours = ModelBuilder.build(self.cfg)
        weights_path = os.path.join(self.cfg.data.weights_dir, "model_full.keras")
        if not os.path.exists(weights_path):
            weights_path = os.path.join(self.cfg.data.results_dir, "model_full.keras")

        # model_ours.load_weights(weights_path)
        try:
            model_ours.load_weights(weights_path)
        except Exception as e:
            logger.error(f"Could not load ours_full weights: {e}")

        reconstructor = VolumeReconstructor(model_ours, self.cfg)
        reconstructor.cfg.batch_mode = True
        prof_ours = InferenceProfiler("ours_full_masked", self.results_dir)

        # for f in tqdm(test_files, desc="Ours Masked"):
        #     vol_id = os.path.basename(f).replace('.nii.gz', '').replace('.nii', '')
        #     out_path = os.path.join(self.results_dir, f"ours_full_masked_{vol_id}.npy")
        #     if os.path.exists(out_path): continue

        #     vol_obj = VolumeLoader.load(f)
        #     mask_np = VisualizationSuite._sample_real_brats_mask(vol_obj.t1, brats_pool)
        #     if np.sum(mask_np) < 10: continue
        for vol_id, vol_obj, mask_np in tqdm(eval_data, desc="Ours Masked"):
            out_path = os.path.join(self.results_dir, f"ours_full_masked_{vol_id}.npy")
            if os.path.exists(out_path): continue
            
            prof_ours.start()
            recon_vol = reconstructor.autoregressive_restore(vol_obj.t1, 0, vol_obj.t1.shape[0], 'forward', mask_volume=mask_np)
            prof_ours.stop_and_log(vol_id)
            np.save(out_path, recon_vol)

        del model_ours
        tf.keras.backend.clear_session()
        gc.collect()

        # --- 2. MONAI 3D LDM & 3D VQ-GAN ---
        self._load_3d_models()
        prof_3d_ldm = InferenceProfiler("monai_3d_ldm", self.results_dir)
        prof_vqgan = InferenceProfiler("monai_vqgan", self.results_dir)
        
        # for f in tqdm(test_files, desc="MONAI 3D Inference"):
        #     vol_id = os.path.basename(f).replace('.nii.gz', '').replace('.nii', '')
        #     out_ldm = os.path.join(self.results_dir, f"monai_3d_ldm_{vol_id}.npy")
        #     out_vq = os.path.join(self.results_dir, f"monai_vqgan_{vol_id}.npy")

        #     vol_obj = VolumeLoader.load(f)
        #     mask_np = VisualizationSuite._sample_real_brats_mask(vol_obj.t1, brats_pool)
        #     if np.sum(mask_np) < 10: continue
        for vol_id, vol_obj, mask_np in tqdm(eval_data, desc="MONAI 3D Inference"):
            out_ldm = os.path.join(self.results_dir, f"monai_3d_ldm_{vol_id}.npy")
            out_vq = os.path.join(self.results_dir, f"monai_vqgan_{vol_id}.npy")

            if os.path.exists(out_ldm) and os.path.exists(out_vq): continue

            vol_np = (vol_obj.t1 * 2.0) - 1.0 
            vol_t = torch.tensor(vol_np, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(self.device)
            mask_t = torch.tensor(mask_np, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(self.device)

            pad_z, pad_y, pad_x = max(0, 128 - vol_t.shape[2]), max(0, 160 - vol_t.shape[3]), max(0, 160 - vol_t.shape[4])
            vol_t_pad = F.pad(vol_t, (0, pad_x, 0, pad_y, 0, pad_z), mode='constant', value=-1.0)
            mask_t_pad = F.pad(mask_t, (0, pad_x, 0, pad_y, 0, pad_z), mode='constant', value=0.0)

            # VQ-GAN Inference (Upper Bound)
            if not os.path.exists(out_vq):
                prof_vqgan.start()
                with torch.no_grad():
                    latent = self.autoencoder_3d.encode_stage_2_inputs(vol_t_pad)
                    pred_vq_pad = self.autoencoder_3d.decode_stage_2_outputs(latent)
                prof_vqgan.stop_and_log(vol_id)

                pred_vq = pred_vq_pad[:, :, :vol_t.shape[2], :vol_t.shape[3], :vol_t.shape[4]]
                pred_vq_np = np.clip((pred_vq.squeeze().cpu().numpy() + 1.0) / 2.0, 0.0, 1.0)
                np.save(out_vq, (pred_vq_np * mask_np) + (vol_obj.t1 * (1.0 - mask_np)))

            # 3D LDM Inference
            if not os.path.exists(out_ldm):
                prof_3d_ldm.start()
                pred_ldm_pad = self._repaint_3d_latent(vol_t_pad, mask_t_pad, num_inference_steps=50)
                prof_3d_ldm.stop_and_log(vol_id)

                pred_ldm = pred_ldm_pad[:, :, :vol_t.shape[2], :vol_t.shape[3], :vol_t.shape[4]]
                pred_ldm_np = np.clip((pred_ldm.squeeze().cpu().numpy() + 1.0) / 2.0, 0.0, 1.0)
                np.save(out_ldm, (pred_ldm_np * mask_np) + (vol_obj.t1 * (1.0 - mask_np)))

        # Clear VRAM
        del self.autoencoder_3d, self.unet_3d
        self.autoencoder_3d = None
        torch.cuda.empty_cache()
        gc.collect()

        # --- 3. MONAI 2D LDM ---
        self._load_2d_models()
        prof_2d_ldm = InferenceProfiler("monai_2d_ldm", self.results_dir)
        
        # for f in tqdm(test_files, desc="MONAI 2D Inference"):
        #     vol_id = os.path.basename(f).replace('.nii.gz', '').replace('.nii', '')
        #     out_2d = os.path.join(self.results_dir, f"monai_2d_ldm_{vol_id}.npy")
        #     if os.path.exists(out_2d): continue
            
        #     vol_obj = VolumeLoader.load(f)
        #     mask_np = VisualizationSuite._sample_real_brats_mask(vol_obj.t1, brats_pool)
        #     if np.sum(mask_np) < 10: continue
        for vol_id, vol_obj, mask_np in tqdm(eval_data, desc="MONAI 2D Inference"):
            out_2d = os.path.join(self.results_dir, f"monai_2d_ldm_{vol_id}.npy")
            if os.path.exists(out_2d): continue

            vol_np = (vol_obj.t1 * 2.0) - 1.0 
            # pred_vol_np = np.zeros_like(vol_np)
            pred_vol_np = vol_np.copy() # default to GT

            prof_2d_ldm.start()

            # 2D models expect 240x240, we pad slices individually
            # for z in range(vol_np.shape[0]):
            #     if np.sum(mask_np[z]) == 0:
            #         pred_vol_np[z] = vol_np[z]
            #         continue

            #     slice_t = torch.tensor(vol_np[z], dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(self.device)
            #     mask_t = torch.tensor(mask_np[z], dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(self.device)

            #     pad_y, pad_x = max(0, 240 - slice_t.shape[2]), max(0, 240 - slice_t.shape[3])
            #     slice_pad = F.pad(slice_t, (0, pad_x, 0, pad_y), mode='constant', value=-1.0)
            #     mask_pad = F.pad(mask_t, (0, pad_x, 0, pad_y), mode='constant', value=0.0)

            #     pred_pad = self._repaint_2d_pixel(slice_pad, mask_pad, num_inference_steps=50)
            #     pred_vol_np[z] = pred_pad[0, 0, :slice_t.shape[2], :slice_t.shape[3]].cpu().numpy()

            # Find all slices that actually have a tumor mask to inpaint
            active_z = [z for z in range(vol_np.shape[0]) if np.sum(mask_np[z]) > 0]

            if active_z:
                # # Stack them into a single batch: (B, 1, H, W)
                # slices_t = torch.tensor(vol_np[active_z], dtype=torch.float32).unsqueeze(1).to(self.device)
                # masks_t = torch.tensor(mask_np[active_z], dtype=torch.float32).unsqueeze(1).to(self.device)

                # # 2D models expect 240x240, pad the whole batch
                # pad_y, pad_x = max(0, 240 - slices_t.shape[2]), max(0, 240 - slices_t.shape[3])
                # slices_pad = F.pad(slices_t, (0, pad_x, 0, pad_y), mode='constant', value=-1.0)
                # masks_pad = F.pad(masks_t, (0, pad_x, 0, pad_y), mode='constant', value=0.0)

                # # Run RePaint ONCE for the entire batch
                # preds_pad = self._repaint_2d_pixel(slices_pad, masks_pad, num_inference_steps=50)

                # # Unpack the batch back into the volume
                # for i, z in enumerate(active_z):
                #     pred_vol_np[z] = preds_pad[i, 0, :slices_t.shape[2], :slices_t.shape[3]].cpu().numpy()

                # Process in safe mini-batches to prevent 50+ GB VRAM explosions
                mini_batch_size = 4

                for b_start in range(0, len(active_z), mini_batch_size):
                    chunk_z = active_z[b_start : b_start + mini_batch_size]

                    # Stack them into a mini-batch: (B, 1, H, W)
                    slices_t = torch.tensor(vol_np[chunk_z], dtype=torch.float32).unsqueeze(1).to(self.device)
                    masks_t = torch.tensor(mask_np[chunk_z], dtype=torch.float32).unsqueeze(1).to(self.device)

                    # 2D models expect 240x240, pad the mini-batch
                    pad_y, pad_x = max(0, 240 - slices_t.shape[2]), max(0, 240 - slices_t.shape[3])
                    slices_pad = F.pad(slices_t, (0, pad_x, 0, pad_y), mode='constant', value=-1.0)
                    masks_pad = F.pad(masks_t, (0, pad_x, 0, pad_y), mode='constant', value=0.0)

                    # Run RePaint for the mini-batch
                    preds_pad = self._repaint_2d_pixel(slices_pad, masks_pad, num_inference_steps=50)

                    # Unpack the mini-batch back into the volume
                    for i, z in enumerate(chunk_z):
                        pred_vol_np[z] = preds_pad[i, 0, :slices_t.shape[2], :slices_t.shape[3]].cpu().numpy()

                    # Clear VRAM after each mini-batch
                    torch.cuda.empty_cache()

            prof_2d_ldm.stop_and_log(vol_id)

            pred_final_np = np.clip((pred_vol_np + 1.0) / 2.0, 0.0, 1.0)
            np.save(out_2d, (pred_final_np * mask_np) + (vol_obj.t1 * (1.0 - mask_np)))

# Run Pipeline

In [ ]:
# %mv -v kaggle-brats-results.zip kaggle-brats-results-$(date +%Y%m%d-%H%M%S).zip

In [ ]:
# !zip -r -9 kaggle-brats-results.zip ablation_results paper_figures memory_telemetry.csv

In [ ]:
# from IPython.display import FileLink


# FileLink('kaggle-brats-results.zip')

In [ ]:
# %rm -rfv ablation_results/nifti
# %rm -rf ablation_results/fastsurfer_masks
# %rm -rf ablation_results paper_figures

In [ ]:
import random


if __name__ == '__main__':
    try:
        # To trigger the standard training, keep using run_pipeline(mode='train')
        # We will add a quick manual branch here to run the ablation study

        # mode = 'ablation'  # сhange to 'train', 'hpo', 'eda', 'ablation', 'cvpr', 'viz' as needed
        # mode = 'sota'  # 'train', 'hpo', 'eda', 'ablation', 'dsc', 'frechet', 'cvpr', 'sota' etc...
        # 'train', 'hpo', 'eda', 'ablation', 'sota', 'dsc', 'frechet', 'cvpr', 'all'
        mode = 'cvpr'  

        if mode == 'all':
            logger.info(">>> 🚀 INITIATING FULL MASTER EVALUATION PIPELINE 🚀 <<<")

            # --- 0. PRELOAD PREVIOUS RESULTS (RESUMABILITY) ---
            if os.path.exists(CFG.data.preload_dir):
                logger.info(f"📥 Preloading existing results from {CFG.data.preload_dir}...")
                os.makedirs(CFG.data.results_dir, exist_ok=True)
                import shutil
                # Direct 1:1 copy from the read-only mount to our writable results directory
                shutil.copytree(CFG.data.preload_dir, CFG.data.results_dir, dirs_exist_ok=True)
                logger.info("✅ Preload complete. Existing volumes will be skipped!")

            # --- 1. ABLATION STUDY (Our Models) ---
            logger.info("\n" + "=" * 50 + "\n PHASE 1: ABLATION STUDY \n" + "=" * 50)
            run_ablation_study(CFG)

            # --- 2. SOTA EVALUATION (MONAI 3D LDM) ---
            logger.info("\n" + "=" * 50 + "\n PHASE 2: MONAI SOTA INFERENCE \n" + "=" * 50)
            t1_files = find_t1_files(CFG.data.data_root_ixi)
            brats_list = get_brats_subjects(CFG.data.data_root_brats)

            random.seed(42)
            random.shuffle(t1_files)
            ixi_split = int(0.9 * len(t1_files))
            _, v_files = t1_files[:ixi_split], t1_files[ixi_split:]

            brats_pool =[]
            for b in brats_list[:20]: 
                v = VolumeLoader.load(b['t1'], b['seg'])
                if v: brats_pool.append(v)

            sota_eval = MonaiSotaEvaluator(CFG)
            sota_eval.setup()
            num_sota_vols = 5 if CFG.run_type == 'Interactive' else 20
            # sota_eval.evaluate_masked_inpainting(v_files, brats_pool, num_volumes=num_sota_vols)
            sota_eval.evaluate_sota_models(v_files, brats_pool, num_volumes=num_sota_vols)

            # --- 3. FRÉCHET METRICS (3D-FID & FVD) ---
            logger.info("\n" + "=" * 50 + "\n PHASE 3: FRÉCHET METRICS \n" + "=" * 50)
            frechet_eval = FrechetEvaluator(CFG)
            frechet_eval.evaluate()

            # --- 4. ANATOMICAL VALIDATION (FastSurfer DSC) ---
            logger.info("\n" + "=" * 50 + "\n PHASE 4: ANATOMICAL VALIDATION \n" + "=" * 50)
            dsc_eval = FastSurferEvaluator(CFG)
            dsc_eval.setup()
            dsc_eval.convert_to_nifti()
            dsc_eval.run_prediction()
            dsc_eval.calculate_dsc()

            # --- 5. CVPR PLOTTING ---
            logger.info("\n" + "=" * 50 + "\n PHASE 5: CVPR PLOTTING \n" + "=" * 50)
            run_cvpr_rendering(render_supp=False)

            logger.info(">>> 🎉 MASTER PIPELINE COMPLETE! CHECK 'paper_figures' DIRECTORY 🎉 <<<")

        elif mode == 'ablation':
            # Run the study
            run_ablation_study(CFG)
            run_cvpr_rendering(render_supp=True)
        elif mode == 'dsc':
            # evaluator = SynthSegEvaluator(CFG)
            evaluator = FastSurferEvaluator(CFG)
            evaluator.setup()

            # Check if we have data to process
            npy_files = glob.glob(os.path.join(CFG.data.results_dir, "*.npy"))
            if not npy_files:
                logger.warning(f"No .npy files found in '{CFG.data.results_dir}'. Running 'ablation' mode...")
                run_ablation_study(CFG)

            evaluator.convert_to_nifti()
            evaluator.run_prediction()
            evaluator.calculate_dsc()
            # Generate the specific plot
            generate_figure_4_anatomical_dsc()
        elif mode == 'frechet':
            # Run 3D-FID and FVD evaluation
            evaluator = FrechetEvaluator(CFG)

            # Check if we have data to process
            npy_files = glob.glob(os.path.join(CFG.data.results_dir, "*.npy"))
            if not npy_files:
                logger.warning(f"No .npy files found in '{CFG.data.results_dir}'. Running 'ablation' mode...")
                run_ablation_study(CFG)

            evaluator.evaluate()
            generate_figure_5_frechet_distances()
        elif mode == 'sota':
            # 1. Setup Data
            t1_files = find_t1_files(CFG.data.data_root_ixi)
            brats_list = get_brats_subjects(CFG.data.data_root_brats)

            random.seed(42)
            random.shuffle(t1_files)
            ixi_split = int(0.9 * len(t1_files))
            _, v_files = t1_files[:ixi_split], t1_files[ixi_split:]

            # Load BraTS pool for realistic masks
            brats_pool = []
            for b in brats_list[:20]: 
                v = VolumeLoader.load(b['t1'], b['seg'])
                if v: brats_pool.append(v)

            # 2. Run MONAI SOTA
            sota_eval = MonaiSotaEvaluator(CFG)
            sota_eval.setup()
            sota_eval.evaluate_masked_inpainting(
                v_files, brats_pool,
                num_volumes=5 if CFG.run_type == 'Interactive' else 20
            )

        elif mode == 'cvpr':
            # --- 0. PRELOAD PREVIOUS RESULTS (RESUMABILITY) ---
            if os.path.exists(CFG.data.preload_dir):
                logger.info(f"📥 Preloading existing results from {CFG.data.preload_dir}...")
                os.makedirs(CFG.data.results_dir, exist_ok=True)
                import shutil
                # Direct 1:1 copy from the read-only mount to our writable results directory
                shutil.copytree(CFG.data.preload_dir, CFG.data.results_dir, dirs_exist_ok=True)
                logger.info("✅ Preload complete. Existing volumes will be skipped!")

            run_cvpr_rendering(render_supp=True)
        else:
            run_pipeline(mode=mode)

    except Exception as e:
        logger.error(f"Pipeline Failed: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
%rm -rf FastSurfer SynthSeg
%ls -allah *

In [ ]:
%cat /kaggle/working/ablation_results/inference_performance.csv

In [ ]:
import os
import glob
import random

from IPython.display import display, Image, HTML


def display_random_dashboards(base_dirs=None):
    """
    Searches the specified EDA directories (or auto-detects all 'eda_*' folders),
    picks one random dashboard per dataset per axis, and displays it in native resolution.
    """
    print(">>> Searching for generated EDA dashboards...\n")

    # Auto-detect directories if not provided
    if base_dirs is None:
        base_dirs =[
            d for d in os.listdir('.') if os.path.isdir(d) and d.startswith('eda_')
        ]

    if not base_dirs:
        print("No 'eda_' directories found. Did the simulation finish?")
        return

    axes = ['axial', 'coronal', 'sagittal']

    for dset in sorted(base_dirs):
        for axis in axes:
            # Search for JPGs in the specific dataset/axis folder
            search_path = os.path.join(dset, axis, "*.jpg")
            files = glob.glob(search_path)

            if not files:
                continue

            # Pick one random dashboard
            chosen_file = random.choice(files)
            filename = os.path.basename(chosen_file)

            # Print a nice formatted HTML header
            header = f"""
            <hr>
            <h3 style='margin-bottom: 5px;'>
                Dataset: <span style='color: #E67E22;'>{dset}</span> | 
                Projection: <span style='color: #2ECC71;'>{axis.capitalize()}</span>
            </h3>
            <p style='margin-top: 0px; color: #7F8C8D;'>File: {filename}</p>
            """
            display(HTML(header))

            # Display the actual image natively
            display(Image(filename=chosen_file))

# Use it:
display_random_dashboards()

In [ ]:
# %%time
# import sys
# import pkg_resources
# import subprocess
# from collections import defaultdict


# # Get a list of all imported modules
# imported_modules = {
#     name: module
#     for name, module in sys.modules.items()
#     if module and getattr(module, '__file__', None)
# }

# # Get the version and dependencies of each imported module
# package_info = defaultdict(dict)

# for name in imported_modules:
#     try:
#         # Skip non-package modules
#         if name.startswith('_') or name in sys.builtin_module_names:
#             continue

#         dist = pkg_resources.get_distribution(name)
#         version = dist.version
#         package_info[name]['version'] = version

#         # Get dependencies using pip
#         result = subprocess.run(
#             ['pip', 'show', name], capture_output=True, text=True
#         )
#         dependencies = []
#         for line in result.stdout.split('\n'):
#             if line.startswith('Requires: '):
#                 dependencies = [
#                     d.strip()
#                     for d in line.split('Requires: ')[1].split(',')
#                     if d.strip()
#                 ]
#                 break

#         package_info[name]['dependencies'] = dependencies
#     except (pkg_resources.DistributionNotFound, subprocess.CalledProcessError):
#         pass

# # Print the package information
# for name, info in package_info.items():
#     print(f"Package: {name}")
#     print(f"Version: {info.get('version', 'N/A')}")
#     print(f"Dependencies: {', '.join(info.get('dependencies', []))}")
#     print()